# What Does Resting-State EEG Add to Demographics in Alzheimer's Disease?
### Incremental Value, Confounding, and Transportability Across Two Cohorts — análises para o journal

**Objective.** To determine whether resting-state EEG adds information about Alzheimer's disease beyond
sociodemographic variables, and whether this added information survives confounder control and transfer across cohorts.

**RQ1 (primary) — Incremental value.** Does an EEG-derived score improve the identification of AD beyond age, sex, and
education?
- *RQ1a.* In BrainLat, beyond age, sex, and education.
- *RQ1b.* In the Greek cohort (ds004504), beyond age and sex. Education was not recorded, so an E-value quantifies how
  strong unmeasured confounding would have to be to explain the effect away.
- *RQ1c.* Is the incremental effect consistent across the two cohorts?

**RQ2 — Confounding removed by design.** Does the EEG score still discriminate AD from controls when sociodemographic
confounding is removed by design?
- *RQ2a.* In BrainLat, within education-matched pairs and under overlap weighting for age, sex, and education.
- *RQ2b.* In the Greek cohort, under overlap weighting for age and sex.

**RQ3 — Transportability.** Does the EEG model keep its discrimination when moved, frozen, to the other cohort?
- *RQ3a.* BrainLat → Greek cohort (primary).
- *RQ3b.* Greek cohort → BrainLat (secondary).

**Exploratory.** Within AD, does the EEG score track global cognitive severity (MoCA in BrainLat; MMSE in the Greek
cohort) beyond age and, where available, education?

**Onde cada pergunta é respondida**

| Pergunta | O que o notebook estima | Seção |
|---|---|---|
| — | Download do ds004504 v1.0.9, conferido por SHA-256 | 1.2b |
| RQ1a | BrainLat: escore de EEG além de idade, sexo e escolaridade (OR por 1 DP; p por bootstrap nulo do LOSO) | 5.1 |
| RQ1b | Grego (ds004504): além de idade e sexo; E-value para a escolaridade, que não foi medida | 5.2 |
| RQ1 (confirmação) | Escore do modelo da outra coorte além da demografia (dados independentes do modelo) | 7.1, 7.2 |
| RQ1c | O efeito é o mesmo nas duas coortes? (modelo conjunto com termo coorte × EEG) | 5.3 |
| RQ2a | BrainLat: pares 1:1 por escolaridade, pares comparáveis e overlap weights (idade, sexo, escolaridade) | 6.1 |
| RQ2b | Grego: overlap weights (idade, sexo) e pares comparáveis (mesmo sexo, idade ±2 anos) | 6.2 |
| RQ3a | BrainLat → grego, modelo congelado (principal) | 7.1 |
| RQ3b | Grego → BrainLat | 7.2 |
| RQ3 (secundária) | Transporte com as features padronizadas dentro de cada coorte, sem rótulos | 7.3 |
| Exploratória | Dentro dos AD, o escore acompanha a gravidade (MoCA; MMSE)? | 8.1 |
| Descritiva | Quais das 11 features diferem além da demografia, e na mesma direção nas duas coortes? | 8.2 |
| Sensibilidade | Só as 7 espectrais; sem épocas de gama alto; SVM e XGBoost | 9 |
| Submissão | Fluxograma dos participantes, gravidade por coorte, checklist TRIPOD+AI e pacote do preprint | 13 |

**Decidido antes de abrir os rótulos gregos (SPEC, seção 1.3):** as mesmas 11 features do artigo da ACDSA, calculadas
nas 19 posições 10-20 que existem nas duas coortes; um único modelo principal (o Random Forest do artigo); LOSO dentro de
cada coorte; regras de leitura. Na primeira execução a SPEC é congelada (hash + data) e os rótulos do grego só são lidos
depois, na seção 2.2. A ordem fica gravada em `outputs_journal/log_execucao.json`.

**Como usar**

1. Confira os caminhos em 1.2 (ou defina `JOURNAL_BRAINLAT_DIR`, `JOURNAL_DS004504_DIR` e `JOURNAL_OUT`).
2. O ds004504 é baixado pelo próprio notebook (seção 1.2b: versão 1.0.9, `participants.tsv` + `derivatives/`, 2,95 GB).
   Se você já tem o banco, aponte `JOURNAL_DS004504_DIR` para a pasta: o notebook confere cada arquivo por SHA-256 e só
   baixa o que faltar ou estiver diferente.
3. Leia a SPEC (1.3) **antes de rodar qualquer coisa**: a primeira execução congela a SPEC e registra quando os rótulos
   gregos foram lidos.
4. `Kernel → Restart & Run All`.
5. Me mande `outputs_journal/para_enviar.zip` (resumo, tabelas e figuras).
6. A seção 13 monta o material da submissão em `outputs_journal/pacote_preprint/` (e `pacote_preprint.zip`). Ela também roda
   sozinha depois das seções 1 a 3, que são rápidas porque as features ficam em cache.

**Tempo (primeira execução):** download, depende da internet (2,95 GB); extração das features, 5–15 min; LOSOs do Random
Forest, 10–20 min; **bootstrap nulo da RQ1 (seções 5.1 e 5.2), de 30 min a 6 h**: ele refaz o LOSO até 199 vezes por
coorte e para cedo quando o resultado é claramente não significativo, então demora mais justamente quando há efeito.
Sensibilidades, 30–60 min. Tudo fica em cache, inclusive cada réplica do bootstrap: se parar no meio, rodar de novo continua
de onde parou. Vale deixar rodando à noite. `JOURNAL_RAPIDO=1` existe só para testar o código com dados sintéticos; com os
dados reais, rode direto a versão completa.

O que cada escolha tem de base na literatura está na seção 12, com os links.

## 1. Preparação

### 1.1 Imports, versões e estilo das figuras

In [ ]:
# 1.1 — Imports, versões e estilo das figuras
import os, sys, re, json, time, pickle, hashlib, platform, warnings, datetime, logging, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats
from scipy.optimize import linear_sum_assignment, brentq
from scipy.signal import hilbert, butter, sosfiltfilt

import sklearn
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

try:
    import xgboost
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
try:
    import mne
    from mne.time_frequency import psd_array_welch
    mne.set_log_level("ERROR")
    MNE_OK = True
except ImportError:
    MNE_OK = False
try:
    from IPython.display import display
except ImportError:  # fora do Jupyter
    display = print

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

plt.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
COL1, COL2 = 3.31, 6.85   # larguras de coluna da Springer (84 mm e 174 mm), em polegadas
CORES = {"EEG": "#D55E00", "EEG_outra": "#CC79A7", "Demo": "#0072B2", "Demo_EEG": "#009E73"}

VERSOES = {
    "python": platform.python_version(), "sistema": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__ if XGB_OK else "AUSENTE",
    "mne": mne.__version__ if MNE_OK else "AUSENTE", "matplotlib": matplotlib.__version__,
}
for k, v in VERSOES.items():
    print(f"{k:13s} {v}")
if not MNE_OK:
    raise ImportError("Este notebook precisa do MNE para ler os .set: pip install mne")
if not XGB_OK:
    print("\nAVISO: xgboost não instalado; a sensibilidade com XGBoost será pulada (pip install xgboost).")

### 1.2 Caminhos e opções de execução

Nada aqui entra na SPEC. `RAPIDO` existe só para testar o código: menos árvores, menos reamostragens e no máximo 20
épocas por pessoa. **Nunca use números do modo rápido.**

In [ ]:
# 1.2 — Caminhos e opções de execução (não fazem parte da SPEC)
BRAINLAT_DIR = Path(os.environ.get(
    "JOURNAL_BRAINLAT_DIR",
    r"<BRAINLAT_DIR>"))
DIR_AD = BRAINLAT_DIR / "1_AD"
DIR_HC = BRAINLAT_DIR / "5_HC"
DS_DIR = Path(os.environ.get("JOURNAL_DS004504_DIR", r"<DS004504_DIR>"))
OUT = Path(os.environ.get("JOURNAL_OUT", "outputs_journal")).resolve()

RAPIDO = os.environ.get("JOURNAL_RAPIDO", "0") == "1"   # só para testar o código: NUNCA para o artigo
USAR_CACHE = True        # reaproveita extrações e LOSOs já rodados
N_JOBS = -1              # núcleos do Random Forest e do XGBoost (-1 = todos)

for sub in ("tabelas", "figuras", "cache", "features", "modelos"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
T0_GLOBAL = time.time()
INICIO = datetime.datetime.now().isoformat(timespec="seconds")

for nome, p in (("BRAINLAT_DIR", BRAINLAT_DIR), ("DIR_AD", DIR_AD), ("DIR_HC", DIR_HC),
                ("DS_DIR", DS_DIR), ("OUT", OUT)):
    print(f"{nome:13s} {'existe   ' if p.exists() else 'NÃO EXISTE (o ds004504 é baixado em 1.2b)' if nome == 'DS_DIR' else 'NÃO EXISTE'}  {p}")
faltando = [n for n, p in (("DIR_AD", DIR_AD), ("DIR_HC", DIR_HC)) if not p.exists()]
if faltando:
    raise FileNotFoundError(f"Pastas não encontradas: {faltando}. Ajuste os caminhos acima "
                            "(ou a variável de ambiente JOURNAL_BRAINLAT_DIR).")
(OUT / "ambiente.txt").write_text("\n".join(f"{k}: {v}" for k, v in VERSOES.items())
                                  + f"\ninicio: {INICIO}\nrapido: {RAPIDO}\n", encoding="utf-8")
if RAPIDO:
    print("\n" + "!" * 78 + "\nMODO RÁPIDO: só para testar o código. NÃO use estes números.\n" + "!" * 78)

### 1.2b Download do ds004504 (versão 1.0.9)

> **O que faz:** baixa do OpenNeuro só o que o notebook usa: `participants.tsv`, os metadados e a pasta `derivatives/`
> (88 arquivos `.set`, 2,95 GB). Antes de baixar, confere o que já existe na pasta: arquivo com tamanho e SHA-256 iguais aos
> da versão 1.0.9 não é baixado de novo. O caminho principal é o bucket público do OpenNeuro na AWS (o mesmo de
> `aws s3 sync --no-sign-request s3://openneuro.org/ds004504`), sem instalar nada. Se algum arquivo falhar, tenta pelo
> `openneuro-py` com a versão fixada em 1.0.9 (instala o pacote se precisar).
>
> **Por que fixar a versão e conferir o SHA-256:** os resultados ficam presos a uma versão citável do banco
> (doi:10.18112/openneuro.ds004504.v1.0.9). Os tamanhos e SHA-256 vêm das chaves git-annex do espelho oficial
> (github.com/OpenNeuroDatasets/ds004504, tag 1.0.9). Se o OpenNeuro publicar outra versão com os dados mudados, o notebook
> avisa em vez de misturar versões.
>
> Os rótulos (coluna `Group`) não são lidos aqui; só na seção 2.2, depois do congelamento da SPEC.

In [ ]:
# 1.2b — Download do ds004504 (versão 1.0.9), conferido por tamanho e SHA-256
import urllib.request, urllib.error, urllib.parse, subprocess

DS_VERSAO = "1.0.9"
DS_DOI = "10.18112/openneuro.ds004504.v1.0.9"
BAIXAR_DS004504 = os.environ.get("JOURNAL_BAIXAR", "1") == "1"   # False: só confere o que já existe
INSTALAR_OPENNEURO_PY = True     # se o download direto falhar, instala o openneuro-py (pip) e tenta por ele
DS_BASE_URL = os.environ.get("JOURNAL_DS_BASE_URL", "https://s3.amazonaws.com/openneuro.org/ds004504")

# Manifesto da versão 1.0.9 (tamanho em bytes e SHA-256). derivatives: sub-NNN/eeg/sub-NNN_task-eyesclosed_eeg.set
META = [
    ("participants.tsv", 1706, "bd17cf02ec9733c9029d4a6d977b2918309b3c667ff9ceb1b5dce6e3cebccbc1"),
    ("participants.json", 613, "e479ba99b89ca756d8894ec14665780eada7c0e831ca2422699439d9ae98740c"),
    ("dataset_description.json", 2852, "5e9b3a8be6b0d682825815b0eeda1ef8ffcf6653a8ca9919285fb349d8231632"),
    ("README", 4168, "a551f7e4844f09515499da342605cbdc8aff86a627057048da95748ea6d3b48a"),
    ("CHANGES", 500, "c8dede802008619762fb5c3d5c5f706774b21eb2f6d2182e23b26cd61d1d01a0"),
]
DERIV = [
    ("001", 24357408, "2bee41ac31694c06688bd4e36e4fd1233b4744c5ac80a822ec2a9cd67c5efe0d"),
    ("002", 32186728, "7996d30b453a6438d39f0059547b8ef50669f9f084111d05bfec694a14b59a2c"),
    ("003", 13076664, "6689383d189c2469b9f9786d2e0487d92cc215ca1eb1f729a792ed51ad9a9f9c"),
    ("004", 30076328, "e9a582edcdd5a6fffd1bb7232007ec11d29486dba46d0e5c45f260cb9e7b92b8"),
    ("005", 32631904, "befa4f7ac305c6faefb92fa0f82028327f51c277f5c5ecaf6271d669e7800291"),
    ("006", 26945984, "3b87b043aa7540910d7b9e4b871e1f11af1f2437ee2f1cfb6f6b14d86f36ee04"),
    ("007", 32640632, "d1aa601da8b8ecdf4f3ae7be1857aaca414209d2791b3cfdf02630a9258cfa1e"),
    ("008", 33789504, "30d3b20abae8fe15f9c4771da03ac090403a0adfeed2292fae67a4484a297b05"),
    ("009", 26102248, "6a22b93a1901593f2eb1fc600c03af8a54f318f4a8695745f5a65bfb63f76657"),
    ("010", 54545192, "514316b75b01095d987b50f286f8d1843da2f13d24dbac42d245af1378d4c777"),
    ("011", 32788688, "5f99ea3575b9b2f5c33f1c091d27c7a2119b105feab11dfbfa5187b764f6a998"),
    ("012", 37624784, "871259ec1ff9fa4bfda7f2ea92b1a9a597de716c28e4a4181616267206ec5b13"),
    ("013", 35687368, "cb34050f098bf22a3f8c2d2ad310ad7a26b336dd80707cd6c845c9ea051dde8a"),
    ("014", 39786624, "7de89cab9ba48b48f7c7d470f78ed1164488910e8ab905a293be376a3a2d8448"),
    ("015", 38380232, "ecc8fb5fd0c67f5ecfa61a98ad8c99c9fbb91ded460ae622e766ccbd3bb7b694"),
    ("016", 41399792, "5ed950cc3b031ca26d6edb8666225f6d8136fbbf6758869130ab27eef67ce914"),
    ("017", 35912800, "9ef28e09655f1b57fdd44eed4b594bbc4f89ce8630eb8a9acfb2ded9ece241e5"),
    ("018", 35984528, "a7a62ed9c6a3155d421fd0750c3a4cdec7f60b27dfc95be687092e1ede1552c7"),
    ("019", 39108448, "91b718e4ec88613f4b3f6a1105c909a8d524e9176801f46b69dcf0aabd010b09"),
    ("020", 35272512, "31ad3071940f98dc3abe7fb6079f2fbb65d4c07f7e81c9c318bd719035533050"),
    ("021", 39179520, "f5cee362ad4f1b7c7983bec413fbeff2813ad53c120663c2b62ae228f87abe65"),
    ("022", 34999360, "c6aa0f028980b37694c8f6c5eebf33a30251da88f82a275409fd84f0a14044f3"),
    ("023", 35559904, "6f76aa5f0f7e76e7c924f87dfe5cd4a54bdf205da1a39d050e51263462f28ea6"),
    ("024", 32362568, "20f9c037debb0a5123e50ddb23a9c724902573d9730ddf313d347154806468f2"),
    ("025", 29160688, "9f69af89ee60139680ee1255d8133433da3d858e786c88d34b698b89883f1c48"),
    ("026", 38269992, "cee439f692a00cca2ebadc9cab9bfbfdf0c53c80da92984874a0621486bd82cb"),
    ("027", 35182824, "8647c35865d88d35b96641774465a1848374da726e3d27af10cac3e67b4d9e1b"),
    ("028", 34842544, "456cef7bceecea7d73aa05204d40462939c46c0d587f727c2d258a872e766197"),
    ("029", 31377552, "976381a11cc2d05ab83dfa18ed2ea5b823de0b628123a7a87b56261a6e4d1c7e"),
    ("030", 23616688, "52ac5b13f2b9f63404d3ed9fa441f62cdcb007e18d0536d3948d2d6219ea415b"),
    ("031", 48977152, "b3aa5706a8105488c1804ebd3af84e0feb3617f60fd919fbc118cc1a5b3c93c5"),
    ("032", 34006464, "85df1c92af6da493a20aff8714ebe54b50a9ce0067800b2879141d225a8e38f1"),
    ("033", 30047768, "d84c49e0663506d46db48c6862d1591b88df3d15d50e0e51da73a87e138fb3dd"),
    ("034", 41287168, "d3bc3e0b4be18da274907ff65b3bbe3355ad6ef8d6b1ca6f3cbefae88585d9b6"),
    ("035", 31571112, "09deb5d9eab734e989c4c38d0296763b3697ab93ed46445f1a0fabf3fca964de"),
    ("036", 35830920, "0e2d814176e6f0f55245d339ec0fec12c1c014ac2933ecffd1e1da495e797c3c"),
    ("037", 31538736, "9921624e8bd740202cbb116a23f66fe9258378998bd0032bfefa47b7bf9d35d9"),
    ("038", 37952112, "cdc9cf5e704517ae645483f7f4af73644407510c99aede2b2ef9d114a5d7b8bb"),
    ("039", 36199128, "5576afeb06073f77760f4803dc4333e2c77ddb25e40cf3af54343b665214e593"),
    ("040", 41142360, "20279f151615d595e925631f226110b6ee02691f0c5dee1223f8176f1e645e4f"),
    ("041", 37725664, "6643ac4b575fcb6cf03329274034a345077ebd39cbf0e4d8daa647d5c395eadb"),
    ("042", 41001992, "88aed8afce7b3ffc2ff879713c5dcf95048c3583691c01d7bef850fc680c294b"),
    ("043", 33644736, "b16d7982c0660c627c25e758698937ce58f046f08f39fe4df17452dab2f65f1b"),
    ("044", 35770656, "89f84d55abd8ca9d37fae77bd5735956f4f98fcc72a8e9d8e33b9881499eb422"),
    ("045", 36267832, "06872ef2c34385452a7bb106acfc2c773b88a72898e27554b822568ed6ac3e3b"),
    ("046", 32074040, "1a1dc77cc8be6b6405e89876757da3db75cd11c08a33f7613c02a4606c91fe1a"),
    ("047", 34364248, "94ed76a3ddc87d8adb0db946fefdbfa18dbfae7cdfae4ee9e1aec9dbb7d39eb7"),
    ("048", 42150744, "b749f9e3058dbb3398ce498d1f205c68b5a4312e4f766c8e37f3452b8ed26bd0"),
    ("049", 33226992, "98f2b1baf832eae2357754a0d5be50ca01ad0ed5b9d0f0e786e85d51d6572a3e"),
    ("050", 34821776, "a5340e8a7feb7e35595ca512bf4c4e0478ed1bfac762afd9d014be3e77731c54"),
    ("051", 32103976, "568e5f06f280ae7dc31256b0637f1f610183fd5ff72f30dca6214e33f3c739e0"),
    ("052", 32371008, "068e3687d2dff59ff1b790485236551439136c6d139e2e88d9d26e47d7d83b47"),
    ("053", 33250384, "c06142af0699bdf65b61e6457609cf40b92587e59d98f03fd84d69816285b1ed"),
    ("054", 35668024, "53e5b0f56b67f33d6c02ad277fda1d57d383ccb27b43c2c19f39beb5aaf7a3ca"),
    ("055", 34692736, "9d65d41bac4803e47e328b8c586aa82866805d89e77c2a6183c520df467df0f9"),
    ("056", 33755288, "9a8fbb0923d836bd5aa29010124828f794648beb4b52eabf7c046e4586b41ba3"),
    ("057", 32324112, "4a0b8095a0f0f75ffb65ba96d1555c50a3e2522d53f71013957d3a287322e800"),
    ("058", 32233184, "f5a8ca25a68bfdf219d2399d31575ff901c824afc5b1cb76f84ac7d1bfc10df9"),
    ("059", 33520392, "fb6bd8c4d81efa76eb54c9025b788c04f0944a9bab602d97999995b01555a16b"),
    ("060", 31878360, "fe203229a97787a46cecde4d8e1abfe90b60183b65e99e35393793bff8e76adf"),
    ("061", 34208384, "edb69e0e90c8489d63fe801fe8c5fec4cedc655ec6b14f56b18a42f726280f65"),
    ("062", 38269768, "b75b58ca0f404bec77680fac09555091799dc902338376a8f262cbb3bdb4b246"),
    ("063", 34278776, "5c61ab2332d33e62e16ba3f4749625825d23ea12c50630d92ec3a576525e1f14"),
    ("064", 36111008, "4e0162e511499bfdd958ae8ba84aad1048b6cf106580643135eb67866358a0a0"),
    ("065", 35871912, "a9220b46810a2b187037e669bc59840d2923161ada06bc471a2b962dc1895b43"),
    ("066", 23431728, "cd323034017eea41f5c9ecc1f7535a30c0326f59e1c4d85bf3a6cec7ab8bda52"),
    ("067", 26111736, "d7e2b5f66437c30643ad28ca8730c4b1b7b5bbb6037ab38249f8d29d78bdba8f"),
    ("068", 24423840, "62fac557c99a1914d7f819343b7ddba66053e74f095f337240c3a5214856b7fe"),
    ("069", 27101336, "d16e501268da0b4d4fd873c32906fd404a9d7c527dac384ffdd08377efb80282"),
    ("070", 19469064, "9f50e2e825a33961ffed6f6da35d942de20c92268feafdecd610bad188781faf"),
    ("071", 26371896, "73639ad2c30a4d33305640fef7da944699cb80fbf205f51991e55294bef7fe79"),
    ("072", 28024888, "94feae216ee0e44d2858bf66b85b573fabc5a45fd4c45e590025107106cfcc7a"),
    ("073", 36387608, "13462b8c825e45cb2d4e5097cbbed5e46a52cb4ce007fb359bcc47ca2883d00b"),
    ("074", 43190576, "323d400ad2e18106de90ca7ed5192deaa7d8b8f3e2aa3188f13744db3e3acbaf"),
    ("075", 31865608, "98d0bffea203d140f1385a41c219e9f5bc3e6b71ad89c3c825e61a7875e0c271"),
    ("076", 33227256, "2b5f954943899f88157eace7532785a34d62a5c7cdcc98f34cf926be6ac3238d"),
    ("077", 29782968, "5d23325eca733ae0ef9bd4ce1d1352d9869830ffd0933e2d90c77e8a7a0276bb"),
    ("078", 37082904, "3dd1baae55cceaaae0359a5bfcd83e8dbd65c6c0e28f0f2a2ff1e9e870970504"),
    ("079", 34780448, "c53402e00fe8e65d283a257c84365abe53e74cc97a21bd54463f6bde793a6fdd"),
    ("080", 38969512, "a8dd6fccc53a582fdd3278995edd654bb39aafffb92d20eb1dca803328328991"),
    ("081", 35109744, "22352b2257891716a26e8787b812e4ff11df389abde1bb4f2554662def9b0c72"),
    ("082", 31559320, "18aa69628a62676b2b679edeb90c123d5de237f1a4f7d63b7e75d58a753445d3"),
    ("083", 37050784, "8cd414126e564a2d0e34bf622a69602e8f378d3e8dd16612a9b7d303823eab88"),
    ("084", 26475912, "0b99404c8b7eef25be6746686ceaf12975d9c6ecf4569c45e24a5dc61042f098"),
    ("085", 23872736, "3498d4a95aeb30d28034a7713b480e450fb3fc04faa0df17a94672122d5bf6e9"),
    ("086", 24714304, "7e5e1da143fff33cd62b88735a358b1b05a388e1d775c8c8fe23c0544babe2e5"),
    ("087", 25685272, "0974feca278c7e5d7cd9f0c51ba74f1b037d61662c7be111a1d7c9949ba1e417"),
    ("088", 33421328, "f34f5e7e62dad1a4722acf5b007dc1f2744018fff3d5f5fa5caa1872c61debc3"),
]
ESSENCIAIS = {"participants.tsv"}   # além dos .set: se não conferirem, é alerta (dado diferente da versão 1.0.9)
if os.environ.get("JOURNAL_DS_MANIFESTO"):                 # só para os testes com dados sintéticos
    MANIFESTO = [tuple(a) for a in json.loads(Path(os.environ["JOURNAL_DS_MANIFESTO"]).read_text())["arquivos"]]
else:
    MANIFESTO = META + [(f"derivatives/sub-{n}/eeg/sub-{n}_task-eyesclosed_eeg.set", s, h) for n, s, h in DERIV]
AVISOS_DOWNLOAD = []


def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for bloco in iter(lambda: fh.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


f_conf = OUT / "download_ds004504.json"
CONF_DL = json.loads(f_conf.read_text(encoding="utf-8")) if f_conf.exists() else {}
CONF_DL.setdefault("conferidos", {})


def _confere(rel, tam, sha):
    # True se o arquivo existe com o tamanho e o SHA-256 da versão; guarda o resultado (não refaz o hash à toa)
    p = DS_DIR / rel
    if not p.exists() or p.stat().st_size != tam:
        return False
    marca = [p.stat().st_size, p.stat().st_mtime_ns, sha]
    if CONF_DL["conferidos"].get(rel) == marca:
        return True
    ok = _sha256(p) == sha
    if ok:
        CONF_DL["conferidos"][rel] = marca
    return ok


def _baixar_direto(rel, tam, sha):
    # Bucket público do OpenNeuro (HTTPS, sem credenciais); grava em .part e só renomeia se tamanho e SHA-256 conferirem
    url = f"{DS_BASE_URL}/{urllib.parse.quote(rel)}"
    destino = DS_DIR / rel
    destino.parent.mkdir(parents=True, exist_ok=True)
    tmp = destino.with_name(destino.name + ".part")
    ultimo = None
    for tentativa in range(4):
        try:
            h = hashlib.sha256()
            req = urllib.request.Request(url, headers={"User-Agent": "journal-eeg-notebook"})
            with urllib.request.urlopen(req, timeout=120) as r, open(tmp, "wb") as fh:
                for bloco in iter(lambda: r.read(1 << 20), b""):
                    fh.write(bloco)
                    h.update(bloco)
            if tmp.stat().st_size != tam:
                raise IOError(f"tamanho {tmp.stat().st_size} em vez de {tam}")
            if h.hexdigest() != sha:
                raise IOError("SHA-256 diferente do da versão 1.0.9")
            os.replace(tmp, destino)
            return
        except urllib.error.HTTPError as exc:
            ultimo = exc
            if exc.code in (403, 404):
                break
        except Exception as exc:
            ultimo = exc
        time.sleep(2 ** tentativa)
    if tmp.exists():
        tmp.unlink()
    raise RuntimeError(f"{rel}: {ultimo}")


def _baixar_openneuro_py(rels):
    try:
        import openneuro
    except ImportError:
        if not INSTALAR_OPENNEURO_PY:
            raise
        print("Instalando openneuro-py (pip) ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openneuro-py"])
        import openneuro
    openneuro.download(dataset="ds004504", tag=DS_VERSAO, target_dir=str(DS_DIR), include=list(rels))


DS_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.time()
faltam = [(r, s, h) for r, s, h in MANIFESTO if not _confere(r, s, h)]
print(f"ds004504 v{DS_VERSAO}: {len(MANIFESTO) - len(faltam)} de {len(MANIFESTO)} arquivos já estão na pasta e conferem "
      f"({time.time() - t0:.0f} s de conferência)")
metodo = "nenhum (já estava completo)" if not faltam else "nenhum (download desligado)"
if faltam and BAIXAR_DS004504:
    gb = sum(s for _, s, _ in faltam) / 1e9
    print(f"Baixando {len(faltam)} arquivo(s), {gb:.2f} GB, de {DS_BASE_URL} ...")
    metodo, falhas, feito, t0 = "bucket público do OpenNeuro (HTTPS)", [], 0, time.time()
    for k, (r, s, h) in enumerate(faltam):
        try:
            _baixar_direto(r, s, h)
            feito += s
        except Exception as exc:
            falhas.append((r, s, h))
            print(f"  falhou: {exc}")
            if len(falhas) >= 3 and feito == 0:      # nada funcionou: não insiste arquivo por arquivo
                falhas += [x for x in faltam[k + 1:]]
                break
        if (k + 1) % 10 == 0 or k + 1 == len(faltam):
            dt = time.time() - t0
            print(f"  {k + 1}/{len(faltam)} | {feito / 1e9:.2f} GB | {feito / 1e6 / max(dt, 1e-9):.1f} MB/s")
    if falhas:
        print(f"{len(falhas)} arquivo(s) não vieram pelo bucket; tentando pelo openneuro-py (versão {DS_VERSAO}) ...")
        try:
            _baixar_openneuro_py([r for r, _, _ in falhas])
            metodo += " + openneuro-py"
        except Exception as exc:
            print(f"  openneuro-py também falhou: {str(exc)[:300]}")
    faltam = [(r, s, h) for r, s, h in MANIFESTO if not _confere(r, s, h)]
elif faltam:
    print("Download desligado (JOURNAL_BAIXAR=0): só conferi o que já existe.")

ruins_essenciais = [r for r, _, _ in faltam if r in ESSENCIAIS or r.endswith(".set")]
ruins_meta = [r for r, _, _ in faltam if r not in ruins_essenciais]
if ruins_meta:
    AVISOS_DOWNLOAD.append(f"ds004504: {len(ruins_meta)} arquivo(s) de metadados ausentes ou de outra versão: {ruins_meta}")
if ruins_essenciais:
    AVISOS_DOWNLOAD.append(f"ds004504: {len(ruins_essenciais)} arquivo(s) de dados ausentes ou diferentes da versão "
                           f"{DS_VERSAO} (primeiros: {ruins_essenciais[:5]}). Os participantes afetados podem ficar de fora.")
for a in AVISOS_DOWNLOAD:
    print("⚠️  ", a)
if not (DS_DIR / "participants.tsv").exists():
    raise FileNotFoundError(
        f"Não consegui obter {DS_DIR / 'participants.tsv'}. Baixe o ds004504 v{DS_VERSAO} manualmente "
        "(https://openneuro.org/datasets/ds004504/versions/1.0.9 → Download, ou "
        "'aws s3 sync --no-sign-request s3://openneuro.org/ds004504 <pasta>') e aponte DS_DIR para a pasta.")
doi_local = ""
if (DS_DIR / "dataset_description.json").exists():
    try:
        doi_local = json.loads((DS_DIR / "dataset_description.json").read_text(encoding="utf-8")).get("DatasetDOI", "")
    except Exception:
        pass
CONF_DL.update({"versao": DS_VERSAO, "doi_esperado": DS_DOI, "doi_na_pasta": doi_local, "metodo": metodo,
                "conferido_em": datetime.datetime.now().isoformat(timespec="seconds"),
                "arquivos_ok": len(MANIFESTO) - len(faltam), "arquivos_manifesto": len(MANIFESTO),
                "avisos": AVISOS_DOWNLOAD})
f_conf.write_text(json.dumps(CONF_DL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"DOI na pasta: {doi_local or 'n/d'} | método: {metodo}")

### 1.3 Especificação congelada (`SPEC`)

Tudo o que define as análises está aqui. Na primeira execução a SPEC é gravada com hash e data em
`outputs_journal/SPEC_journal.json`; nas seguintes, o notebook para se ela mudar. Se precisar mudar algo depois (por
exemplo, corrigir um erro, sem ter olhado resultados), registre data, o quê e por quê em `historico_alteracoes`, apague o
arquivo e rode de novo: o registro aparece no resumo final.

O sexo no BrainLat segue o dicionário do próprio banco (`brainlat_dataset_dictionary.csv`: 0 = feminino).

In [ ]:
# 1.3 — Especificação das análises. Edite ANTES da primeira execução real.
SPEC = {
    "versao": "journal — BrainLat + ds004504, plano de 19/11/2026",
    "titulo_de_trabalho": ("What Does Resting-State EEG Add to Demographics in Alzheimer's Disease? "
                           "Incremental Value, Confounding, and Transportability Across Two Cohorts"),
    "semente": 42,
    "rotulos": {"HC": 0, "AD": 1},
    "coortes": {
        "brainlat": {"pastas": {"AD": "1_AD", "HC": "5_HC"},
                     "sexo_valor_feminino": 0,
                     "fonte_codigo_sexo": "brainlat_dataset_dictionary.csv: sex 0 = female, 1 = male",
                     "covariaveis": ["Age", "sex_fem", "years_education"]},
        "grego": {"dataset": "ds004504 (OpenNeuro; Miltiadous et al., 2023)", "versao_openneuro": "1.0.9",
                  "doi": "10.18112/openneuro.ds004504.v1.0.9", "grupos": {"A": 1, "C": 0},
                  "excluidos": "F (demência frontotemporal)", "usar_derivatives": True,
                  "covariaveis": ["Age", "sex_fem"], "escolaridade": "não medida: E-value"},
    },
    "canais_1020": ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4",
                    "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"],
    "mapeamento_brainlat": ("eletrodo BioSemi mais próximo de cada posição 10-20 (ângulo na esfera unitária, "
                            "atribuição única por linear_sum_assignment)"),
    "preprocessamento": {"referencia": "média dos 19 canais", "filtro": "FIR (firwin) 0,5–45 Hz, fase zero",
                         "harmonizacao_banda_grego": {
                             "motivo": ("o BrainLat foi distribuído com passa-banda Butterworth 0,5–40 Hz, ordem 8, fase "
                                        "zero (Prado et al., 2023); os derivatives do ds004504, com 0,5–45 Hz (README)"),
                             "h_freq": 40.0, "ordem": 8, "tipo": "butter", "fase": "zero"},
                         "epoca_s": 4.0, "normalizacao": "RMS por sujeito (não altera as features)",
                         "welch_resolucao_hz": 1.0,
                         "taxa_amostragem": "nativa (512 Hz no BrainLat; 500 Hz no ds004504)",
                         "max_epocas_por_sujeito": None,
                         "rejeicao_epocas": "nenhuma, como no artigo (só épocas com NaN/inf são descartadas)"},
    "features": {
        "todas": ["Rel_Theta_mean", "Rel_Alpha_mean", "Rel_Beta_mean", "Rel_Gamma_mean",
                  "Razao_Theta_Alpha", "Razao_Theta_Beta", "Spectral_Entropy",
                  "PLV_Theta_mean", "PLV_Alpha_mean", "PLV_Beta_mean", "Corr_Broadband_mean"],
        "espectrais": ["Rel_Theta_mean", "Rel_Alpha_mean", "Rel_Beta_mean", "Rel_Gamma_mean",
                       "Razao_Theta_Alpha", "Razao_Theta_Beta", "Spectral_Entropy"],
    },
    "modelo_primario": "RandomForest",
    "classificadores": {
        "RandomForest": {"n_estimators": 300, "max_depth": None, "max_features": "sqrt",
                         "min_samples_leaf": 1, "bootstrap": True, "class_weight": "balanced",
                         "random_state": 42},
        "XGBoost": {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5, "subsample": 0.8,
                    "colsample_bytree": 0.8, "eval_metric": "logloss", "random_state": 42,
                    "scale_pos_weight": "n_HC/n_AD nas épocas de treino de cada fold"},
        "SVM": {"kernel": "rbf", "C": 1.0, "gamma": "scale", "calibracao": "sigmoid",
                "folds_calibracao": 3, "calibracao_agrupada_por_sujeito": True, "random_state": 42},
    },
    "validacao": {"esquema": ("LOSO por sujeito dentro de cada coorte; treino no nível de época; "
                              "P(AD) do sujeito = média das épocas"),
                  "bootstrap_n": 2000,
                  "bootstrap": "sujeitos reamostrados com reposição, estratificado por classe"},
    "rq1": {"escore": "logit de P(AD) out-of-fold, padronizado dentro da coorte (OR por 1 DP)",
            "teste": ("estatística: razão de verossimilhança (1 gl) da logística demografia + escore; p-valor por "
                      "bootstrap paramétrico sob H0 de todo o LOSO, porque o teste ingênuo com escores de validação "
                      "cruzada pode ser liberal (Höfling & Tibshirani, 2008; Shang et al., 2025); IC do OR por "
                      "verossimilhança perfilada (descritivo)"),
            "bootstrap_nulo": {"B_max": 199, "parar_apos_excedencias": 10,
                               "esquema": ("y* ~ Bernoulli(P(AD | covariáveis)) do modelo nulo; LOSO refeito com y*; "
                                           "parada sequencial de Besag & Clifford (1991)")},
            "independente": ("verificação válida sem reamostragem: escore do modelo treinado na OUTRA coorte, além da "
                             "demografia (Dinga et al., 2020: teste paramétrico vale em dados independentes)"),
            "regra_firth": "Firth se a MLE não convergir ou se algum |β| padronizado passar de 10",
            "grego_unilateral": True, "direcao": "OR > 1, fixada pelo BrainLat",
            "delta_auc": "LOO logística balanceada (demografia vs demografia + escore EEG); descritivo, sem teste",
            "e_value": "VanderWeele & Ding (2017); RR ≈ √OR porque o desfecho é comum",
            "e_value_referencia": "no BrainLat, OR do EEG ajustado por idade + sexo vs + escolaridade (quanto a escolaridade pesa)",
            "verificacao_sitio_brainlat": "idade + sexo + escolaridade + sítio; verificação, fora da regra de leitura",
            "conjunto": ("AD ~ coorte + covariáveis com efeito próprio em cada coorte (BrainLat: idade, sexo, "
                         "escolaridade; grego: idade, sexo) + z_EEG comum; heterogeneidade: + coorte × z_EEG (LR, 1 gl)")},
    "rq2": {"pareamento_brainlat": {"variavel": "years_education", "calibre_anos": 2, "razao": "1:1",
                                    "algoritmo": "ótimo (linear_sum_assignment), sem reposição, desempate pela idade",
                                    "ic": "bootstrap de pares (Austin & Small, 2014)",
                                    "min_pares": 10, "n_desempates_aleatorios": 200},
            "pares_comparaveis_brainlat": {"calibre_escolaridade_anos": 2},
            "overlap_brainlat": ["Age", "sex_fem", "years_education"],
            "overlap_grego": ["Age", "sex_fem"],
            "pares_comparaveis_grego": {"mesmo_sexo": True, "calibre_idade_anos": 2},
            "aauc": ("verificação: AUC ajustada por covariáveis (Janes & Pepe, 2009), com o escore padronizado pela "
                     "regressão linear nos controles (BrainLat: idade, sexo, escolaridade; grego: idade, sexo)"),
            "escore": "P(AD) out-of-fold do RF, sem retreino"},
    "rq3": {"principal": "BrainLat → grego: RF treinado em todo o BrainLat, congelado e aplicado uma única vez",
            "secundaria": "grego → BrainLat",
            "recalibracao": "nenhuma; AUC por sujeito é a métrica principal",
            "calibracao": ("descritiva: inclinação, intercepto com inclinação fixa em 1 (calibration-in-the-large; "
                           "Van Calster et al., 2016) e Brier"),
            "limiar_metricas": 0.5,
            "padronizacao_por_coorte": ("secundária: cada feature padronizada (média e DP de todas as épocas) dentro "
                                        "da própria coorte, sem rótulos, antes de treinar e aplicar; fora da regra de leitura")},
    "exploratoria": {"executar": True,
                     "brainlat": "MoCA nos AD (paralelo ao MMSE); ρ de Spearman parcial para idade e escolaridade",
                     "brainlat_secundaria": "IFS nos AD (continuidade com o artigo da ACDSA); mesmo ajuste",
                     "grego": "MMSE nos AD; ρ de Spearman parcial para idade"},
    "descritiva_por_feature": ("cada feature (média por sujeito, padronizada) além das covariáveis da coorte; OR por 1 DP, "
                               "IC perfilado, q de Benjamini–Hochberg dentro da coorte; fora da regra de leitura"),
    "sensibilidade": {"so_espectrais": True, "sem_gama": True, "excluir_gama_acima_de": 0.15,
                      "classificadores": ["SVM", "XGBoost"]},
    "regra_de_leitura": {
        "RQ1a": "acrescenta se o p do bootstrap nulo (bilateral) for menor que 0,05",
        "RQ1b": "acrescenta se o p unilateral do bootstrap nulo (OR > 1) for menor que 0,05",
        "RQ1c": "consistente se a interação coorte × EEG tiver p ≥ 0,05 e os OR apontarem para o mesmo lado",
        "RQ2": "sobrevive se o IC95% da AUC (ou da concordância) sem confusão ficar acima de 0,5",
        "RQ3": "transporta se o IC95% da AUC na outra coorte ficar acima de 0,5",
        "resto": "não demonstrado (nunca 'o EEG não tem informação')"},
    "historico_alteracoes": [],   # se mudar algo depois de congelar: data + o quê + por quê
}

SEED = int(SPEC["semente"])
np.random.seed(SEED)
FEATURES = list(SPEC["features"]["todas"])
ESPECTRAIS = list(SPEC["features"]["espectrais"])
CANAIS = list(SPEC["canais_1020"])
RT = {"rf_trees": SPEC["classificadores"]["RandomForest"]["n_estimators"],
      "xgb_trees": SPEC["classificadores"]["XGBoost"]["n_estimators"],
      "n_boot": SPEC["validacao"]["bootstrap_n"],
      "n_desempates": SPEC["rq2"]["pareamento_brainlat"]["n_desempates_aleatorios"],
      "max_epocas": SPEC["preprocessamento"]["max_epocas_por_sujeito"],
      "boot_nulo_B": SPEC["rq1"]["bootstrap_nulo"]["B_max"]}
if RAPIDO:
    RT.update(rf_trees=40, xgb_trees=40, n_boot=200, n_desempates=20, max_epocas=20, boot_nulo_B=9)
TAG = "rapido" if RAPIDO else "completo"
assert SPEC["coortes"]["grego"]["versao_openneuro"] == DS_VERSAO, "versão do ds004504 diferente da baixada em 1.2b"

# ---- congelamento ----------------------------------------------------------
SPEC_HASH = hashlib.sha256(json.dumps(SPEC, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:16]
f_spec = OUT / "SPEC_journal.json"
if f_spec.exists():
    antigo = json.loads(f_spec.read_text(encoding="utf-8"))
    if antigo["hash"] != SPEC_HASH:
        mudou = sorted(k for k in set(SPEC) | set(antigo["spec"])
                       if json.dumps(SPEC.get(k), sort_keys=True, ensure_ascii=False)
                       != json.dumps(antigo["spec"].get(k), sort_keys=True, ensure_ascii=False))
        raise RuntimeError(
            f"A SPEC mudou depois de congelada em {antigo['congelada_em']} (chaves alteradas: {mudou}).\n"
            "Se a mudança foi decidida sem olhar resultados, registre data, o que mudou e por quê em "
            f"SPEC['historico_alteracoes'], apague {f_spec} e rode de novo.")
    SPEC_CONGELADA_EM = antigo["congelada_em"]
    print(f"SPEC confere com a versão congelada em {SPEC_CONGELADA_EM} (hash {SPEC_HASH}).")
else:
    SPEC_CONGELADA_EM = datetime.datetime.now().isoformat(timespec="seconds")
    f_spec.write_text(json.dumps({"hash": SPEC_HASH, "congelada_em": SPEC_CONGELADA_EM, "spec": SPEC},
                                 indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SPEC congelada agora ({SPEC_CONGELADA_EM}), hash {SPEC_HASH}: {f_spec}")

f_log = OUT / "log_execucao.json"
LOG = json.loads(f_log.read_text(encoding="utf-8")) if f_log.exists() else {}
LOG.setdefault("spec_congelada_em", SPEC_CONGELADA_EM)          # primeiro congelamento (fica para sempre)
CONGS = LOG.setdefault("congelamentos", [])
if not any(c["hash"] == SPEC_HASH for c in CONGS):
    CONGS.append({"hash": SPEC_HASH, "congelada_em": SPEC_CONGELADA_EM})
if len(CONGS) > 1:
    print(f"A SPEC já foi congelada {len(CONGS)} vezes: {CONGS}. O histórico vai para o resumo final.")
LOG.setdefault("execucoes", []).append({"inicio": INICIO, "rapido": RAPIDO, "spec_hash": SPEC_HASH})


def salvar_log():
    f_log.write_text(json.dumps(LOG, indent=2, ensure_ascii=False), encoding="utf-8")


salvar_log()
print(f"Execução: RF com {RT['rf_trees']} árvores | bootstrap {RT['n_boot']} | "
      f"máx. épocas por sujeito: {RT['max_epocas'] or 'todas'}")

### 1.4 Funções comuns: registro, formatação, AUC e cache

`auc_ci` e `delta_boot` são as mesmas funções do notebook do camera-ready (bootstrap estratificado de sujeitos), para os
intervalos serem comparáveis com o artigo da ACDSA.

In [ ]:
# 1.4 — Registro de resultados, formatação, AUC com IC, cache e figuras
REG, ALERTAS, VEREDITO = [], list(AVISOS_DOWNLOAD), {}


def alerta(msg):
    ALERTAS.append(msg)
    print("⚠️  ALERTA:", msg)


def registrar(rq, item, valor, onde="", nota=""):
    REG.append({"rq": rq, "item": item, "valor": valor, "onde": onde, "nota": nota})


def fmt(a, nd=3):
    try:
        return "n/d" if a is None or not np.isfinite(float(a)) else f"{float(a):.{nd}f}"
    except (TypeError, ValueError):
        return str(a)


def fmt_ci(a, lo, hi, nd=3):
    return f"{fmt(a, nd)} [{fmt(lo, nd)}, {fmt(hi, nd)}]"


def fmt_p(p):
    return "n/d" if p is None or not np.isfinite(p) else ("< 0.001" if p < 0.001 else f"{p:.3f}")


def logit_seguro(p):
    p = np.clip(np.asarray(p, float), 1e-3, 1 - 1e-3)
    return np.log(p / (1 - p))


def padronizar(M):
    M = np.asarray(M, float)
    sd = M.std(axis=0, ddof=1)
    sd = np.where(np.isfinite(sd) & (sd > 0), sd, 1.0)
    return (M - M.mean(axis=0)) / sd


def auc_ci(y, p, n_boot=None, seed=SEED):
    # AUC e IC95% por bootstrap estratificado de sujeitos (igual à do notebook do camera-ready)
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    boots = []
    for _ in range(n_boot):
        idx = np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]
        boots.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(roc_auc_score(y, p)), float(lo), float(hi)


def delta_boot(y, p1, p2, n_boot=None, seed=SEED):
    # ΔAUC (p1 − p2) nos MESMOS sujeitos, com IC por bootstrap pareado
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    y, p1, p2 = np.asarray(y).astype(int), np.asarray(p1, float), np.asarray(p2, float)
    d = roc_auc_score(y, p1) - roc_auc_score(y, p2)
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    diffs = []
    for _ in range(n_boot):
        idx = np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]
        diffs.append(roc_auc_score(y[idx], p1[idx]) - roc_auc_score(y[idx], p2[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(d), float(lo), float(hi)


def metricas_limiar(y, p, limiar=0.5):
    # Métricas com limiar fixo (descritivas), para comparar com a literatura do ds004504 (Miltiadous et al., 2026)
    y, yh = np.asarray(y).astype(int), (np.asarray(p, float) >= limiar).astype(int)
    vp, vn = int(((y == 1) & (yh == 1)).sum()), int(((y == 0) & (yh == 0)).sum())
    fp, fn = int(((y == 0) & (yh == 1)).sum()), int(((y == 1) & (yh == 0)).sum())
    sens = vp / max(vp + fn, 1)
    espec = vn / max(vn + fp, 1)
    den = np.sqrt(float((vp + fp) * (vp + fn) * (vn + fp) * (vn + fn)))
    return {"sens": sens, "espec": espec, "acc_bal": (sens + espec) / 2, "acc": (vp + vn) / max(len(y), 1),
            "F1": 2 * vp / max(2 * vp + fp + fn, 1), "MCC": (vp * vn - fp * fn) / den if den > 0 else 0.0}


def fmt_metricas(m):
    return (f"sens {m['sens']:.3f}, espec {m['espec']:.3f}, acurácia balanceada {m['acc_bal']:.3f}, "
            f"acurácia {m['acc']:.3f}, F1 {m['F1']:.3f}, MCC {m['MCC']:.3f}")


def fingerprint(df, cols):
    # impressão digital dos dados (números arredondados a 10 casas, para não depender do último bit)
    d = df[list(cols)].copy()
    num = d.select_dtypes("number").columns
    d[num] = d[num].astype(float).round(10)
    return hashlib.sha1(pd.util.hash_pandas_object(d, index=False).values.tobytes()).hexdigest()[:12]


def chave_cache(**kw):
    return hashlib.sha1(json.dumps(kw, sort_keys=True, default=str).encode()).hexdigest()[:16]


def cache_get(nome, chave):
    f = OUT / "cache" / f"{nome}_{chave}.pkl"
    if USAR_CACHE and f.exists():
        with open(f, "rb") as fh:
            return pickle.load(fh)
    return None


def cache_put(nome, chave, obj):
    with open(OUT / "cache" / f"{nome}_{chave}.pkl", "wb") as fh:
        pickle.dump(obj, fh)


def salvar_fig(fig, nome):
    # PDF (fontes TrueType embutidas) + PNG de pré-visualização; confere fontes Type 3
    pdf = OUT / "figuras" / f"{nome}.pdf"
    fig.savefig(pdf)
    fig.savefig(OUT / "figuras" / f"{nome}.png", dpi=200)
    if b"/Type3" in pdf.read_bytes():
        alerta(f"{pdf.name} contém fontes Type 3; confira rcParams['pdf.fonttype'].")
    return pdf


print("Funções comuns definidas.")

### 1.5 Regressão logística, teste incremental, E-value e correlação parcial

`logit_fit` e `ic_perfil` são as do camera-ready (Newton-Raphson, Firth quando há separação, IC por verossimilhança
perfilada). `teste_incremental` compara demografia com demografia + escore de EEG (razão de verossimilhança, 1 grau de
liberdade), como em Dinga et al. (2020). O E-value segue VanderWeele & Ding (2017), com RR ≈ √OR porque AD não é raro nas
amostras.

In [ ]:
# 1.5 — Logística (MLE/Firth), razão de verossimilhança, IC perfilado, E-value e Spearman parcial
def _sig(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -35, 35)))


def logit_fit(X, y, firth=False, fixos=None, max_iter=200, tol=1e-8):
    # Regressão logística por Newton-Raphson (X já com intercepto); firth=True: Firth (Heinze & Schemper, 2002);
    # fixos={coluna: valor}: coeficientes fixados (perfil de verossimilhança)
    X, y = np.asarray(X, float), np.asarray(y, float)
    n, k = X.shape
    fixos = fixos or {}
    livres = np.array([j for j in range(k) if j not in fixos], dtype=int)
    beta = np.zeros(k)
    for j, v in fixos.items():
        beta[j] = v

    def loglik(b):
        p = _sig(X @ b)
        ll = float(np.sum(y * np.log(np.clip(p, 1e-300, None)) + (1 - y) * np.log(np.clip(1 - p, 1e-300, None))))
        if firth:
            w = p * (1 - p)
            sign, logdet = np.linalg.slogdet(X.T @ (X * w[:, None]))
            ll += 0.5 * logdet if sign > 0 else -np.inf
        return ll

    ll, convergiu, it = loglik(beta), False, 0
    for it in range(1, max_iter + 1):
        p = _sig(X @ beta)
        w = p * (1 - p)
        info = X.T @ (X * w[:, None])
        if firth:
            info_inv = np.linalg.pinv(info)
            h = np.einsum("ij,jk,ik->i", X, info_inv, X) * w
            score = X.T @ (y - p + h * (0.5 - p))
        else:
            score = X.T @ (y - p)
        sub = info[np.ix_(livres, livres)]
        try:
            passo = np.linalg.solve(sub, score[livres])
        except np.linalg.LinAlgError:
            passo = np.linalg.lstsq(sub, score[livres], rcond=None)[0]
        maior = np.max(np.abs(passo)) if len(passo) else 0.0
        if maior > 5:
            passo = passo * 5 / maior
        t = 1.0
        while True:
            b_new = beta.copy()
            b_new[livres] += t * passo
            ll_new = loglik(b_new)
            if ll_new >= ll - 1e-10 or t < 1e-6:
                break
            t /= 2
        dif = np.max(np.abs(b_new - beta)) if len(passo) else 0.0
        beta, ll = b_new, ll_new
        if dif < tol:
            convergiu = True
            break
    p = _sig(X @ beta)
    info = X.T @ (X * (p * (1 - p))[:, None])
    try:
        cov = np.linalg.inv(info)
    except np.linalg.LinAlgError:
        cov = np.full((k, k), np.nan)
    return {"beta": beta, "ll": ll, "cov": cov, "convergiu": convergiu, "n_iter": it, "p": p}


def ic_perfil(Xd, yv, j, firth, fit_full, nivel=0.95):
    # IC por verossimilhança (penalizada, se Firth) perfilada para o coeficiente j
    alvo = stats.chi2.ppf(nivel, 1)
    b_hat, ll_max = fit_full["beta"][j], fit_full["ll"]
    v = fit_full["cov"][j, j]
    se = float(np.sqrt(v)) if np.isfinite(v) and v > 0 else 0.5

    def f(b):
        return 2 * (ll_max - logit_fit(Xd, yv, firth=firth, fixos={j: b})["ll"]) - alvo

    lims = []
    for sinal in (-1, 1):
        passo, k = max(se, 0.05), 0
        b_out = b_hat + sinal * passo
        while f(b_out) < 0 and k < 25:
            passo *= 1.8
            b_out = b_hat + sinal * passo
            k += 1
        try:
            lims.append(brentq(f, min(b_hat, b_out), max(b_hat, b_out), xtol=1e-6))
        except Exception:
            lims.append(np.nan)
    return lims[0], lims[1]


def lr_aninhado(X_red, X_full, yv, j):
    # Razão de verossimilhança (1 gl) para a coluna j de X_full, com a regra de Firth da SPEC
    res = {}
    for metodo, firth in (("MLE", False), ("Firth", True)):
        fr, ff = logit_fit(X_red, yv, firth=firth), logit_fit(X_full, yv, firth=firth)
        lr = max(2 * (ff["ll"] - fr["ll"]), 0.0)
        lo_b, hi_b = ic_perfil(X_full, yv, j, firth, ff)
        b = float(ff["beta"][j])
        p2 = float(stats.chi2.sf(lr, 1))
        res[metodo] = {"LR": lr, "p": p2, "p_unilateral": p2 / 2 if b > 0 else 1 - p2 / 2, "beta": b,
                       "OR": float(np.exp(b)),
                       "OR_lo": float(np.exp(lo_b)) if np.isfinite(lo_b) else np.nan,
                       "OR_hi": float(np.exp(hi_b)) if np.isfinite(hi_b) else np.nan,
                       "convergiu": bool(fr["convergiu"] and ff["convergiu"]),
                       "max_beta": float(max(np.max(np.abs(fr["beta"][1:])), np.max(np.abs(ff["beta"][1:]))))}
    separacao = (not res["MLE"]["convergiu"]) or res["MLE"]["max_beta"] > 10
    metodo = "Firth" if separacao else "MLE"
    return {"metodo": metodo, **res[metodo]}


def teste_incremental(tab, covs, col_p, rotulo):
    # Escore EEG (logit de P(AD), padronizado na coorte) além das covariáveis
    covs = list(covs)
    t = tab.dropna(subset=covs + [col_p, "y"]).reset_index(drop=True)
    yv = t["y"].to_numpy(int)
    z = padronizar(logit_seguro(t[col_p])[:, None])[:, 0]
    X_red = np.column_stack([np.ones(len(yv)), padronizar(t[covs].to_numpy(float))])
    X_full = np.column_stack([X_red, z])
    r = lr_aninhado(X_red, X_full, yv, X_full.shape[1] - 1)
    return {"coorte": rotulo, "covariaveis": " + ".join(covs), "n": len(yv), "n_AD": int(yv.sum()),
            "n_HC": int((1 - yv).sum()), **r}


def e_value(or_, lo, hi, desfecho_comum=True):
    # E-value (VanderWeele & Ding, 2017) para o OR e para o limite do IC mais próximo de 1
    def rr(o):
        return np.sqrt(o) if desfecho_comum else o

    def ev(r):
        r = 1 / r if r < 1 else r
        return float(r + np.sqrt(r * (r - 1)))

    if not np.isfinite(or_):
        return np.nan, np.nan
    e_pt = ev(rr(or_))
    if or_ >= 1:
        e_ci = 1.0 if (not np.isfinite(lo) or lo <= 1) else ev(rr(lo))
    else:
        e_ci = 1.0 if (not np.isfinite(hi) or hi >= 1) else ev(rr(hi))
    return e_pt, e_ci


def spearman_parcial(x, y, Z=None):
    # ρ de Spearman parcial: correlação dos resíduos dos postos de x e y regredidos nos postos de Z
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    if Z is not None and np.asarray(Z).shape[1] > 0:
        Z = np.asarray(Z, float)
        R = np.column_stack([np.ones(len(rx))] + [stats.rankdata(Z[:, k]) for k in range(Z.shape[1])])
        rx = rx - R @ np.linalg.lstsq(R, rx, rcond=None)[0]
        ry = ry - R @ np.linalg.lstsq(R, ry, rcond=None)[0]
    if np.std(rx) == 0 or np.std(ry) == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def spearman_parcial_ci(x, y, Z=None, n_boot=None, seed=SEED):
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    x, y = np.asarray(x, float), np.asarray(y, float)
    Z = None if Z is None else np.asarray(Z, float)
    rho = spearman_parcial(x, y, Z)
    k = 0 if Z is None else Z.shape[1]
    df = len(x) - 2 - k
    p = (float(2 * stats.t.sf(abs(rho * np.sqrt(df / max(1 - rho ** 2, 1e-12))), df))
         if np.isfinite(rho) and df > 0 else np.nan)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        r = spearman_parcial(x[i], y[i], None if Z is None else Z[i])
        if np.isfinite(r):
            boots.append(r)
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return rho, float(lo), float(hi), p


print("Funções de inferência definidas.")

### 1.6 Funções de desenho: escore de propensão, overlap weights, balanço, pareamento e concordância

Mesmas funções do camera-ready, generalizadas para as covariáveis de cada coorte.

In [ ]:
# 1.6 — Escore de propensão, overlap weights, balanço, pareamento 1:1 e concordância em pares comparáveis
def escore_propensao(tab, covs):
    # P(AD | covs) por logística; Firth se a MLE não convergir ou houver separação
    Xd = np.column_stack([np.ones(len(tab)), padronizar(tab[list(covs)].to_numpy(float))])
    fit, metodo = logit_fit(Xd, tab["y"].to_numpy(int)), "MLE"
    if (not fit["convergiu"]) or np.max(np.abs(fit["beta"][1:])) > 10:
        fit, metodo = logit_fit(Xd, tab["y"].to_numpy(int), firth=True), "Firth"
    return _sig(Xd @ fit["beta"]), metodo


def ess(w):
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / np.sum(w ** 2)) if np.sum(w ** 2) > 0 else np.nan


def auc_ponderada(yv, pv, wv):
    yv, pv, wv = np.asarray(yv, int), np.asarray(pv, float), np.asarray(wv, float)
    pa, ph, wa, wh = pv[yv == 1], pv[yv == 0], wv[yv == 1], wv[yv == 0]
    K = (pa[:, None] > ph[None, :]) + 0.5 * (pa[:, None] == ph[None, :])
    return float(wa @ K @ wh / (wa.sum() * wh.sum()))


def auc_ponderada_ci(tab, covs, col_p, n_boot=None, seed=SEED):
    # AUC com overlap weights; IC por bootstrap estratificado com o escore de propensão re-estimado em cada réplica
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    t = tab.dropna(subset=list(covs) + [col_p, "y"]).reset_index(drop=True)
    ps, metodo = escore_propensao(t, covs)
    w = np.where(t["y"] == 1, 1 - ps, ps)
    a = auc_ponderada(t["y"], t[col_p], w)
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(t.y == 1)[0], np.where(t.y == 0)[0]
    bw = []
    for _ in range(n_boot):
        tb = t.iloc[np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]].reset_index(drop=True)
        ps_b, _ = escore_propensao(tb, covs)
        bw.append(auc_ponderada(tb["y"], tb[col_p], np.where(tb["y"] == 1, 1 - ps_b, ps_b)))
    lo, hi = np.percentile(bw, [2.5, 97.5])
    yy = t["y"].to_numpy(int)
    return {"AUC": a, "lo": float(lo), "hi": float(hi), "metodo_ps": metodo, "n": len(t),
            "ess_AD": ess(w[yy == 1]), "ess_HC": ess(w[yy == 0])}


def aauc_controles(tab, covs, col_p="p_eeg", n_boot=None, seed=SEED):
    # AUC ajustada por covariáveis (Janes & Pepe, 2009): o escore (logit) é padronizado pela regressão linear nos
    # CONTROLES; a AUC dos resíduos estima P(escore_AD > escore_HC | mesmas covariáveis). IC: bootstrap estratificado
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    t = tab.dropna(subset=list(covs) + [col_p, "y"]).reset_index(drop=True)

    def est(tt):
        m = logit_seguro(tt[col_p])
        Z = np.column_stack([np.ones(len(tt)), tt[list(covs)].to_numpy(float)])
        hc = tt["y"].to_numpy(int) == 0
        beta = np.linalg.lstsq(Z[hc], m[hc], rcond=None)[0]
        return float(roc_auc_score(tt["y"], m - Z @ beta))

    a = est(t)
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(t.y == 1)[0], np.where(t.y == 0)[0]
    boots = [est(t.iloc[np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]].reset_index(drop=True))
             for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"AUC": a, "lo": float(lo), "hi": float(hi), "n": len(t)}


def balanco(tab, variaveis, w=None):
    # Médias por grupo e diferença padronizada (SMD), com pesos opcionais. variaveis = {coluna: rótulo}
    yy = tab["y"].to_numpy(int)
    ww = np.ones(len(tab)) if w is None else np.asarray(w, float)
    linhas = []
    for var, nome in variaveis.items():
        if var not in tab.columns:
            continue
        x = tab[var].to_numpy(float)
        ok = np.isfinite(x)
        a, h = ok & (yy == 1), ok & (yy == 0)
        if a.sum() == 0 or h.sum() == 0:
            continue
        m1, m0 = np.average(x[a], weights=ww[a]), np.average(x[h], weights=ww[h])
        v1 = np.average((x[a] - m1) ** 2, weights=ww[a])
        v0 = np.average((x[h] - m0) ** 2, weights=ww[h])
        linhas.append({"variável": nome, "AD": m1, "HC": m0,
                       "SMD": (m1 - m0) / np.sqrt((v1 + v0) / 2) if v1 + v0 > 0 else 0.0})
    return pd.DataFrame(linhas)


def parear_1a1(tab, cal, var="years_education", desempate="Age", rng=None):
    # Pareamento ótimo 1:1 sem reposição com calibre; rng≠None: desempate aleatório entre pares válidos
    ad = tab[tab.y == 1].dropna(subset=[var]).reset_index(drop=True)
    hc = tab[tab.y == 0].dropna(subset=[var]).reset_index(drop=True)
    vazio = pd.DataFrame(columns=["par", "ad_id", "hc_id", "var_ad", "var_hc", "idade_ad", "idade_hc"])
    if len(ad) == 0 or len(hc) == 0:
        return vazio
    de = np.abs(ad[var].to_numpy(float)[:, None] - hc[var].to_numpy(float)[None, :])
    if rng is None:
        da = np.nan_to_num(np.abs(ad[desempate].to_numpy(float)[:, None]
                                  - hc[desempate].to_numpy(float)[None, :]), nan=99.0)
        custo = np.where(de <= cal, de + 1e-4 * da, 1e6)
    else:
        custo = np.where(de <= cal, rng.random(de.shape), 1e6)
    r, c = linear_sum_assignment(custo)
    ok = custo[r, c] < 1e6
    r, c = r[ok], c[ok]
    if len(r) == 0:
        return vazio
    return pd.DataFrame({
        "par": np.arange(len(r)), "ad_id": ad["subject_id"].to_numpy()[r], "hc_id": hc["subject_id"].to_numpy()[c],
        "var_ad": ad[var].to_numpy()[r], "var_hc": hc[var].to_numpy()[c],
        "idade_ad": ad["Age"].to_numpy()[r], "idade_hc": hc["Age"].to_numpy()[c]})


def auc_pareada_ci(pares, P, n_boot=None, seed=SEED):
    # AUC no subconjunto pareado; IC por bootstrap de pares (Austin & Small, 2014)
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    pa, ph = P.loc[list(pares.ad_id)].to_numpy(float), P.loc[list(pares.hc_id)].to_numpy(float)
    n = len(pa)
    yv = np.r_[np.ones(n), np.zeros(n)]
    a = float(roc_auc_score(yv, np.r_[pa, ph]))
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        boots.append(roc_auc_score(yv, np.r_[pa[i], ph[i]]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return a, float(lo), float(hi)


def concordancia_pares(tab_p, elegivel, col_p="p_eeg", n_boot=None, seed=SEED):
    # P(P_AD > P_HC) entre os pares AD–HC elegíveis; IC por bootstrap de sujeitos dentro de cada grupo
    n_boot = RT["n_boot"] if n_boot is None else n_boot
    ad = tab_p[tab_p.y == 1].reset_index(drop=True)
    hc = tab_p[tab_p.y == 0].reset_index(drop=True)
    Mm = np.asarray(elegivel(ad, hc), float)
    if Mm.sum() == 0:
        return {"C": np.nan, "lo": np.nan, "hi": np.nan, "n_pares": 0, "n_AD": 0, "n_HC": 0}
    pa, ph = ad[col_p].to_numpy(float), hc[col_p].to_numpy(float)
    K = (pa[:, None] > ph[None, :]) + 0.5 * (pa[:, None] == ph[None, :])
    KM = K * Mm
    C = float(KM.sum() / Mm.sum())
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        a = np.bincount(rng.integers(0, len(pa), len(pa)), minlength=len(pa)).astype(float)
        b = np.bincount(rng.integers(0, len(ph), len(ph)), minlength=len(ph)).astype(float)
        den = a @ Mm @ b
        if den > 0:
            boots.append((a @ KM @ b) / den)
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return {"C": C, "lo": float(lo), "hi": float(hi), "n_pares": int(Mm.sum()),
            "n_AD": int((Mm.sum(axis=1) > 0).sum()), "n_HC": int((Mm.sum(axis=0) > 0).sum())}


print("Funções de desenho definidas.")

### 1.7 Modelos, LOSO e transporte

As fábricas de modelos e o LOSO são os do camera-ready (treino por época, P(AD) do sujeito = média das épocas). Em
`transportar`, o modelo é treinado em toda a coorte de origem e aplicado **uma única vez** à outra, sem olhar os rótulos
dela; o resultado fica gravado com o hash do modelo e não é reaplicado em execuções seguintes.

In [ ]:
# 1.7 — Fábricas de modelos, LOSO por sujeito, transporte entre coortes e logística LOO no nível do sujeito
def positive_class_score(model, X):
    # Probabilidade da classe AD (rótulo 1), pela coluna correta de predict_proba
    proba = model.predict_proba(X)
    classes = np.asarray(getattr(model, "classes_", [0, 1]))
    if 1 in classes:
        return proba[:, int(np.where(classes == 1)[0][0])]
    return np.zeros(len(X))


def fabrica_rf(y_tr=None, g_tr=None):
    c = SPEC["classificadores"]["RandomForest"]
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", RandomForestClassifier(
                         n_estimators=RT["rf_trees"], max_depth=c["max_depth"], max_features=c["max_features"],
                         min_samples_leaf=c["min_samples_leaf"], bootstrap=c["bootstrap"],
                         class_weight=c["class_weight"], random_state=c["random_state"], n_jobs=N_JOBS))])


def fabrica_xgb(y_tr, g_tr=None):
    c = SPEC["classificadores"]["XGBoost"]
    y_tr = np.asarray(y_tr)
    spw = float(np.sum(y_tr == 0)) / max(float(np.sum(y_tr == 1)), 1.0)   # só o treino do fold
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", XGBClassifier(
                         n_estimators=RT["xgb_trees"], learning_rate=c["learning_rate"], max_depth=c["max_depth"],
                         subsample=c["subsample"], colsample_bytree=c["colsample_bytree"],
                         eval_metric=c["eval_metric"], scale_pos_weight=spw,
                         random_state=c["random_state"], n_jobs=N_JOBS, verbosity=0))])


def fabrica_svm(y_tr, g_tr):
    c = SPEC["classificadores"]["SVM"]
    base = SVC(kernel=c["kernel"], C=c["C"], gamma=c["gamma"], random_state=c["random_state"])
    if c["calibracao_agrupada_por_sujeito"]:
        cv = list(StratifiedGroupKFold(n_splits=c["folds_calibracao"]).split(
            np.zeros(len(y_tr)), np.asarray(y_tr), np.asarray(g_tr)))
    else:
        cv = c["folds_calibracao"]
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", CalibratedClassifierCV(base, cv=cv, method=c["calibracao"]))])


FABRICAS = {"RandomForest": fabrica_rf, "SVM": fabrica_svm}
if XGB_OK:
    FABRICAS["XGBoost"] = fabrica_xgb


def run_loso(df, feats, modelo, nome, verbose=True):
    # LOSO por sujeito; retorna DataFrame(subject_id, y, p, n_epochs, fold). Cache por nome + dados + parâmetros.
    feats = list(feats)
    X = df[feats].to_numpy(float)
    y = df["y"].to_numpy(int)
    g = df["subject_id"].astype(str).to_numpy()
    chave = chave_cache(nome=nome, feats=feats, modelo=modelo, dados=fingerprint(df, feats + ["y", "subject_id"]),
                        cfg=SPEC["classificadores"][modelo], rt=RT)
    hit = cache_get(nome, chave)
    if hit is not None:
        if verbose:
            print(f"[cache] {nome}: {len(hit['pred'])} sujeitos (rodado antes em {hit['segundos']:.0f} s)")
        return hit["pred"]
    folds = list(LeaveOneGroupOut().split(X, y, g))
    rows, t0 = [], time.time()
    for k, (tr, te) in enumerate(folds):
        if len(np.unique(y[tr])) < 2:
            s = np.full(len(te), float(y[tr][0]))
        else:
            m = FABRICAS[modelo](y[tr], g[tr])
            m.fit(X[tr], y[tr])
            s = positive_class_score(m, X[te])
        rows.append({"subject_id": g[te][0], "y": int(y[te][0]), "p": float(np.mean(s)),
                     "n_epochs": int(len(te)), "fold": k})
        if verbose and ((k + 1) % 10 == 0 or k + 1 == len(folds)):
            dt = time.time() - t0
            print(f"  {nome}: {k + 1}/{len(folds)} folds | {dt:.0f} s | faltam ~{dt / (k + 1) * (len(folds) - k - 1):.0f} s")
    pred = pd.DataFrame(rows)
    cache_put(nome, chave, {"pred": pred, "segundos": time.time() - t0})
    return pred


def transportar(df_tr, df_te, feats, modelo, nome):
    # Treina em toda a coorte de origem e aplica UMA vez na outra (sem usar os rótulos dela)
    feats = list(feats)
    meta = {"nome": nome, "modelo": modelo, "feats": feats, "spec_hash": SPEC_HASH, "rt": RT,
            "dados_treino": fingerprint(df_tr, feats + ["y", "subject_id"]),
            "dados_teste": fingerprint(df_te, feats + ["subject_id"])}
    f_json = OUT / "modelos" / f"{nome}_{TAG}.json"
    f_pred = OUT / "modelos" / f"{nome}_{TAG}_predicoes.csv"
    if f_json.exists() and f_pred.exists():
        antigo = json.loads(f_json.read_text(encoding="utf-8"))
        if all(antigo.get(k) == meta[k] for k in ("modelo", "feats", "spec_hash", "dados_treino", "dados_teste")):
            print(f"[já aplicado em {antigo['aplicado_em']}] {nome}: não reaplicado")
            return pd.read_csv(f_pred, dtype={"subject_id": str})
        alerta(f"{nome}: dados, features ou SPEC mudaram desde a aplicação de {antigo['aplicado_em']}; reaplicado.")
    Xtr, ytr = df_tr[feats].to_numpy(float), df_tr["y"].to_numpy(int)
    m = FABRICAS[modelo](ytr, df_tr["subject_id"].astype(str).to_numpy()).fit(Xtr, ytr)
    f_mod = OUT / "modelos" / f"{nome}_{TAG}.pkl"
    with open(f_mod, "wb") as fh:
        pickle.dump(m, fh)
    pe = positive_class_score(m, df_te[feats].to_numpy(float))
    pred = (pd.DataFrame({"subject_id": df_te["subject_id"].astype(str).to_numpy(), "p": pe})
            .groupby("subject_id", sort=True)["p"].mean().reset_index())
    pred.to_csv(f_pred, index=False)
    meta.update({"aplicado_em": datetime.datetime.now().isoformat(timespec="seconds"),
                 "sha1_modelo": hashlib.sha1(f_mod.read_bytes()).hexdigest()[:12], "n_sujeitos_teste": len(pred)})
    f_json.write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    return pred


def bootstrap_nulo(df_ep, subj, covs, nome, unilateral=False, modelo="RandomForest", feats=None):
    # p-valor do escore de EEG além das covariáveis por bootstrap paramétrico sob H0 (Shang et al., 2025, adaptado a
    # desfecho binário): simula y* do modelo nulo P(AD | covariáveis), refaz TODO o LOSO com y* e recalcula a estatística.
    # Mantém a correlação entre EEG e covariáveis, que é o que torna o teste ingênuo liberal. Cada réplica fica em cache.
    feats = FEATURES if feats is None else list(feats)
    h_parar = SPEC["rq1"]["bootstrap_nulo"]["parar_apos_excedencias"]
    B = RT["boot_nulo_B"]
    t = subj.dropna(subset=list(covs) + ["p_eeg"]).reset_index(drop=True)
    obs = teste_incremental(t, covs, "p_eeg", nome)

    def estat(r):
        return float(np.sign(r["beta"]) * np.sqrt(r["LR"])) if unilateral else float(r["LR"])

    T_obs = estat(obs)
    Xc = np.column_stack([np.ones(len(t)), padronizar(t[list(covs)].to_numpy(float))])
    fit = logit_fit(Xc, t["y"].to_numpy(int))
    if (not fit["convergiu"]) or np.max(np.abs(fit["beta"][1:])) > 10:
        fit = logit_fit(Xc, t["y"].to_numpy(int), firth=True)
    p0 = _sig(Xc @ fit["beta"])
    Ystar = (np.random.default_rng(SEED + 1000).random((B, len(t))) < p0[None, :]).astype(int)
    ids = t["subject_id"].astype(str).to_numpy()
    sid_ep = df_ep["subject_id"].astype(str).to_numpy()
    y_ep = df_ep["y"].to_numpy(int)
    T, excede, t0 = [], 0, time.time()
    for b in range(B):
        mapa = dict(zip(ids, Ystar[b]))
        d = df_ep.copy()
        d["y"] = [int(mapa.get(s, yy)) for s, yy in zip(sid_ep, y_ep)]    # sem covariáveis: mantém o rótulo observado
        pred = run_loso(d, feats, modelo, f"{nome}_nulo", verbose=False)
        s_ = t[["subject_id"] + list(covs)].merge(pred[["subject_id", "y", "p"]], on="subject_id")
        Tb = estat(teste_incremental(s_, covs, "p", nome))
        T.append(Tb)
        excede += int(Tb >= T_obs)
        if (b + 1) % 10 == 0 or excede >= h_parar or b + 1 == B:
            dt = time.time() - t0
            print(f"  bootstrap nulo {nome}: {b + 1}/{B} réplicas | {excede} ≥ observado | {dt / 60:.1f} min")
        if excede >= h_parar:          # Besag & Clifford (1991): p claramente acima de 0,05
            break
    nb = len(T)
    p = excede / nb if excede >= h_parar else (excede + 1) / (nb + 1)
    return {"T_obs": T_obs, "T": np.asarray(T), "p": float(p), "replicas": nb, "excedencias": excede,
            "parou_cedo": excede >= h_parar, "unilateral": unilateral, "naive": obs}


def loo_lr_sujeito(tab, cols):
    # Logística balanceada em leave-one-subject-out no nível do sujeito (como no camera-ready)
    X, yv = tab[cols].to_numpy(float), tab["y"].to_numpy(int)
    p = np.zeros(len(yv))
    for i in range(len(yv)):
        tr = np.arange(len(yv)) != i
        mdl = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LogisticRegression(max_iter=5000, class_weight="balanced"))
        mdl.fit(X[tr], yv[tr])
        p[i] = mdl.predict_proba(X[i:i + 1])[0, 1]
    return pd.DataFrame({"subject_id": tab["subject_id"].to_numpy(), "y": yv, "p": p})


print(f"Modelos disponíveis: {list(FABRICAS)}")

---
## 2. Coortes

### 2.1 BrainLat: arquivos, rótulos, sítio, demografia e cognição

Mesma leitura do camera-ready. O sexo usa o código do dicionário do BrainLat (0 = feminino). Quem não tiver EEG legível
(por exemplo, os controles do Chile sem o `.fdt`) sai na seção 3, e isso fica registrado.

In [ ]:
# 2.1 — BrainLat: tabela por sujeito
def extract_subject_id(fpath):
    fpath = Path(fpath)
    m = re.search(r"(sub-\d+)", fpath.stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    m = re.search(r"sub[-_]?(\d+)", fpath.stem, flags=re.IGNORECASE)
    return f"sub-{m.group(1)}" if m else fpath.stem


def infer_country(fpath):
    parts = {p.upper() for p in Path(fpath).parts}
    return "Argentina" if "AR" in parts else "Chile" if "CL" in parts else "Other"


def achar_csv(pasta, nome):
    pasta = Path(pasta)
    if (pasta / nome).exists():
        return pasta / nome
    cand = sorted(pasta.rglob(nome)) if pasta.exists() else []
    return cand[0] if cand else None


def find_cognitive_csv(folder):
    if folder is None or not Path(folder).exists():
        return None
    for f in sorted(Path(folder).rglob("*.csv")):
        try:
            cols = [c.strip().lower() for c in pd.read_csv(f, nrows=0).columns]
        except Exception:
            continue
        if "moca_total" in cols or "ifs_total_score" in cols:
            return f
    return None


def read_cognitive(f):
    d = pd.read_csv(f)
    d.columns = d.columns.str.strip().str.lower()
    for id_col in ("id eeg", "subject_id", "id"):
        if id_col in d.columns:
            d = d.rename(columns={id_col: "subject_id"})
            break
    d["subject_id"] = d["subject_id"].astype(str).str.strip()
    return d[["subject_id"] + [c for c in ("moca_total", "ifs_total_score") if c in d.columns]]


arq = []
for grupo, pasta in (("AD", DIR_AD), ("HC", DIR_HC)):
    for f in sorted(pasta.rglob("*.set")):
        arq.append({"subject_id": extract_subject_id(f), "y": SPEC["rotulos"][grupo],
                    "site": infer_country(f), "arquivo": str(f)})
arq_bl = pd.DataFrame(arq)
inconsist = arq_bl.groupby("subject_id")["y"].nunique()
if (inconsist > 1).any():
    raise ValueError(f"Sujeitos nas duas pastas: {inconsist[inconsist > 1].index.tolist()}")
dup = arq_bl[arq_bl.subject_id.duplicated(keep="first")]
if len(dup):
    alerta(f"{len(dup)} arquivo(s) .set repetido(s) para o mesmo sujeito; usado o primeiro: {dup.subject_id.tolist()[:5]}")
    arq_bl = arq_bl.drop_duplicates("subject_id", keep="first")

f_demo = {"AD": achar_csv(DIR_AD, "demographics_ad_eeg_data.csv"),
          "HC": achar_csv(DIR_HC, "demographics_hc_eeg_data.csv")}
if any(v is None for v in f_demo.values()):
    raise FileNotFoundError(f"Demografia não encontrada: {f_demo}. Confira BRAINLAT_DIR em 1.2.")
demo_all = pd.concat([pd.read_csv(f).assign(grupo_pasta=g) for g, f in f_demo.items()], ignore_index=True)
demo_all = demo_all.rename(columns={"id EEG": "subject_id"})
demo_all["subject_id"] = demo_all["subject_id"].astype(str).str.strip()
cog_files = [f for f in (find_cognitive_csv(DIR_AD), find_cognitive_csv(DIR_HC)) if f is not None]
df_cog = (pd.concat([read_cognitive(f) for f in cog_files], ignore_index=True).drop_duplicates("subject_id")
          if cog_files else pd.DataFrame(columns=["subject_id", "moca_total", "ifs_total_score"]))
for c in ("moca_total", "ifs_total_score"):
    if c not in df_cog.columns:
        df_cog[c] = np.nan

subj_bl = (arq_bl.merge(demo_all[["subject_id", "sex", "Age", "years_education"]].drop_duplicates("subject_id"),
                        on="subject_id", how="left")
                 .merge(df_cog[["subject_id", "moca_total", "ifs_total_score"]], on="subject_id", how="left"))
for c in ("sex", "Age", "years_education", "moca_total", "ifs_total_score"):
    subj_bl[c] = pd.to_numeric(subj_bl[c], errors="coerce")
FEM_BL = SPEC["coortes"]["brainlat"]["sexo_valor_feminino"]
subj_bl["sex_fem"] = np.where(subj_bl["sex"].isna(), np.nan, (subj_bl["sex"] == FEM_BL).astype(float))
subj_bl["site_cl"] = (subj_bl["site"] == "Chile").astype(float)
subj_bl["coorte"] = "BrainLat"
subj_bl = subj_bl.sort_values("subject_id").reset_index(drop=True)

print(f"BrainLat: {len(subj_bl)} arquivos .set ({int(subj_bl.y.sum())} AD, {int((1 - subj_bl.y).sum())} HC)")
display(pd.crosstab(subj_bl["y"].map({1: "AD", 0: "HC"}), subj_bl["site"], margins=True))
print(f"Com idade, sexo e escolaridade: {int(subj_bl[['Age', 'sex', 'years_education']].notna().all(axis=1).sum())}"
      f" | arquivos de cognição: {[f.name for f in cog_files]}")
print("Código de sexo (contagens brutas por grupo):")
display(pd.crosstab(subj_bl["y"].map({1: "AD", 0: "HC"}), subj_bl["sex"]))

### 2.2 ds004504: participantes AD e controles

> **Atenção:** é aqui que os rótulos do banco grego são lidos pela primeira vez. A SPEC já está congelada (1.3), e a hora da
> primeira leitura fica gravada em `log_execucao.json`. Os participantes com demência frontotemporal (grupo F) saem.

In [ ]:
# 2.2 — ds004504: participants.tsv (AD = A, controles = C) e arquivos .set (derivatives/ primeiro)
GRUPOS_GR = SPEC["coortes"]["grego"]["grupos"]
part = pd.read_csv(DS_DIR / "participants.tsv", sep="\t")
part.columns = [c.strip().lower() for c in part.columns]
faltam_col = {"participant_id", "group", "age", "gender", "mmse"} - set(part.columns)
if faltam_col:
    raise ValueError(f"participants.tsv sem as colunas {faltam_col}: confira se é o ds004504.")
part["group"] = part["group"].astype(str).str.strip().str.upper()
print("Grupos no participants.tsv:", part["group"].value_counts().to_dict())
part = part[part["group"].isin(GRUPOS_GR)].copy()
part["y"] = part["group"].map(GRUPOS_GR).astype(int)
part["sex_fem"] = part["gender"].astype(str).str.strip().str.upper().str[0].map({"F": 1.0, "M": 0.0})
part["Age"] = pd.to_numeric(part["age"], errors="coerce")
part["mmse"] = pd.to_numeric(part["mmse"], errors="coerce")
TEM_ESCOL_GR = any("educ" in c for c in part.columns)


def achar_set_ds(pid):
    bases = ([DS_DIR / "derivatives" / pid / "eeg"] if SPEC["coortes"]["grego"]["usar_derivatives"] else []) + \
            [DS_DIR / pid / "eeg"]
    for b in bases:
        cand = sorted(b.glob("*.set")) if b.exists() else []
        if cand:
            return str(cand[0])
    return None


part["arquivo"] = [achar_set_ds(p) for p in part["participant_id"]]
part["origem"] = ["ausente" if not isinstance(f, str) else "derivatives" if "derivatives" in Path(f).parts
                  else "bruto" for f in part["arquivo"]]
subj_gr = (part.rename(columns={"participant_id": "subject_id"})
               [["subject_id", "y", "Age", "sex_fem", "mmse", "arquivo", "origem"]]
               .assign(coorte="Grego").sort_values("subject_id").reset_index(drop=True))
subj_gr["subject_id"] = subj_gr["subject_id"].astype(str).str.strip()

if "rotulos_gregos_lidos_em" not in LOG:
    LOG["rotulos_gregos_lidos_em"] = datetime.datetime.now().isoformat(timespec="seconds")
    salvar_log()
print(f"SPEC congelada em {LOG['spec_congelada_em']} | rótulos gregos lidos pela primeira vez em "
      f"{LOG['rotulos_gregos_lidos_em']}")
if LOG["rotulos_gregos_lidos_em"] < LOG["spec_congelada_em"]:
    alerta("Os rótulos gregos foram lidos ANTES do congelamento da SPEC registrado neste log.")

print(f"\nds004504: {len(subj_gr)} participantes ({int(subj_gr.y.sum())} AD, {int((1 - subj_gr.y).sum())} controles)")
print("Origem dos arquivos:", subj_gr["origem"].value_counts().to_dict())
print(f"Escolaridade nos metadados: {'sim' if TEM_ESCOL_GR else 'não (como esperado: E-value em 5.2)'}")
sem_arquivo = subj_gr[subj_gr.arquivo.isna()]
if len(sem_arquivo):
    alerta(f"{len(sem_arquivo)} participante(s) do ds004504 sem .set: {sem_arquivo.subject_id.tolist()}")

---
## 3. Features harmonizadas (19 canais, 11 features)

### 3.1 Funções de extração

As funções das features são as do artigo (notebook 2 → camera-ready), sem mudança. O que muda é só a montagem: as duas
coortes passam pelas mesmas 19 posições 10-20, com referência média desses 19 canais, FIR de 0,5–45 Hz, épocas de 4 s,
normalização RMS por sujeito e Welch com resolução de 1 Hz. A conectividade passa a ser a média de 171 pares (19 canais).

**Banda passante igualada.** O BrainLat foi distribuído já filtrado em 0,5–40 Hz (Butterworth de ordem 8, fase zero;
Prado et al., 2023), e os `derivatives` do ds004504 em 0,5–45 Hz (README do banco). Sem igualar, a potência entre 40 e
45 Hz existiria só no grego e mudaria a gama relativa e a entropia espectral por causa do pré-processamento, não da doença.
Por isso o ds004504 recebe o mesmo passa-baixa de 40 Hz antes das features. Diferenças que não dá para igualar (aparelho,
referência original, ICA só ocular no BrainLat e ocular + mandíbula no grego, ASR no grego) aparecem na tabela S2 e são
tratadas na RQ3.

In [ ]:
# 3.1 — Funções de pré-processamento e extração (lógica idêntica ao artigo)
BANDS = {"Delta": (0.5, 4.0), "Theta": (4.0, 8.0), "Alpha": (8.0, 13.0), "Beta": (13.0, 30.0), "Gamma": (30.0, 45.0)}
FMIN, FMAX = 0.5, 45.0
CONN_BANDS = {"Theta": (4.0, 8.0), "Alpha": (8.0, 13.0), "Beta": (13.0, 30.0)}
EPOCA_S = float(SPEC["preprocessamento"]["epoca_s"])
RES_HZ = float(SPEC["preprocessamento"]["welch_resolucao_hz"])
ALIASES = {"T3": ["T3", "T7"], "T4": ["T4", "T8"], "T5": ["T5", "P7"], "T6": ["T6", "P8"]}


def load_raw_safe(fpath):
    # Lê um .set do EEGLAB; se o .fdt pareado falhar, tenta uma cópia temporária
    fpath = Path(fpath)
    try:
        return mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=False)
    except Exception as exc:
        msg = str(exc).lower()
        fdt = fpath.with_suffix(".fdt")
        if ("fdt" in msg or "not found" in msg) and fdt.exists():
            from tempfile import TemporaryDirectory
            with TemporaryDirectory() as tmp:
                tmp_p = Path(tmp)
                (tmp_p / fpath.name).write_bytes(fpath.read_bytes())
                (tmp_p / fdt.name).write_bytes(fdt.read_bytes())
                return mne.io.read_raw_eeglab(str(tmp_p / fpath.name), preload=True, verbose=False)
        raise


def extract_spectral_features(data, sfreq, n_fft=None):
    data = np.asarray(data, dtype=float)
    n_fft = min(512, data.shape[-1]) if n_fft is None else min(int(n_fft), data.shape[-1])
    psds, freqs = psd_array_welch(data, sfreq=sfreq, fmin=FMIN, fmax=FMAX, n_fft=n_fft, verbose=False)
    psd_mean = np.mean(psds, axis=0)
    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    band_power = {}
    for band, (flo, fhi) in BANDS.items():
        idx = (freqs >= flo) & (freqs < fhi)
        band_power[band] = float(integrate(psd_mean[idx], freqs[idx])) if np.sum(idx) >= 2 else 0.0
    total = sum(band_power.values()) + 1e-12
    feats = {f"Rel_{b}_mean": band_power[b] / total for b in BANDS}
    feats["Razao_Theta_Alpha"] = feats["Rel_Theta_mean"] / (feats["Rel_Alpha_mean"] + 1e-12)
    feats["Razao_Theta_Beta"] = feats["Rel_Theta_mean"] / (feats["Rel_Beta_mean"] + 1e-12)
    psd_norm = psd_mean / (np.sum(psd_mean) + 1e-12)
    feats["Spectral_Entropy"] = float(-np.sum(psd_norm * np.log2(psd_norm + 1e-12)))
    return feats


def extract_connectivity_features(data, sfreq):
    data = np.asarray(data, dtype=float)
    n_ch, n_samp = data.shape
    out = {"PLV_Theta_mean": np.nan, "PLV_Alpha_mean": np.nan, "PLV_Beta_mean": np.nan, "Corr_Broadband_mean": np.nan}
    if n_ch < 2:
        return out
    iu = np.triu_indices(n_ch, k=1)
    out["Corr_Broadband_mean"] = float(np.nanmean(np.abs(np.corrcoef(data)[iu])))
    nyq, order = sfreq / 2.0, 4
    for band, (flo, fhi) in CONN_BANDS.items():
        fhi_safe = min(fhi, nyq * 0.99)
        if flo >= fhi_safe or n_samp < 3 * order * 6:
            continue
        sos = butter(order, [flo / nyq, fhi_safe / nyq], btype="band", output="sos")
        phase_exp = np.exp(1j * np.angle(hilbert(sosfiltfilt(sos, data, axis=1), axis=1)))
        plv = np.abs((phase_exp @ phase_exp.conj().T) / n_samp)
        out[f"PLV_{band}_mean"] = float(np.nanmean(plv[iu]))
    return out


def localizar_canais(ch_names, mapa):
    # mapa = {nome_no_arquivo: posição 10-20}; aceita T7/T8/P7/P8 no lugar de T3/T4/T5/T6
    low = {c.lower(): c for c in ch_names}
    pares = []
    for origem, alvo in mapa.items():
        cands = [origem] + (ALIASES.get(alvo, []) if origem == alvo else [])
        achado = next((low[c.lower()] for c in cands if c.lower() in low), None)
        if achado is None:
            raise KeyError(f"canal {origem} (posição {alvo}) não encontrado no arquivo")
        pares.append((achado, alvo))
    return pares


def features_19ch(fpath, mapa, sid, max_epocas=None, harmonizar_banda=False):
    try:
        raw = mne.io.read_raw_eeglab(str(fpath), preload=False, verbose=False)
        pares = localizar_canais(raw.ch_names, mapa)
        raw.pick([o for o, _ in pares]).load_data()
    except KeyError:
        raise
    except Exception:
        raw = load_raw_safe(fpath)
        pares = localizar_canais(raw.ch_names, mapa)
        raw.pick([o for o, _ in pares])
    renomear = {o: a for o, a in pares if o != a}
    if renomear:
        raw.rename_channels(renomear)
    raw.set_channel_types({c: "eeg" for c in raw.ch_names})
    raw.reorder_channels(CANAIS)
    raw.set_eeg_reference("average", projection=False)
    if harmonizar_banda:   # mesma banda passante do BrainLat distribuído (Butterworth 0,5–40 Hz, ordem 8, fase zero)
        hb = SPEC["preprocessamento"]["harmonizacao_banda_grego"]
        raw.filter(None, hb["h_freq"], method="iir", phase="zero", verbose=False,
                   iir_params=dict(order=hb["ordem"], ftype=hb["tipo"], output="sos"))
    raw.filter(FMIN, FMAX, fir_design="firwin", verbose=False)
    ep = mne.make_fixed_length_epochs(raw, duration=EPOCA_S, preload=True, verbose=False).get_data().astype(np.float64)
    n_total = len(ep)
    if max_epocas:
        ep = ep[:int(max_epocas)]
    if len(ep) == 0:
        raise RuntimeError("nenhuma época de 4 s")
    sf = float(raw.info["sfreq"])
    rms = np.sqrt(np.mean(ep ** 2))
    ep = ep / (rms if np.isfinite(rms) and rms > 1e-12 else 1.0)
    n_fft = int(round(sf / RES_HZ))
    linhas, ruins = [], 0
    for i, e in enumerate(ep):
        f = extract_spectral_features(e, sf, n_fft=n_fft)
        f.update(extract_connectivity_features(e, sf))
        if not all(np.isfinite(f[k]) for k in FEATURES):
            ruins += 1
            continue
        f.update({"subject_id": sid, "epoch_id": i, "sfreq": sf})
        linhas.append(f)
    n_bound = int(sum(str(d).strip().lower() == "boundary" for d in raw.annotations.description))
    info = {"subject_id": sid, "epocas_total": n_total, "epocas_usadas": len(linhas), "epocas_invalidas": ruins,
            "sfreq": sf, "duracao_min": float(raw.times[-1]) / 60, "marcas_boundary": n_bound}
    return linhas, info


print("Funções de extração definidas.")

### 3.2 Os 19 canais do BrainLat mais próximos das posições 10-20

> **O que faz:** pega as posições dos 128 eletrodos de um arquivo do BrainLat (e, se os nomes forem os da BioSemi, também o
> modelo `biosemi128` do MNE), centra as montagens por ajuste de esfera, projeta na esfera unitária e atribui a cada posição
> 10-20 o eletrodo de menor ângulo, sem repetir eletrodos. Fica com a fonte de posições que der o menor ângulo máximo.
>
> **Onde entra:** Methods (uma frase) e material suplementar (tabela de correspondência).

In [ ]:
# 3.2 — Correspondência BioSemi-128 → 10-20 por coordenadas
def nome_na_montagem(c, pos):
    # aceita T7/T8/P7/P8 quando a montagem não tem T3/T4/T5/T6
    return next((x for x in [c] + ALIASES.get(c, []) if x in pos), None)


def montagem_padrao_1020():
    # 'standard_1020' virou 'colin27_1020' nas versões novas do MNE; tenta as duas
    for nome in ("colin27_1020", "standard_1020", "colin27_1005", "standard_1005"):
        try:
            pos = mne.channels.make_standard_montage(nome).get_positions()["ch_pos"]
        except Exception:
            continue
        if all(nome_na_montagem(c, pos) for c in CANAIS):
            return pos, nome
    raise RuntimeError("Nenhuma montagem padrão do MNE tem as 19 posições 10-20.")


def unitarias(pos_dict):
    nomes = [k for k, v in pos_dict.items() if v is not None and np.all(np.isfinite(v))]
    P = np.array([pos_dict[k] for k in nomes], float)
    A = np.column_stack([2 * P, np.ones(len(P))])          # |p - c|² = r²  →  2p·c + k = |p|²
    sol = np.linalg.lstsq(A, (P ** 2).sum(axis=1), rcond=None)[0]
    U = P - sol[:3]
    return nomes, U / np.linalg.norm(U, axis=1, keepdims=True)


def mapear(pos_origem, U_st, n_st):
    n_o, U_o = unitarias(pos_origem)
    U_alvo = np.array([U_st[n_st.index(ALVO_1020[c])] for c in CANAIS])
    ang = np.degrees(np.arccos(np.clip(U_alvo @ U_o.T, -1, 1)))      # 19 × n
    r_idx, c_idx = linear_sum_assignment(ang)
    return pd.DataFrame({"canal_1020": [CANAIS[i] for i in r_idx], "canal_brainlat": [n_o[j] for j in c_idx],
                         "angulo_graus": ang[r_idx, c_idx]})


raw0, arq0 = None, None
for f in subj_bl["arquivo"]:
    try:
        raw0 = mne.io.read_raw_eeglab(str(f), preload=False, verbose=False)
        arq0 = f
        break
    except Exception:
        continue
if raw0 is None:
    raise RuntimeError("Nenhum .set do BrainLat pôde ser lido.")
POS_1020, nome_mt = montagem_padrao_1020()
n_st, U_st = unitarias(POS_1020)
ALVO_1020 = {c: nome_na_montagem(c, POS_1020) for c in CANAIS}
candidatos = {}
mont0 = raw0.get_montage()
pos_arq = {k: v for k, v in (mont0.get_positions()["ch_pos"].items() if mont0 is not None else [])
           if k in raw0.ch_names and v is not None and np.all(np.isfinite(v)) and np.linalg.norm(v) > 0}
if len(pos_arq) >= 100:
    candidatos["posições do próprio arquivo"] = mapear(pos_arq, U_st, n_st)
tmpl = mne.channels.make_standard_montage("biosemi128").get_positions()["ch_pos"]
pos_tmpl = {k: tmpl[k] for k in raw0.ch_names if k in tmpl}
if len(pos_tmpl) >= 100:
    candidatos["modelo biosemi128 do MNE (por nome de canal)"] = mapear(pos_tmpl, U_st, n_st)
if not candidatos:
    raise RuntimeError("Não foi possível obter as posições dos eletrodos do BrainLat (nem do arquivo, nem por nome).")
for k, m in candidatos.items():
    print(f"{k}: ângulo médio {m.angulo_graus.mean():.1f}°, máximo {m.angulo_graus.max():.1f}°")
FONTE_POS = min(candidatos, key=lambda k: candidatos[k].angulo_graus.max())
MAPA_BL = candidatos[FONTE_POS].sort_values("canal_1020", key=lambda s: s.map(CANAIS.index)).reset_index(drop=True)
display(MAPA_BL.round(1))
print(f"Usado: {FONTE_POS} | referência 10-20: {nome_mt} | arquivo: {Path(arq0).name}")
if MAPA_BL.angulo_graus.max() > 20:
    alerta("Algum canal 10-20 ficou a mais de 20° do eletrodo escolhido: confira a orientação das posições.")
MAPA_BL.to_csv(OUT / "tabelas" / "S1_mapa_canais_brainlat_1020.csv", index=False)
registrar("Métodos", "Correspondência BioSemi → 10-20",
          f"{FONTE_POS}; ângulo médio {MAPA_BL.angulo_graus.mean():.1f}°, máximo {MAPA_BL.angulo_graus.max():.1f}°",
          "Methods; Suppl. Table S1")

### 3.3 Extração nas duas coortes

> **O que faz:** extrai as 11 features por época nas duas coortes, sem usar rótulos. Grava os CSVs em
> `outputs_journal/features/` e os reaproveita se os parâmetros forem os mesmos. Quem não tiver nenhuma época válida
> (arquivo ilegível, por exemplo) sai das análises, e isso fica registrado.

In [ ]:
# 3.3 — Extração harmonizada (19 canais, 11 features), sem rótulos
def extrair_conjunto(itens, nome, params, harmonizar_banda=False):
    f_csv = OUT / "features" / f"{nome}.csv"
    f_info = OUT / "features" / f"{nome}_por_sujeito.csv"
    f_fal = OUT / "features" / f"{nome}_falhas.csv"
    f_par = OUT / "features" / f"{nome}.json"
    def ler():
        # sempre a partir do CSV, para a 1ª execução e as seguintes usarem exatamente os mesmos números
        fal = pd.read_csv(f_fal, dtype={"subject_id": str}) if f_fal.exists() and f_fal.stat().st_size > 25 \
            else pd.DataFrame(columns=["subject_id", "arquivo", "erro"])
        return (pd.read_csv(f_csv, dtype={"subject_id": str}, float_precision="round_trip"),
                pd.read_csv(f_info, dtype={"subject_id": str}, float_precision="round_trip"), fal)

    if USAR_CACHE and f_csv.exists() and f_info.exists() and f_par.exists() \
            and json.loads(f_par.read_text(encoding="utf-8")) == params:
        print(f"[cache] {nome}")
        return ler()
    linhas, infos, falhas, t0 = [], [], [], time.time()
    for k, (sid, fpath, mapa) in enumerate(itens):
        try:
            l, info = features_19ch(fpath, mapa, sid, RT["max_epocas"], harmonizar_banda)
            linhas += l
            infos.append(info)
        except Exception as exc:
            falhas.append({"subject_id": sid, "arquivo": str(fpath), "erro": str(exc)[:150]})
        if (k + 1) % 10 == 0 or k + 1 == len(itens):
            dt = time.time() - t0
            print(f"  {nome}: {k + 1}/{len(itens)} arquivos | {dt:.0f} s | faltam ~{dt / (k + 1) * (len(itens) - k - 1):.0f} s")
    if not linhas:
        raise RuntimeError(f"{nome}: nenhum arquivo gerou features. Primeiros erros: {falhas[:3]}")
    pd.DataFrame(linhas).to_csv(f_csv, index=False)
    pd.DataFrame(infos).to_csv(f_info, index=False)
    pd.DataFrame(falhas, columns=["subject_id", "arquivo", "erro"]).to_csv(f_fal, index=False)
    f_par.write_text(json.dumps(params, ensure_ascii=False), encoding="utf-8")
    return ler()


mapa_bl = dict(zip(MAPA_BL["canal_brainlat"], MAPA_BL["canal_1020"]))
mapa_gr = {c: c for c in CANAIS}
base_params = {"canais": CANAIS, "prep": SPEC["preprocessamento"], "features": FEATURES,
               "max_epocas": RT["max_epocas"], "versao": 3}
itens_bl = [(r.subject_id, r.arquivo, mapa_bl) for r in subj_bl.itertuples()]
itens_gr = [(r.subject_id, r.arquivo, mapa_gr) for r in subj_gr.itertuples() if isinstance(r.arquivo, str)]
feats_bl, info_bl, fal_bl = extrair_conjunto(
    itens_bl, f"features_19ch_brainlat_{TAG}",
    {**base_params, "mapa": mapa_bl, "arquivos": [[s, Path(f).name] for s, f, _ in itens_bl]})
feats_gr, info_gr, fal_gr = extrair_conjunto(
    itens_gr, f"features_19ch_grego_{TAG}",
    {**base_params, "mapa": mapa_gr, "arquivos": [[s, Path(f).name] for s, f, _ in itens_gr],
     "derivatives": SPEC["coortes"]["grego"]["usar_derivatives"], "harmonizacao_banda": True},
    harmonizar_banda=True)

for nome_c, fal in (("BrainLat", fal_bl), ("ds004504", fal_gr)):
    if len(fal):
        print(f"\n{nome_c}: {len(fal)} arquivo(s) sem features:")
        display(fal)

# Rótulos entram só depois da extração
df_bl = feats_bl.merge(subj_bl[["subject_id", "y"]], on="subject_id", how="inner").reset_index(drop=True)
df_gr = feats_gr.merge(subj_gr[["subject_id", "y"]], on="subject_id", how="inner").reset_index(drop=True)
n_antes = {"BrainLat": len(subj_bl), "Grego": len(subj_gr)}
subj_bl = subj_bl[subj_bl.subject_id.isin(df_bl.subject_id)].merge(info_bl, on="subject_id", how="left").reset_index(drop=True)
subj_gr = subj_gr[subj_gr.subject_id.isin(df_gr.subject_id)].merge(info_gr, on="subject_id", how="left").reset_index(drop=True)
for nome_c, s, d in (("BrainLat", subj_bl, df_bl), ("Grego", subj_gr, df_gr)):
    print(f"{nome_c}: {len(s)} de {n_antes[nome_c]} com EEG ({int(s.y.sum())} AD, {int((1 - s.y).sum())} HC) | "
          f"{len(d)} épocas | taxa(s) de amostragem {sorted(set(d.sfreq))} Hz")
    nb_ = s["marcas_boundary"].fillna(0) if "marcas_boundary" in s else pd.Series(0, index=s.index)
    if (nb_ > 0).any():
        print(f"  {int((nb_ > 0).sum())} participante(s) com marcas 'boundary' (trechos cortados no pré-processamento "
              f"original; mediana {nb_[nb_ > 0].median():.0f} por arquivo). As épocas não foram descartadas, como no artigo.")
        registrar("Métodos", f"{nome_c}: arquivos com marcas 'boundary'",
                  f"{int((nb_ > 0).sum())} de {len(s)}; mediana {nb_[nb_ > 0].median():.0f} por arquivo", "Methods (se relevante)")
    registrar("Métodos", f"{nome_c}: participantes com EEG analisável",
              f"{len(s)} de {n_antes[nome_c]} ({int(s.y.sum())} AD, {int((1 - s.y).sum())} HC); {len(d)} épocas",
              "Methods; Table 1")
excl_bl = sorted(set(arq_bl.subject_id) - set(subj_bl.subject_id))
if excl_bl:
    registrar("Métodos", "BrainLat: sem EEG analisável", f"{len(excl_bl)}: {', '.join(excl_bl[:20])}", "Methods")

### 3.4 Tabela 1 e mudança de domínio entre as coortes

> **O que faz:** monta a tabela de participantes das duas coortes (testes: Mann–Whitney e Fisher) e compara as features
> entre coortes. A segunda tabela não é resultado: ela mostra quanto aparelho, montagem e população mudam as features,
> o que ajuda a ler a RQ3.
>
> **Onde entra:** Table 1; material suplementar.

In [ ]:
# 3.4 — Tabela 1 (participantes) e comparação das features entre coortes
def p_fisher(tab):
    tab = np.asarray(tab)
    if tab.shape != (2, 2):
        return np.nan
    r = stats.fisher_exact(tab)
    return float(getattr(r, "pvalue", r[1]))


def p_mwu(a, b):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if len(a) < 2 or len(b) < 2 or (a.nunique() == 1 and b.nunique() == 1 and a.iloc[0] == b.iloc[0]):
        return np.nan
    return float(stats.mannwhitneyu(a, b).pvalue)


def media_dp(s, com_n=False, nd=1):
    s = pd.Series(s).dropna()
    if len(s) == 0:
        return "—"
    txt = f"{s.mean():.{nd}f} ± {s.std():.{nd}f}"
    return f"{txt} ({len(s)})" if com_n else txt


def tabela_participantes(s, coorte):
    AD, HC = s[s.y == 1], s[s.y == 0]
    L = [{"coorte": coorte, "variável": "n", "AD": len(AD), "HC": len(HC), "p": "", "teste": ""},
         {"coorte": coorte, "variável": "Age (years)", "AD": media_dp(AD.Age), "HC": media_dp(HC.Age),
          "p": fmt_p(p_mwu(AD.Age, HC.Age)), "teste": "Mann–Whitney"}]
    fa, fh = AD.sex_fem.dropna(), HC.sex_fem.dropna()
    tab = [[int(fa.sum()), int((1 - fa).sum())], [int(fh.sum()), int((1 - fh).sum())]]
    L.append({"coorte": coorte, "variável": "Female, n (%)", "AD": f"{int(fa.sum())} ({100 * fa.mean():.1f})",
              "HC": f"{int(fh.sum())} ({100 * fh.mean():.1f})", "p": fmt_p(p_fisher(tab)), "teste": "Fisher"})
    if "years_education" in s:
        L.append({"coorte": coorte, "variável": "Education (years)", "AD": media_dp(AD.years_education),
                  "HC": media_dp(HC.years_education), "p": fmt_p(p_mwu(AD.years_education, HC.years_education)),
                  "teste": "Mann–Whitney"})
    if "site" in s:
        tab = [[int((AD.site == "Argentina").sum()), int((AD.site == "Chile").sum())],
               [int((HC.site == "Argentina").sum()), int((HC.site == "Chile").sum())]]
        L.append({"coorte": coorte, "variável": "Site (Argentina / Chile)", "AD": f"{tab[0][0]} / {tab[0][1]}",
                  "HC": f"{tab[1][0]} / {tab[1][1]}", "p": fmt_p(p_fisher(tab)), "teste": "Fisher"})
    for col, rot in (("moca_total", "MoCA"), ("ifs_total_score", "IFS"), ("mmse", "MMSE")):
        if col in s and s[col].notna().any():
            L.append({"coorte": coorte, "variável": rot, "AD": media_dp(AD[col], True), "HC": media_dp(HC[col], True),
                      "p": fmt_p(p_mwu(AD[col], HC[col])), "teste": "Mann–Whitney"})
    L.append({"coorte": coorte, "variável": "Recording analysed (min)", "AD": media_dp(AD.duracao_min),
              "HC": media_dp(HC.duracao_min), "p": fmt_p(p_mwu(AD.duracao_min, HC.duracao_min)), "teste": "Mann–Whitney"})
    L.append({"coorte": coorte, "variável": "Epochs per participant", "AD": media_dp(AD.epocas_usadas, nd=0),
              "HC": media_dp(HC.epocas_usadas, nd=0), "p": fmt_p(p_mwu(AD.epocas_usadas, HC.epocas_usadas)),
              "teste": "Mann–Whitney"})
    return pd.DataFrame(L)


TAB1 = pd.concat([tabela_participantes(subj_bl, "BrainLat"), tabela_participantes(subj_gr, "Grego (ds004504)")],
                 ignore_index=True)
display(TAB1)
TAB1.to_csv(OUT / "tabelas" / "T1_participantes.csv", index=False, encoding="utf-8-sig")

med = (pd.concat([df_bl.assign(coorte="BrainLat"), df_gr.assign(coorte="Grego")])
       .assign(grupo=lambda d: np.where(d.y == 1, "AD", "HC"))
       .groupby(["coorte", "grupo"])[FEATURES].median().T)
med.columns = [f"{c} {g}" for c, g in med.columns]
print("Medianas por época (coorte × grupo):")
display(med.round(3))
smd = {}
for f in FEATURES:
    a = df_bl[df_bl.y == 0].groupby("subject_id")[f].mean()
    b = df_gr[df_gr.y == 0].groupby("subject_id")[f].mean()
    smd[f] = (b.mean() - a.mean()) / np.sqrt((a.var() + b.var()) / 2)
DOMINIO = med.assign(**{"SMD entre coortes (HC, grego − BrainLat)": pd.Series(smd)})
DOMINIO.to_csv(OUT / "tabelas" / "S2_features_por_coorte.csv", encoding="utf-8-sig")
grandes = [f for f, v in smd.items() if abs(v) > 0.8]
print(f"Features com |SMD| > 0,8 entre os controles das duas coortes: {grandes or 'nenhuma'}")
registrar("Métodos", "Mudança de domínio (controles, |SMD| > 0,8 entre coortes)", ", ".join(grandes) or "nenhuma",
          "Results (RQ3); Suppl. Table S2")

### 3.5 Desenho das análises de confusão (antes de qualquer predição)

> **O que faz:** define, só com demografia, os pares 1:1 por escolaridade do BrainLat, os pares comparáveis das duas coortes
> e os overlap weights. Grava tudo com hash **antes** da seção 4, que é a primeira a gerar predições.
>
> **Onde entra:** Methods; tabelas de balanço no material suplementar.

In [ ]:
# 3.5 — Pareamento, pares comparáveis e overlap weights (desenho)
R2 = SPEC["rq2"]
CAL = R2["pareamento_brainlat"]["calibre_anos"]
COV_BL = list(SPEC["coortes"]["brainlat"]["covariaveis"])
COV_GR = list(SPEC["coortes"]["grego"]["covariaveis"])
VAR_BAL_BL = {"years_education": "Education (years)", "Age": "Age (years)", "sex_fem": "Female (proportion)",
              "site_cl": "Chile (proportion)"}
VAR_BAL_GR = {"Age": "Age (years)", "sex_fem": "Female (proportion)"}

pares_bl = parear_1a1(subj_bl, CAL)
N_PARES = len(pares_bl)
pareados = subj_bl[subj_bl.subject_id.isin(set(pares_bl.ad_id) | set(pares_bl.hc_id))]
print(f"BrainLat — pares 1:1 com |Δescolaridade| ≤ {CAL}: {N_PARES} ({2 * N_PARES} de {len(subj_bl)} sujeitos)")
BAL = {"BrainLat pareamento": balanco(subj_bl, VAR_BAL_BL).merge(balanco(pareados, VAR_BAL_BL), on="variável",
                                                                    suffixes=("_todos", "_pareados"))}
f_pares = OUT / "tabelas" / "desenho_pares_1a1_brainlat.csv"
pares_bl.to_csv(f_pares, index=False, encoding="utf-8-sig")

ELEG_BL = lambda a, h: np.abs(a["years_education"].to_numpy(float)[:, None]
                              - h["years_education"].to_numpy(float)[None, :]) <= R2["pares_comparaveis_brainlat"]["calibre_escolaridade_anos"]
CAL_IDADE_GR = R2["pares_comparaveis_grego"]["calibre_idade_anos"]
ELEG_GR = lambda a, h: ((a["sex_fem"].to_numpy(float)[:, None] == h["sex_fem"].to_numpy(float)[None, :])
                        & (np.abs(a["Age"].to_numpy(float)[:, None] - h["Age"].to_numpy(float)[None, :]) <= CAL_IDADE_GR))
tbl_c = subj_bl.dropna(subset=["years_education"])
n_comp_bl = int(np.asarray(ELEG_BL(tbl_c[tbl_c.y == 1], tbl_c[tbl_c.y == 0])).sum())
tgr_c = subj_gr.dropna(subset=["Age", "sex_fem"])
n_comp_gr = int(np.asarray(ELEG_GR(tgr_c[tgr_c.y == 1], tgr_c[tgr_c.y == 0])).sum())
print(f"Pares comparáveis — BrainLat (escolaridade ±2 anos): {n_comp_bl} | grego (mesmo sexo, idade ±{CAL_IDADE_GR}): {n_comp_gr}")

DESENHO = {"gerado_em": datetime.datetime.now().isoformat(timespec="seconds"),
           "observacao": "gerado na seção 3.5, antes de qualquer predição (seção 4)",
           "n_pares_1a1": N_PARES, "sha1_pares": hashlib.sha1(f_pares.read_bytes()).hexdigest()[:12],
           "n_pares_comparaveis_brainlat": n_comp_bl, "n_pares_comparaveis_grego": n_comp_gr}
for nome_c, s, covs, vb in (("BrainLat", subj_bl, R2["overlap_brainlat"], VAR_BAL_BL),
                            ("Grego", subj_gr, R2["overlap_grego"], VAR_BAL_GR)):
    t = s.dropna(subset=covs).reset_index(drop=True)
    ps, metodo = escore_propensao(t, covs)
    w = np.where(t["y"] == 1, 1 - ps, ps)
    yy = t["y"].to_numpy(int)
    BAL[f"{nome_c} overlap"] = balanco(t, vb).merge(balanco(t, vb, w), on="variável", suffixes=("_bruto", "_ponderado"))
    DESENHO[f"overlap_{nome_c}"] = {"covariaveis": covs, "metodo_ps": metodo,
                                    "ess_AD": ess(w[yy == 1]), "ess_HC": ess(w[yy == 0]), "n": len(t)}
    print(f"{nome_c} — overlap weights ({' + '.join(covs)}; {metodo}): tamanho efetivo AD {ess(w[yy == 1]):.1f} de "
          f"{int(yy.sum())}, HC {ess(w[yy == 0]):.1f} de {int((1 - yy).sum())}")
for k, b in BAL.items():
    print(f"\nBalanço — {k}:")
    display(b.round(3))
    b.to_csv(OUT / "tabelas" / f"S3_balanco_{re.sub(r'[^A-Za-z]+', '_', k)}.csv", index=False, encoding="utf-8-sig")
(OUT / "tabelas" / "desenho_confundimento.json").write_text(json.dumps(DESENHO, indent=2, ensure_ascii=False,
                                                                       default=str), encoding="utf-8")
if N_PARES < R2["pareamento_brainlat"]["min_pares"]:
    alerta(f"Só {N_PARES} pares 1:1 (< {R2['pareamento_brainlat']['min_pares']}): a AUC pareada será só descrita.")

---
## 4. Escore de EEG em cada coorte (LOSO)

> **O que faz:** roda o Random Forest da SPEC em LOSO dentro de cada coorte (19 canais, 11 features). O P(AD) out-of-fold
> de cada pessoa é o escore de EEG usado nas RQ1 e RQ2. Além da AUC, mostra sensibilidade, especificidade, acurácia,
> F1 e MCC com limiar 0,5, para comparar com os estudos que usaram o ds004504.
>
> **Base:** validação com sujeitos independentes, todo o pré-processamento dependente de dados dentro do treino de cada
> fold, épocas como unidade de análise e LOSO cumprem os critérios C1–C7 do benchmark do ds004504 (Miltiadous et al., 2026).
> Nesse benchmark, os estudos que vazavam épocas entre treino e teste relatavam acurácias quase perfeitas.
>
> **Onde entra:** Results (primeiro parágrafo) e Table 2.

In [ ]:
# 4 — LOSO do RF primário nas duas coortes
PRED = {}
PRED["BrainLat"] = run_loso(df_bl, FEATURES, "RandomForest", "BL_RF_11")
PRED["Grego"] = run_loso(df_gr, FEATURES, "RandomForest", "GR_RF_11")
AUC_EEG, MET_EEG = {}, {}
LIMIAR = SPEC["rq3"]["limiar_metricas"]
for nome_c, pr in PRED.items():
    AUC_EEG[nome_c] = auc_ci(pr["y"], pr["p"])
    MET_EEG[nome_c] = metricas_limiar(pr["y"], pr["p"], LIMIAR)
    pr.to_csv(OUT / "tabelas" / f"predicoes_oof_{nome_c}_RF_11.csv", index=False)
    print(f"{nome_c}: AUC do EEG (RF, 11 features, 19 canais, LOSO) = {fmt_ci(*AUC_EEG[nome_c])} (n = {len(pr)})")
    print(f"   limiar {LIMIAR}: {fmt_metricas(MET_EEG[nome_c])}")
    registrar("Base", f"{nome_c}: AUC do EEG (RF, LOSO)", fmt_ci(*AUC_EEG[nome_c]), "Results; Table 2")
    registrar("Base", f"{nome_c}: métricas com limiar {LIMIAR} (RF, LOSO)", fmt_metricas(MET_EEG[nome_c]),
              "Table 2", "descritivo; comparável aos estudos do ds004504")
print("\nReferência no ds004504 (AD vs controles): estudos com validação por sujeito (Validity 1 do benchmark de "
      "Miltiadous et al., 2026) têm acurácia média de 82,1% (DP 6,9); com vazamento entre épocas, 97,1%.")
subj_bl = subj_bl.drop(columns=["p_eeg"], errors="ignore").merge(
    PRED["BrainLat"][["subject_id", "p"]].rename(columns={"p": "p_eeg"}), on="subject_id", how="left")
subj_gr = subj_gr.drop(columns=["p_eeg"], errors="ignore").merge(
    PRED["Grego"][["subject_id", "p"]].rename(columns={"p": "p_eeg"}), on="subject_id", how="left")
print("\nReferência: no artigo da ACDSA (128 canais), o RF deu AUC 0.726 [0.592, 0.839] no BrainLat.")

---
## 5. RQ1 — Valor incremental

### 5.1 RQ1a — BrainLat: além de idade, sexo e escolaridade

*RQ1a. In BrainLat, does the EEG score improve the identification of AD beyond age, sex, and education?*

> **O que faz:** compara a logística com idade, sexo e escolaridade com a mesma logística + escore de EEG (estatística de
> razão de verossimilhança, 1 gl; OR por 1 DP com IC perfilado). O p-valor vem de um **bootstrap paramétrico sob H0**:
> sorteia diagnósticos que dependem só da demografia, refaz o LOSO inteiro com eles e recalcula a estatística. Também
> calcula as linhas de base (logísticas em leave-one-out: idade; idade + sexo; idade + sexo + escolaridade; demografia +
> escore de EEG) e a diferença de AUC, que é só descritiva. Como verificação fora da regra de leitura, repete o teste
> incluindo o sítio (Argentina/Chile).
>
> **Por que o bootstrap e não o qui-quadrado direto:** o escore é de validação cruzada (cada pessoa recebe a predição de um
> modelo treinado nos outros). Usar esse escore como covariável é o que a estatística chama de *pre-validation*, e o teste
> analítico de 1 gl pode sair enviesado, sobretudo quando o EEG se correlaciona com as covariáveis, como aqui com idade e
> escolaridade (Höfling & Tibshirani, 2008; Shang et al., 2025). Dinga et al. (2020) também recomendam reamostragem quando
> as predições vêm de validação cruzada. O bootstrap sob H0 mantém a correlação entre EEG e demografia, reproduz o pipeline
> inteiro e para cedo quando 10 réplicas passam do observado, porque aí o p já está claramente acima de 0,05 (Besag &
> Clifford, 1991). Cada réplica fica em cache: se o notebook parar, continua de onde estava.
>
> **Por que a ΔAUC é só descritiva:** testar a melhora da AUC equivale a testar o coeficiente do escore e tem muito menos
> poder (Vickers et al., 2011; Pepe et al., 2013). Além disso, a logística combinada usa escores que não são aninhados (o RF
> que gerou o escore dos outros viu a pessoa deixada de fora), o que favorece um pouco o modelo combinado.
>
> **Leitura (SPEC):** acrescenta se o p do bootstrap nulo for menor que 0,05.
>
> **Onde entra:** Results (RQ1), Table 2, Fig. 1 e Fig. 2.

In [ ]:
# 5.1 — RQ1a: BrainLat
INC, BASE, DELTA = {}, {}, {}
EEG_ROT, COMB_ROT = "EEG score (RF, LOSO)", "Demographics + EEG score"


def linhas_de_base(s, covs, nomes):
    # logísticas em LOO no nível do sujeito; o escore de EEG entra pelo logit do P(AD) out-of-fold
    t = s.dropna(subset=covs + ["p_eeg"]).reset_index(drop=True).copy()
    t["eeg_logit"] = logit_seguro(t["p_eeg"])
    out = {rot: loo_lr_sujeito(t, cols) for rot, cols in nomes.items()}
    out[EEG_ROT] = t[["subject_id", "y", "p_eeg"]].rename(columns={"p_eeg": "p"})
    out[COMB_ROT] = loo_lr_sujeito(t, covs + ["eeg_logit"])
    return out


def tabela_aucs(bases, coorte):
    linhas = []
    for nome, d in bases.items():
        a, lo, hi = auc_ci(d["y"], d["p"])
        linhas.append({"coorte": coorte, "modelo": nome, "AUC": a, "IC_inf": lo, "IC_sup": hi, "n": len(d)})
    return pd.DataFrame(linhas)


def mostrar_incremental(r):
    p_txt = f"p = {fmt_p(r['p'])}" + (f" (unilateral {fmt_p(r['p_unilateral'])})" if r["coorte"].startswith("Grego") else "")
    print(f"{r['coorte']} (n = {r['n']}: {r['n_AD']} AD, {r['n_HC']} HC) — escore EEG dado {r['covariaveis']}: "
          f"LR χ²(1) = {r['LR']:.2f}, {p_txt}; OR por 1 DP = {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} ({r['metodo']})")


BOOT = {}


def rodar_bootstrap(coorte, df_ep, subj, covs, nome, unilateral):
    print(f"Bootstrap nulo ({coorte}): refaz o LOSO até {RT['boot_nulo_B']} vezes sob H0; cada réplica fica em cache ...")
    bn = bootstrap_nulo(df_ep, subj, covs, nome, unilateral=unilateral)
    (OUT / "tabelas" / f"bootstrap_nulo_{coorte}.json").write_text(json.dumps(
        {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in bn.items() if k != "naive"},
        indent=1, default=str), encoding="utf-8")
    lado = "unilateral" if unilateral else "bilateral"
    print(f"p do bootstrap nulo ({lado}) = {fmt_p(bn['p'])} | {bn['replicas']} réplicas, {bn['excedencias']} ≥ observado"
          f"{' (parada sequencial: p claramente acima de 0,05)' if bn['parou_cedo'] else ''} | p ingênuo "
          f"{fmt_p(bn['naive']['p_unilateral'] if unilateral else bn['naive']['p'])}")
    return bn


INC["BrainLat"] = teste_incremental(subj_bl, COV_BL, "p_eeg", "BrainLat")
mostrar_incremental(INC["BrainLat"])
r = INC["BrainLat"]
BOOT["BrainLat"] = rodar_bootstrap("BrainLat", df_bl, subj_bl, COV_BL, "BL_RF_11", unilateral=False)
VEREDITO["RQ1a"] = "acrescenta" if BOOT["BrainLat"]["p"] < 0.05 else "não demonstrado"
print(f"Leitura (SPEC): {VEREDITO['RQ1a']}")
registrar("RQ1a", "BrainLat: escore EEG além de idade, sexo e escolaridade",
          f"χ²(1) = {r['LR']:.2f}; p (bootstrap nulo) = {fmt_p(BOOT['BrainLat']['p'])} "
          f"[{BOOT['BrainLat']['replicas']} réplicas]; p ingênuo = {fmt_p(r['p'])}; OR/DP = "
          f"{fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} (IC perfilado, descritivo); {r['metodo']}; n = {r['n']}",
          "Results; Table 2", f"leitura: {VEREDITO['RQ1a']}")

INC["BrainLat + sítio"] = rs = teste_incremental(subj_bl, COV_BL + ["site_cl"], "p_eeg", "BrainLat")
print(f"Verificação com sítio (fora da regra de leitura): OR/DP = {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}, "
      f"p = {fmt_p(rs['p'])} ({rs['metodo']}; n = {rs['n']})")
registrar("RQ1a", "BrainLat: verificação com sítio (idade, sexo, escolaridade, sítio)",
          f"OR/DP = {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}; p = {fmt_p(rs['p'])}; {rs['metodo']}",
          "Results (uma frase)", "verificação; fora da regra de leitura")

BASE["BrainLat"] = linhas_de_base(subj_bl, COV_BL, {"Age": ["Age"], "Age + sex": ["Age", "sex_fem"],
                                                     "Age + sex + education": COV_BL})
TAB_AUC = [tabela_aucs(BASE["BrainLat"], "BrainLat")]
display(TAB_AUC[-1].round(3))
demo = BASE["BrainLat"]["Age + sex + education"]
for rot, outra in ((f"{COMB_ROT} − demographics", BASE["BrainLat"][COMB_ROT]),
                   (f"{EEG_ROT} − demographics", BASE["BrainLat"][EEG_ROT])):
    mm = outra.merge(demo[["subject_id", "p"]], on="subject_id", suffixes=("", "_d"))
    DELTA[("BrainLat", rot)] = delta_boot(mm["y"], mm["p"], mm["p_d"])
    print(f"ΔAUC {rot}: {fmt_ci(*DELTA[('BrainLat', rot)])} (descritivo)")
for _, l in TAB_AUC[-1].iterrows():
    registrar("RQ1a", f"BrainLat: AUC {l['modelo']}", fmt_ci(l["AUC"], l["IC_inf"], l["IC_sup"]), "Table 2",
              "LOSO" if l["modelo"] == EEG_ROT else "logística, leave-one-out")
for (c, rot), v in DELTA.items():
    if c == "BrainLat":
        registrar("RQ1a", f"BrainLat: ΔAUC {rot}", fmt_ci(*v), "Results", "descritivo, sem teste")

### 5.2 RQ1b — Grego: além de idade e sexo (E-value para a escolaridade)

*RQ1b. In the Greek cohort, does the EEG score improve the identification of AD beyond age and sex?*

> **O que faz:** o mesmo teste no ds004504, com idade e sexo, e o mesmo bootstrap nulo. Como é a coorte de réplica, o teste
> é unilateral, na direção fixada na SPEC (OR > 1: escore maior, mais chance de AD). A escolaridade não foi registrada no
> ds004504 (as colunas são participant_id, Gender, Age, Group e MMSE), então o E-value diz quão forte um confundidor não
> medido teria de ser, na associação com o diagnóstico e com o escore, para explicar o efeito (VanderWeele & Ding, 2017).
> Para ler o E-value, o notebook mostra quanto a escolaridade mudou o OR do EEG no BrainLat, onde ela foi medida: comparar
> com a força de um confundidor medido é a leitura recomendada pelos próprios autores do E-value.
>
> **Leitura (SPEC):** acrescenta se o p unilateral do bootstrap nulo for menor que 0,05.
>
> **Onde entra:** Results (RQ1), Table 2, Discussion (limitações).

In [ ]:
# 5.2 — RQ1b: grego
INC["Grego"] = teste_incremental(subj_gr, COV_GR, "p_eeg", "Grego")
mostrar_incremental(INC["Grego"])
r = INC["Grego"]
BOOT["Grego"] = rodar_bootstrap("Grego", df_gr, subj_gr, COV_GR, "GR_RF_11", unilateral=True)
VEREDITO["RQ1b"] = "acrescenta" if BOOT["Grego"]["p"] < 0.05 else "não demonstrado"
E_PT, E_CI = e_value(r["OR"], r["OR_lo"], r["OR_hi"])
print(f"Leitura (SPEC): {VEREDITO['RQ1b']}")
print(f"E-value: {fmt(E_PT, 2)} para o OR e {fmt(E_CI, 2)} para o limite do IC "
      "(RR mínimo de um confundidor não medido, com o diagnóstico e com o escore, para anular o efeito)")

# Referência para ler o E-value: quanto a escolaridade pesou no BrainLat, onde foi medida (mesmos sujeitos)
bl_cc = subj_bl.dropna(subset=COV_BL + ["p_eeg"]).reset_index(drop=True)
r_sem = teste_incremental(bl_cc, ["Age", "sex_fem"], "p_eeg", "BrainLat")
Xe = np.column_stack([np.ones(len(bl_cc)), padronizar(bl_cc[COV_BL].to_numpy(float))])
fe = logit_fit(Xe, bl_cc["y"].to_numpy(int))
OR_EDU = float(np.exp(fe["beta"][1 + COV_BL.index("years_education")]))
print(f"Referência (BrainLat, n = {len(bl_cc)}): OR/DP do EEG = {r_sem['OR']:.2f} com idade + sexo e "
      f"{INC['BrainLat']['OR']:.2f} com + escolaridade (razão {INC['BrainLat']['OR'] / r_sem['OR']:.2f}); "
      f"escolaridade: OR por DP de {OR_EDU:.2f} para AD (RR ≈ {np.sqrt(OR_EDU if OR_EDU >= 1 else 1 / OR_EDU):.2f}), "
      "ajustado por idade e sexo")
registrar("RQ1b", "Referência para o E-value (BrainLat)",
          f"OR/DP do EEG {r_sem['OR']:.2f} (idade + sexo) → {INC['BrainLat']['OR']:.2f} (+ escolaridade); "
          f"escolaridade OR/DP {OR_EDU:.2f} para AD", "Discussion", "o quanto a escolaridade mudou o efeito onde foi medida")

BASE["Grego"] = linhas_de_base(subj_gr, COV_GR, {"Age": ["Age"], "Age + sex": COV_GR})
TAB_AUC.append(tabela_aucs(BASE["Grego"], "Grego"))
display(TAB_AUC[-1].round(3))
demo = BASE["Grego"]["Age + sex"]
for rot, outra in ((f"{COMB_ROT} − demographics", BASE["Grego"][COMB_ROT]),
                   (f"{EEG_ROT} − demographics", BASE["Grego"][EEG_ROT])):
    mm = outra.merge(demo[["subject_id", "p"]], on="subject_id", suffixes=("", "_d"))
    DELTA[("Grego", rot)] = delta_boot(mm["y"], mm["p"], mm["p_d"])
    print(f"ΔAUC {rot}: {fmt_ci(*DELTA[('Grego', rot)])} (descritivo)")
registrar("RQ1b", "Grego: escore EEG além de idade e sexo (unilateral)",
          f"χ²(1) = {r['LR']:.2f}; p unilateral (bootstrap nulo) = {fmt_p(BOOT['Grego']['p'])} "
          f"[{BOOT['Grego']['replicas']} réplicas]; p unilateral ingênuo = {fmt_p(r['p_unilateral'])}; OR/DP = "
          f"{fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)} (IC perfilado, descritivo); {r['metodo']}; n = {r['n']}",
          "Results; Table 2", f"leitura: {VEREDITO['RQ1b']}")
registrar("RQ1b", "Grego: E-value (escolaridade não medida)", f"{fmt(E_PT, 2)} (OR); {fmt(E_CI, 2)} (limite do IC)",
          "Results; Discussion")
for _, l in TAB_AUC[-1].iterrows():
    registrar("RQ1b", f"Grego: AUC {l['modelo']}", fmt_ci(l["AUC"], l["IC_inf"], l["IC_sup"]), "Table 2",
              "LOSO" if l["modelo"] == EEG_ROT else "logística, leave-one-out")
for (c, rot), v in DELTA.items():
    if c == "Grego":
        registrar("RQ1b", f"Grego: ΔAUC {rot}", fmt_ci(*v), "Results", "descritivo, sem teste")

### 5.3 RQ1c — O efeito é o mesmo nas duas coortes?

*RQ1c. Is the incremental effect of the EEG score consistent across the two cohorts?*

> **O que faz:** junta as coortes numa logística em que cada coorte tem o próprio intercepto e os próprios efeitos das
> covariáveis que mediu (BrainLat: idade, sexo, escolaridade; grego: idade, sexo) e o escore de EEG, padronizado dentro de
> cada coorte, tem um efeito comum (modelo de um estágio com intercepto e covariáveis estratificados por coorte, como
> recomendam Burke et al., 2017). O OR conjunto combina as RQ1a e RQ1b; o termo coorte × EEG testa se o efeito difere
> entre elas. Os p-valores aqui são os analíticos: servem para descrever, e as decisões de RQ1a e RQ1b ficam com o
> bootstrap.
>
> **Leitura (SPEC):** consistente se a interação tiver p ≥ 0,05 e os dois OR apontarem para o mesmo lado. Com duas coortes
> pequenas, esse teste tem pouco poder: "consistente" quer dizer ausência de evidência de diferença, não prova de igualdade.
>
> **Onde entra:** Results (RQ1) e Fig. 2.

In [ ]:
# 5.3 — RQ1c: modelo conjunto (covariáveis com efeito próprio em cada coorte) e heterogeneidade
def z_dentro(v):
    return padronizar(np.asarray(v, float)[:, None])[:, 0]


bl_p = subj_bl.dropna(subset=COV_BL + ["p_eeg"]).reset_index(drop=True)
gr_p = subj_gr.dropna(subset=COV_GR + ["p_eeg"]).reset_index(drop=True)
n_b, n_g = len(bl_p), len(gr_p)
coorte = np.r_[np.zeros(n_b), np.ones(n_g)]
Zc = np.zeros((n_b + n_g, len(COV_BL) + len(COV_GR)))      # zeros = covariável da outra coorte
for j, c in enumerate(COV_BL):
    Zc[:n_b, j] = z_dentro(bl_p[c])
for j, c in enumerate(COV_GR):
    Zc[n_b:, len(COV_BL) + j] = z_dentro(gr_p[c])
z_eeg = np.r_[z_dentro(logit_seguro(bl_p["p_eeg"])), z_dentro(logit_seguro(gr_p["p_eeg"]))]
yv = np.r_[bl_p["y"].to_numpy(int), gr_p["y"].to_numpy(int)]
X0 = np.column_stack([np.ones(len(yv)), coorte, Zc])
X1 = np.column_stack([X0, z_eeg])
X2 = np.column_stack([X1, z_eeg * coorte])
R_POOL = lr_aninhado(X0, X1, yv, X1.shape[1] - 1)
R_HET = lr_aninhado(X1, X2, yv, X2.shape[1] - 1)
mesmo_lado = (INC["BrainLat"]["OR"] - 1) * (INC["Grego"]["OR"] - 1) > 0
VEREDITO["RQ1c"] = "consistente" if (R_HET["p"] >= 0.05 and mesmo_lado) else "heterogêneo ou incerto"
print(f"Conjunto (n = {len(yv)}: {n_b} BrainLat + {n_g} grego): OR/DP = "
      f"{fmt_ci(R_POOL['OR'], R_POOL['OR_lo'], R_POOL['OR_hi'], 2)}, LR χ²(1) = {R_POOL['LR']:.2f}, "
      f"p = {fmt_p(R_POOL['p'])} ({R_POOL['metodo']})")
print(f"Interação coorte × EEG: χ²(1) = {R_HET['LR']:.2f}, p = {fmt_p(R_HET['p'])} | "
      f"OR na mesma direção nas duas coortes: {mesmo_lado}")
print(f"Leitura (SPEC): {VEREDITO['RQ1c']}")
registrar("RQ1c", "Conjunto: escore EEG além das covariáveis de cada coorte (efeito comum)",
          f"OR/DP = {fmt_ci(R_POOL['OR'], R_POOL['OR_lo'], R_POOL['OR_hi'], 2)}; χ²(1) = {R_POOL['LR']:.2f}, "
          f"p = {fmt_p(R_POOL['p'])}; {R_POOL['metodo']}; n = {len(yv)}", "Results; Fig. 2")
registrar("RQ1c", "Heterogeneidade (coorte × EEG)", f"χ²(1) = {R_HET['LR']:.2f}, p = {fmt_p(R_HET['p'])}",
          "Results", f"leitura: {VEREDITO['RQ1c']}")
r1, r2 = VEREDITO["RQ1a"], VEREDITO["RQ1b"]
VEREDITO["RQ1"] = ("acrescenta nas duas coortes" if r1 == r2 == "acrescenta" else
                   "acrescenta só no BrainLat" if r1 == "acrescenta" else
                   "acrescenta só no grego" if r2 == "acrescenta" else "não demonstrado em nenhuma coorte")
print(f"\nRQ1 no conjunto: {VEREDITO['RQ1']}")

---
## 6. RQ2 — Confusão removida por desenho

### 6.1 RQ2a — BrainLat: pares por escolaridade e overlap weights

*RQ2a. In BrainLat, does the EEG score still discriminate AD from controls within education-matched pairs and under overlap weighting for age, sex, and education?*

> **O que faz:** usa o escore out-of-fold da seção 4, sem retreinar, nos desenhos definidos em 3.5: AUC nos pares 1:1
> (IC por bootstrap de pares; e a distribuição de AUCs sob desempates aleatórios), concordância entre todos os pares com
> escolaridade ±2 anos e AUC com overlap weights para idade, sexo e escolaridade (escore de propensão re-estimado em cada
> réplica do bootstrap). Como verificação, a AUC ajustada por covariáveis de Janes & Pepe (2009), que compara cada AD com
> controles de mesma idade, sexo e escolaridade por meio de uma regressão ajustada só nos controles.
>
> **Base:** medir o desempenho num subconjunto em que o confundidor não se associa ao diagnóstico, sem retreinar o modelo, é
> o que Chyzhyk et al. (2022) propõem para estimar a acurácia livre de confusão. O pareamento segue Stuart (2010), e o IC por
> bootstrap de pares, Austin & Small (2014). Os overlap weights equilibram exatamente as médias das covariáveis do modelo de
> propensão e dão mais peso a quem tem demografia compatível com os dois grupos (Li et al., 2018). A concordância nos pares
> comparáveis é a versão não paramétrica da AUC ajustada por covariáveis (Janes & Pepe, 2008; 2009).
>
> **Leitura (SPEC):** sobrevive se o IC95% ficar acima de 0,5.
>
> **Onde entra:** Results (RQ2), Table 2 e Fig. 3.

In [ ]:
# 6.1 — RQ2a: BrainLat
CONF = []
P_BL = subj_bl.set_index("subject_id")["p_eeg"]
pares_bl = pd.read_csv(f_pares, dtype={"ad_id": str, "hc_id": str})
assert hashlib.sha1(f_pares.read_bytes()).hexdigest()[:12] == DESENHO["sha1_pares"], "pares diferentes do desenho"
CONF.append({"coorte": "BrainLat", "análise": "Full cohort", **dict(zip(("AUC", "lo", "hi"), AUC_EEG["BrainLat"])),
             "n": f"{len(subj_bl)} participants"})
ok_par = N_PARES >= R2["pareamento_brainlat"]["min_pares"]
if N_PARES >= 2:
    a_m = auc_pareada_ci(pares_bl, P_BL)
    aucs_r = []
    for k in range(RT["n_desempates"]):
        pr = parear_1a1(subj_bl, CAL, rng=np.random.default_rng(SEED + 1 + k))
        if len(pr) >= 2:
            aucs_r.append(roc_auc_score(np.r_[np.ones(len(pr)), np.zeros(len(pr))],
                                        P_BL.loc[list(pr.ad_id) + list(pr.hc_id)].to_numpy()))
    print(f"Pareado 1:1 (±{CAL} anos): AUC {fmt_ci(*a_m)} | {N_PARES} pares"
          + ("" if ok_par else " | ABAIXO do mínimo da SPEC: só descritivo"))
    if aucs_r:
        print(f"  desempates aleatórios ({len(aucs_r)}): mediana {np.median(aucs_r):.3f}, "
              f"faixa {np.min(aucs_r):.3f}–{np.max(aucs_r):.3f}")
    CONF.append({"coorte": "BrainLat", "análise": f"Education-matched 1:1 (±{CAL} y)",
                 **dict(zip(("AUC", "lo", "hi"), a_m)), "n": f"{N_PARES} pairs"})
    registrar("RQ2a", f"BrainLat: AUC nos pares 1:1 por escolaridade (±{CAL} anos)",
              f"{fmt_ci(*a_m)}; {N_PARES} pares" + (f"; desempates: mediana {np.median(aucs_r):.3f}, faixa "
                                                   f"{np.min(aucs_r):.3f}–{np.max(aucs_r):.3f}" if aucs_r else ""),
              "Results; Table 2", "" if ok_par else "abaixo do mínimo de pares: só descritivo")
else:
    a_m = (np.nan, np.nan, np.nan)
    alerta("Menos de 2 pares 1:1: a AUC pareada não pode ser calculada.")

c_bl = concordancia_pares(subj_bl.dropna(subset=["years_education", "p_eeg"]), ELEG_BL)
print(f"Concordância em {c_bl['n_pares']} pares comparáveis (escolaridade ±2 anos): {fmt_ci(c_bl['C'], c_bl['lo'], c_bl['hi'])}")
CONF.append({"coorte": "BrainLat", "análise": "Comparable-pair concordance (education ±2 y)",
             "AUC": c_bl["C"], "lo": c_bl["lo"], "hi": c_bl["hi"], "n": f"{c_bl['n_pares']} pairs"})
registrar("RQ2a", "BrainLat: concordância nos pares comparáveis (escolaridade ±2 anos)",
          f"{fmt_ci(c_bl['C'], c_bl['lo'], c_bl['hi'])}; {c_bl['n_pares']} pares", "Results; Table 2")

ow_bl = auc_ponderada_ci(subj_bl, R2["overlap_brainlat"], "p_eeg")
print(f"Overlap weights (idade, sexo, escolaridade): AUC {fmt_ci(ow_bl['AUC'], ow_bl['lo'], ow_bl['hi'])} | "
      f"tamanho efetivo AD {ow_bl['ess_AD']:.1f}, HC {ow_bl['ess_HC']:.1f}")
CONF.append({"coorte": "BrainLat", "análise": "Overlap-weighted (age, sex, education)",
             "AUC": ow_bl["AUC"], "lo": ow_bl["lo"], "hi": ow_bl["hi"],
             "n": f"ESS {ow_bl['ess_AD']:.0f}/{ow_bl['ess_HC']:.0f}"})
registrar("RQ2a", "BrainLat: AUC com overlap weights (idade, sexo, escolaridade)",
          f"{fmt_ci(ow_bl['AUC'], ow_bl['lo'], ow_bl['hi'])}; ESS AD {ow_bl['ess_AD']:.1f}, HC {ow_bl['ess_HC']:.1f}",
          "Results; Table 2")
aa_bl = aauc_controles(subj_bl, COV_BL)
print(f"AUC ajustada por covariáveis (Janes & Pepe; idade, sexo, escolaridade): {fmt_ci(aa_bl['AUC'], aa_bl['lo'], aa_bl['hi'])} "
      f"(n = {aa_bl['n']}; verificação)")
CONF.append({"coorte": "BrainLat", "análise": "Covariate-adjusted AUC (age, sex, education)",
             "AUC": aa_bl["AUC"], "lo": aa_bl["lo"], "hi": aa_bl["hi"], "n": f"{aa_bl['n']} participants"})
registrar("RQ2a", "BrainLat: AUC ajustada por covariáveis (Janes & Pepe; idade, sexo, escolaridade)",
          fmt_ci(aa_bl["AUC"], aa_bl["lo"], aa_bl["hi"]), "Results; Table 2", "verificação; fora da regra de leitura")
sobrev = [ok_par and np.isfinite(a_m[1]) and a_m[1] > 0.5, ow_bl["lo"] > 0.5]
VEREDITO["RQ2a"] = "sobrevive" if all(sobrev) else ("sobrevive em parte" if any(sobrev) else "não demonstrado")
print(f"Leitura (SPEC): {VEREDITO['RQ2a']} (pares 1:1: {sobrev[0]}; overlap: {sobrev[1]})")

### 6.2 RQ2b — Grego: overlap weights e pares comparáveis

*RQ2b. In the Greek cohort, does it still discriminate under overlap weighting for age and sex?*

> **O que faz:** no ds004504 publicado, a idade é parecida entre os grupos, mas a proporção de mulheres não (confira na
> Tabela 1). Calcula a AUC com overlap weights para idade e sexo e, como verificações, a concordância entre pares do mesmo
> sexo com idade ±2 anos e a AUC ajustada por covariáveis de Janes & Pepe (2009).
>
> **Leitura (SPEC):** sobrevive se o IC95% da AUC ponderada ficar acima de 0,5.
>
> **Onde entra:** Results (RQ2), Table 2 e Fig. 3.

In [ ]:
# 6.2 — RQ2b: grego
CONF.append({"coorte": "Grego", "análise": "Full cohort", **dict(zip(("AUC", "lo", "hi"), AUC_EEG["Grego"])),
             "n": f"{len(subj_gr)} participants"})
ow_gr = auc_ponderada_ci(subj_gr, R2["overlap_grego"], "p_eeg")
print(f"Overlap weights (idade, sexo): AUC {fmt_ci(ow_gr['AUC'], ow_gr['lo'], ow_gr['hi'])} | "
      f"tamanho efetivo AD {ow_gr['ess_AD']:.1f}, HC {ow_gr['ess_HC']:.1f}")
CONF.append({"coorte": "Grego", "análise": "Overlap-weighted (age, sex)", "AUC": ow_gr["AUC"], "lo": ow_gr["lo"],
             "hi": ow_gr["hi"], "n": f"ESS {ow_gr['ess_AD']:.0f}/{ow_gr['ess_HC']:.0f}"})
c_gr = concordancia_pares(subj_gr.dropna(subset=["Age", "sex_fem", "p_eeg"]), ELEG_GR)
print(f"Concordância em {c_gr['n_pares']} pares do mesmo sexo com idade ±{CAL_IDADE_GR} anos: "
      f"{fmt_ci(c_gr['C'], c_gr['lo'], c_gr['hi'])}")
CONF.append({"coorte": "Grego", "análise": f"Comparable-pair concordance (same sex, age ±{CAL_IDADE_GR} y)",
             "AUC": c_gr["C"], "lo": c_gr["lo"], "hi": c_gr["hi"], "n": f"{c_gr['n_pares']} pairs"})
aa_gr = aauc_controles(subj_gr, COV_GR)
print(f"AUC ajustada por covariáveis (Janes & Pepe; idade, sexo): {fmt_ci(aa_gr['AUC'], aa_gr['lo'], aa_gr['hi'])} "
      f"(n = {aa_gr['n']}; verificação)")
CONF.append({"coorte": "Grego", "análise": "Covariate-adjusted AUC (age, sex)", "AUC": aa_gr["AUC"], "lo": aa_gr["lo"],
             "hi": aa_gr["hi"], "n": f"{aa_gr['n']} participants"})
registrar("RQ2b", "Grego: AUC ajustada por covariáveis (Janes & Pepe; idade, sexo)",
          fmt_ci(aa_gr["AUC"], aa_gr["lo"], aa_gr["hi"]), "Results; Table 2", "verificação; fora da regra de leitura")
VEREDITO["RQ2b"] = "sobrevive" if ow_gr["lo"] > 0.5 else "não demonstrado"
print(f"Leitura (SPEC): {VEREDITO['RQ2b']}")
registrar("RQ2b", "Grego: AUC com overlap weights (idade, sexo)",
          f"{fmt_ci(ow_gr['AUC'], ow_gr['lo'], ow_gr['hi'])}; ESS AD {ow_gr['ess_AD']:.1f}, HC {ow_gr['ess_HC']:.1f}",
          "Results; Table 2", f"leitura: {VEREDITO['RQ2b']}")
registrar("RQ2b", f"Grego: concordância em pares do mesmo sexo e idade ±{CAL_IDADE_GR} anos",
          f"{fmt_ci(c_gr['C'], c_gr['lo'], c_gr['hi'])}; {c_gr['n_pares']} pares", "Results; Table 2")
TAB_CONF = pd.DataFrame(CONF)
display(TAB_CONF.round(3))
TAB_CONF.to_csv(OUT / "tabelas" / "T2b_confundimento_por_desenho.csv", index=False, encoding="utf-8-sig")

---
## 7. RQ3 — Transporte entre coortes

### 7.1 RQ3a — BrainLat → grego (principal)

*RQ3a. Does the model trained on BrainLat, frozen, discriminate AD from controls in the Greek cohort?*

> **O que faz:** treina o RF da SPEC em todo o BrainLat, grava o modelo com hash e o aplica **uma única vez** às épocas do
> grego, sem os rótulos. Depois junta os rótulos e calcula a AUC por sujeito (principal) e, como descritivos, a inclinação
> de calibração, o intercepto com inclinação fixa em 1 (*calibration-in-the-large*), o Brier e as métricas com limiar 0,5.
> Também testa se o escore transportado acrescenta algo a idade + sexo no grego.
>
> **Base:** separar discriminação de calibração e reportar inclinação e intercepto segue Van Calster et al. (2016; 2019).
> Com 36 AD, a calibração fica imprecisa (validações externas precisam de pelo menos 100 eventos para isso; Collins et al.,
> 2016), por isso ela é só descritiva. Diferenças de *case-mix* entre as coortes (tabela 1 e S2) ajudam a ler se o resultado
> fala de reprodutibilidade ou de transportabilidade (Debray et al., 2015). O teste do escore transportado além da
> demografia é o teste da RQ1 feito em dados independentes do modelo, situação em que o teste paramétrico é válido sem
> reamostragem (Dinga et al., 2020): é a confirmação mais limpa da RQ1.
>
> **Leitura (SPEC):** transporta se o IC95% da AUC ficar acima de 0,5.
>
> **Onde entra:** Results (RQ3), Table 2 e Fig. 1.

In [ ]:
# 7.1 — RQ3a: BrainLat → grego
TRANSP = {}


def avaliar_transporte(pred, subj_dest, rotulo):
    s = pred.merge(subj_dest[["subject_id", "y"] + [c for c in ("Age", "sex_fem", "years_education") if c in subj_dest]],
                   on="subject_id", how="inner")
    a = auc_ci(s["y"], s["p"])
    Xcal = np.column_stack([np.ones(len(s)), logit_seguro(s["p"])])
    cal = logit_fit(Xcal, s["y"].to_numpy(int))
    citl = logit_fit(Xcal, s["y"].to_numpy(int), fixos={1: 1.0})      # intercepto com inclinação fixa em 1
    out = {"rotulo": rotulo, "AUC": a, "n": len(s), "n_AD": int(s.y.sum()), "n_HC": int((1 - s.y).sum()),
           "inclinacao": float(cal["beta"][1]), "citl": float(citl["beta"][0]),
           "brier": float(brier_score_loss(s["y"], s["p"])), "media_p": float(s["p"].mean()),
           "prevalencia": float(s["y"].mean()), "metricas": metricas_limiar(s["y"], s["p"], LIMIAR), "pred": s}
    print(f"{rotulo} (n = {out['n']}: {out['n_AD']} AD, {out['n_HC']} HC): AUC {fmt_ci(*a)} | inclinação de "
          f"calibração {out['inclinacao']:.2f} | calibration-in-the-large {out['citl']:.2f} | Brier {out['brier']:.3f} | "
          f"média de P(AD) {out['media_p']:.2f} vs prevalência {out['prevalencia']:.2f}")
    print(f"   limiar {LIMIAR}: {fmt_metricas(out['metricas'])}")
    return out


pred_a = transportar(df_bl, df_gr, FEATURES, "RandomForest", "RQ3a_BrainLat_para_grego_RF_11")
TRANSP["RQ3a"] = avaliar_transporte(pred_a, subj_gr, "BrainLat → grego")
VEREDITO["RQ3a"] = "transporta" if TRANSP["RQ3a"]["AUC"][1] > 0.5 else "não demonstrado"
print(f"Leitura (SPEC): {VEREDITO['RQ3a']}")
t3 = TRANSP["RQ3a"]
registrar("RQ3a", "BrainLat → grego (modelo congelado): AUC",
          f"{fmt_ci(*t3['AUC'])}; n = {t3['n']} ({t3['n_AD']} AD, {t3['n_HC']} HC); inclinação de calibração "
          f"{t3['inclinacao']:.2f}; calibration-in-the-large {t3['citl']:.2f}; Brier {t3['brier']:.3f}",
          "Results; Table 2", f"leitura: {VEREDITO['RQ3a']}")
registrar("RQ3a", f"BrainLat → grego: métricas com limiar {LIMIAR}", fmt_metricas(t3["metricas"]), "Table 2", "descritivo")
INC["Grego (modelo do BrainLat)"] = teste_incremental(t3["pred"].rename(columns={"p": "p_transp"}), COV_GR,
                                                      "p_transp", "Grego (modelo do BrainLat)")
mostrar_incremental(INC["Grego (modelo do BrainLat)"])
ri = INC["Grego (modelo do BrainLat)"]
registrar("RQ1 (independente)", "Grego: escore do modelo do BrainLat além de idade e sexo",
          f"OR/DP = {fmt_ci(ri['OR'], ri['OR_lo'], ri['OR_hi'], 2)}; p unilateral {fmt_p(ri['p_unilateral'])} "
          "(teste paramétrico válido: modelo independente)", "Results (RQ1 e RQ3)")

### 7.2 RQ3b — Grego → BrainLat

*RQ3b. Does the model trained on the Greek cohort, frozen, discriminate AD from controls in BrainLat?*

> **O que faz:** o caminho inverso, com a mesma regra (modelo congelado, aplicado uma vez, sem os rótulos do destino).
>
> **Onde entra:** Results (RQ3) e Table 2.

In [ ]:
# 7.2 — RQ3b: grego → BrainLat
pred_b = transportar(df_gr, df_bl, FEATURES, "RandomForest", "RQ3b_grego_para_BrainLat_RF_11")
TRANSP["RQ3b"] = avaliar_transporte(pred_b, subj_bl, "Grego → BrainLat")
VEREDITO["RQ3b"] = "transporta" if TRANSP["RQ3b"]["AUC"][1] > 0.5 else "não demonstrado"
print(f"Leitura (SPEC): {VEREDITO['RQ3b']}")
t3 = TRANSP["RQ3b"]
registrar("RQ3b", "Grego → BrainLat (modelo congelado): AUC",
          f"{fmt_ci(*t3['AUC'])}; n = {t3['n']} ({t3['n_AD']} AD, {t3['n_HC']} HC); inclinação de calibração "
          f"{t3['inclinacao']:.2f}; calibration-in-the-large {t3['citl']:.2f}; Brier {t3['brier']:.3f}",
          "Results; Table 2", f"leitura: {VEREDITO['RQ3b']}")
registrar("RQ3b", f"Grego → BrainLat: métricas com limiar {LIMIAR}", fmt_metricas(t3["metricas"]), "Table 2", "descritivo")
INC["BrainLat (modelo do grego)"] = teste_incremental(t3["pred"].rename(columns={"p": "p_transp"}), COV_BL,
                                                      "p_transp", "BrainLat (modelo do grego)")
mostrar_incremental(INC["BrainLat (modelo do grego)"])
ri = INC["BrainLat (modelo do grego)"]
registrar("RQ1 (independente)", "BrainLat: escore do modelo do grego além de idade, sexo e escolaridade",
          f"OR/DP = {fmt_ci(ri['OR'], ri['OR_lo'], ri['OR_hi'], 2)}; p = {fmt_p(ri['p'])} "
          "(teste paramétrico válido: modelo independente)", "Results (RQ1 e RQ3)")
for k, t in TRANSP.items():
    t["pred"].to_csv(OUT / "tabelas" / f"predicoes_transporte_{k}.csv", index=False)

### 7.3 RQ3 (secundária) — transporte com as features padronizadas dentro de cada coorte

> **O que faz:** aparelho, montagem e pré-processamento mudam o nível das features entre as coortes (tabela S2). Aqui cada
> feature é padronizada (média e DP de todas as épocas) dentro da própria coorte, **sem rótulos**, e o transporte é refeito
> nos dois sentidos com a mesma regra (modelo congelado, aplicado uma vez). Se o transporte bruto falhar e este funcionar,
> a falha vem do deslocamento de nível entre aparelhos, e não da falta de um sinal comum. Fica fora da regra de leitura.
>
> **Base:** é o ajuste de posição e escala por sítio que o ComBat faz para features de EEG de repouso em estudos
> multicêntricos (Jaramillo-Jimenez et al., 2024), sem a parte que preserva covariáveis, porque ela exigiria os rótulos do
> destino. A limitação é que a padronização também tira diferenças reais de *case-mix* entre as coortes; aqui elas são
> parecidas (AD ≈ 52% no BrainLat e 55% no grego), mas a idade não é.
>
> **Onde entra:** Results (RQ3, uma frase) e Discussion.

In [ ]:
# 7.3 — RQ3 (secundária): features padronizadas dentro de cada coorte, sem rótulos
def padronizar_coorte(d):
    d = d.copy()
    sd = d[FEATURES].std(ddof=1).replace(0, 1.0)
    d[FEATURES] = (d[FEATURES] - d[FEATURES].mean()) / sd
    return d


df_bl_z, df_gr_z = padronizar_coorte(df_bl), padronizar_coorte(df_gr)
TRANSP_Z = {}
TRANSP_Z["RQ3a"] = avaliar_transporte(transportar(df_bl_z, df_gr_z, FEATURES, "RandomForest", "RQ3a_z_BrainLat_para_grego"),
                                      subj_gr, "BrainLat → grego, padronizado por coorte")
TRANSP_Z["RQ3b"] = avaliar_transporte(transportar(df_gr_z, df_bl_z, FEATURES, "RandomForest", "RQ3b_z_grego_para_BrainLat"),
                                      subj_bl, "Grego → BrainLat, padronizado por coorte")
for k, t in TRANSP_Z.items():
    t["pred"].to_csv(OUT / "tabelas" / f"predicoes_transporte_{k}_padronizado.csv", index=False)
    registrar(k, f"{t['rotulo']}: AUC (secundária)", f"{fmt_ci(*t['AUC'])}; inclinação de calibração "
              f"{t['inclinacao']:.2f}; Brier {t['brier']:.3f}", "Results (uma frase)", "fora da regra de leitura")

---
## 8. Análises exploratória e descritiva

### 8.1 Exploratória — gravidade cognitiva dentro dos AD

*Exploratory. Within AD, does the EEG score track global cognitive severity (MoCA in BrainLat; MMSE in the Greek cohort) beyond age and, where available, education?*

> **O que faz:** dentro dos AD, correlação de Spearman parcial entre o escore de EEG (logit do P(AD) out-of-fold) e o MoCA
> (BrainLat, parcial para idade e escolaridade) ou o MMSE (grego, parcial para idade). Espera-se ρ negativo (escore maior,
> cognição pior). Os instrumentos são diferentes, então as correlações não são comparadas entre si. O IFS do BrainLat entra
> como secundário, para continuidade com o artigo da ACDSA (lá, com o escore de 128 canais, foi o único com associação
> dentro dos AD).
>
> **Onde entra:** Results (análise exploratória) ou material suplementar.

In [ ]:
# 8.1 — Exploratória: escore EEG × gravidade cognitiva dentro dos AD
EXPL = []
if SPEC["exploratoria"]["executar"]:
    for nome_c, s, col, covs, rot in (("BrainLat", subj_bl, "moca_total", ["Age", "years_education"], "MoCA"),
                                      ("BrainLat", subj_bl, "ifs_total_score", ["Age", "years_education"],
                                       "IFS (secondary)"),
                                      ("Grego", subj_gr, "mmse", ["Age"], "MMSE")):
        ad = s[s.y == 1].dropna(subset=[col, "p_eeg"] + covs)
        if len(ad) < 8:
            print(f"{nome_c}: só {len(ad)} AD com {rot}; pulado.")
            continue
        x = logit_seguro(ad["p_eeg"])
        rho0 = stats.spearmanr(x, ad[col])[0]
        rho, lo, hi, p = spearman_parcial_ci(x, ad[col].to_numpy(float), ad[covs].to_numpy(float))
        EXPL.append({"coorte": nome_c, "escala": rot, "n": len(ad), "rho_simples": rho0, "rho_parcial": rho,
                     "IC_inf": lo, "IC_sup": hi, "p": p, "covariaveis": " + ".join(covs)})
        print(f"{nome_c}: AD com {rot} (n = {len(ad)}): ρ simples {rho0:.3f}; ρ parcial ({' + '.join(covs)}) "
              f"{fmt_ci(rho, lo, hi)}, p = {fmt_p(p)}")
        registrar("Exploratória", f"{nome_c}: ρ parcial entre escore EEG e {rot} nos AD",
                  f"{fmt_ci(rho, lo, hi)}, p = {fmt_p(p)}; n = {len(ad)}", "Results (exploratory)")
    if EXPL:
        pd.DataFrame(EXPL).to_csv(OUT / "tabelas" / "S4_exploratoria_gravidade.csv", index=False, encoding="utf-8-sig")
else:
    print("Desligada na SPEC.")

### 8.2 Descritiva — quais das 11 features carregam a diferença além da demografia?

> **O que faz:** para cada feature (média das épocas de cada pessoa, padronizada na coorte), a mesma logística da RQ1
> (covariáveis da coorte + a feature), com OR por 1 DP, IC perfilado e q de Benjamini–Hochberg dentro de cada coorte. Mostra
> se as features que diferenciam AD de controles, além da demografia, são as mesmas nas duas coortes. É descritiva: não
> entra na regra de leitura e não substitui o escore multivariado.
>
> **Onde entra:** Discussion (interpretação) e Suppl. Table S6.

In [ ]:
# 8.2 — Descritiva: cada feature (média por sujeito) além da demografia, nas duas coortes
def bh(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = np.empty(len(p))
    q[o] = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    return np.minimum(q, 1.0)


FEAT_ROWS = []
for nome_c, d, s, covs in (("BrainLat", df_bl, subj_bl, COV_BL), ("Grego", df_gr, subj_gr, COV_GR)):
    medias = d.groupby("subject_id")[FEATURES].mean().reset_index()
    t = s[["subject_id", "y"] + covs].merge(medias, on="subject_id").dropna().reset_index(drop=True)
    yv_f = t["y"].to_numpy(int)
    Xr = np.column_stack([np.ones(len(t)), padronizar(t[covs].to_numpy(float))])
    for f in FEATURES:
        Xf = np.column_stack([Xr, padronizar(t[[f]].to_numpy(float))[:, 0]])
        r = lr_aninhado(Xr, Xf, yv_f, Xf.shape[1] - 1)
        FEAT_ROWS.append({"coorte": nome_c, "feature": f, "n": len(t), "OR": r["OR"], "lo": r["OR_lo"],
                          "hi": r["OR_hi"], "p": r["p"], "metodo": r["metodo"]})
TAB_FEAT = pd.DataFrame(FEAT_ROWS)
TAB_FEAT["q_BH"] = TAB_FEAT.groupby("coorte")["p"].transform(bh)
larga = TAB_FEAT.pivot(index="feature", columns="coorte").reindex(FEATURES)
S6 = pd.DataFrame({
    "BrainLat OR/SD [95% CI]": [fmt_ci(*larga.loc[f, [("OR", "BrainLat"), ("lo", "BrainLat"), ("hi", "BrainLat")]], 2)
                                for f in FEATURES],
    "BrainLat q": [fmt_p(larga.loc[f, ("q_BH", "BrainLat")]) for f in FEATURES],
    "Greek OR/SD [95% CI]": [fmt_ci(*larga.loc[f, [("OR", "Grego"), ("lo", "Grego"), ("hi", "Grego")]], 2)
                             for f in FEATURES],
    "Greek q": [fmt_p(larga.loc[f, ("q_BH", "Grego")]) for f in FEATURES],
    "same direction": [bool((larga.loc[f, ("OR", "BrainLat")] - 1) * (larga.loc[f, ("OR", "Grego")] - 1) > 0)
                       for f in FEATURES]}, index=FEATURES)
display(S6)
S6.to_csv(OUT / "tabelas" / "S6_features_alem_da_demografia.csv", encoding="utf-8-sig")
ambas = [f for f in FEATURES if larga.loc[f, ("q_BH", "BrainLat")] < 0.05 and larga.loc[f, ("q_BH", "Grego")] < 0.05
         and S6.loc[f, "same direction"]]
print(f"Mesma direção nas duas coortes: {int(S6['same direction'].sum())} de {len(FEATURES)} features | "
      f"q < 0,05 nas duas e na mesma direção: {ambas or 'nenhuma'}")
registrar("Descritiva", "Features além da demografia (OR por DP; BH dentro da coorte)",
          f"mesma direção em {int(S6['same direction'].sum())} de {len(FEATURES)}; q < 0,05 nas duas: "
          f"{', '.join(ambas) or 'nenhuma'}", "Discussion; Suppl. Table S6", "descritiva")

---
## 9. Análises de sensibilidade (pré-especificadas)

> **O que faz:** repete as análises centrais (AUC interna, RQ1a, RQ1b, RQ3a e RQ3b) com (1) só as 7 features espectrais,
> (2) sem a gama relativa, (3) sem as épocas com potência relativa de gama acima de 0,15 e (4) SVM e XGBoost no lugar do RF.
> A última coluna diz se a leitura de alguma pergunta muda em relação à análise principal; se mudar, isso entra no artigo
> como limitação. Refazer o bootstrap nulo para cada variante custaria horas; por isso, aqui a RQ1 usa o p analítico, e a
> comparação é feita com o p analítico da análise principal.
>
> **Base:** a maior parte da potência de EEG de escalpo acima de 20 Hz é de origem muscular (Whitham et al., 2007), e as
> coortes removeram artefatos de forma diferente (ICA de mandíbula só no grego); por isso as variantes (2) e (3).
>
> **Onde entra:** Results (um parágrafo) e Suppl. Table S5.

In [ ]:
# 9 — Sensibilidade
def resumo_variante(nome, modelo, feats, gama=None):
    dbl = df_bl if gama is None else df_bl[df_bl["Rel_Gamma_mean"] <= gama].reset_index(drop=True)
    dgr = df_gr if gama is None else df_gr[df_gr["Rel_Gamma_mean"] <= gama].reset_index(drop=True)
    suf = re.sub(r"[^A-Za-z0-9]+", "_", nome).strip("_")
    if nome == "Principal (RF, 11 features)":
        pbl, pgr = PRED["BrainLat"], PRED["Grego"]
        tbg, tgb = TRANSP["RQ3a"]["pred"], TRANSP["RQ3b"]["pred"]
    else:
        pbl = run_loso(dbl, feats, modelo, f"BL_{suf}")
        pgr = run_loso(dgr, feats, modelo, f"GR_{suf}")
        tbg = transportar(dbl, dgr, feats, modelo, f"S_{suf}_BL_para_GR").merge(subj_gr[["subject_id", "y"]], on="subject_id")
        tgb = transportar(dgr, dbl, feats, modelo, f"S_{suf}_GR_para_BL").merge(subj_bl[["subject_id", "y"]], on="subject_id")
    sb = subj_bl[["subject_id"] + COV_BL].merge(pbl[["subject_id", "y", "p"]], on="subject_id")
    sg = subj_gr[["subject_id"] + COV_GR].merge(pgr[["subject_id", "y", "p"]], on="subject_id")
    ib, ig = teste_incremental(sb, COV_BL, "p", "BrainLat"), teste_incremental(sg, COV_GR, "p", "Grego")
    a_bl, a_gr = auc_ci(sb["y"], sb["p"]), auc_ci(sg["y"], sg["p"])
    a_bg, a_gb = auc_ci(tbg["y"], tbg["p"]), auc_ci(tgb["y"], tgb["p"])
    v = {"RQ1a": "acrescenta" if ib["p"] < 0.05 else "não demonstrado",
         "RQ1b": "acrescenta" if ig["p_unilateral"] < 0.05 else "não demonstrado",
         "RQ3a": "transporta" if a_bg[1] > 0.5 else "não demonstrado",
         "RQ3b": "transporta" if a_gb[1] > 0.5 else "não demonstrado"}
    ref = {"RQ1a": "acrescenta" if INC["BrainLat"]["p"] < 0.05 else "não demonstrado",      # p analítico da principal
           "RQ1b": "acrescenta" if INC["Grego"]["p_unilateral"] < 0.05 else "não demonstrado",
           "RQ3a": VEREDITO["RQ3a"], "RQ3b": VEREDITO["RQ3b"]}
    muda = [k for k in v if v[k] != ref[k]]
    return {"variante": nome, "sujeitos BL/GR": f"{len(sb)}/{len(sg)}",
            "AUC BrainLat": fmt_ci(*a_bl), "RQ1a OR/DP": fmt_ci(ib["OR"], ib["OR_lo"], ib["OR_hi"], 2),
            "RQ1a p": fmt_p(ib["p"]), "AUC grego": fmt_ci(*a_gr),
            "RQ1b OR/DP": fmt_ci(ig["OR"], ig["OR_lo"], ig["OR_hi"], 2), "RQ1b p unilateral": fmt_p(ig["p_unilateral"]),
            "RQ3a BL→GR": fmt_ci(*a_bg), "RQ3b GR→BL": fmt_ci(*a_gb),
            "muda a leitura de": ", ".join(muda) or "nenhuma",
            "_or": {"BrainLat": (ib["OR"], ib["OR_lo"], ib["OR_hi"]), "Grego": (ig["OR"], ig["OR_lo"], ig["OR_hi"])}}


VARIANTES = [("Principal (RF, 11 features)", "RandomForest", FEATURES, None)]
SENS = SPEC["sensibilidade"]
if SENS["so_espectrais"]:
    VARIANTES.append(("RF, 7 spectral features", "RandomForest", ESPECTRAIS, None))
if SENS["sem_gama"]:
    VARIANTES.append(("RF, 10 features (no gamma)", "RandomForest", [f for f in FEATURES if f != "Rel_Gamma_mean"], None))
if SENS["excluir_gama_acima_de"]:
    g = SENS["excluir_gama_acima_de"]
    VARIANTES.append((f"RF, epochs with gamma RP > {g} removed", "RandomForest", FEATURES, g))
    for nome_c, d in (("BrainLat", df_bl), ("Grego", df_gr)):
        rest = d[d["Rel_Gamma_mean"] <= g].groupby("subject_id").size()
        print(f"{nome_c}: épocas com RPγ > {g}: {100 * np.mean(d['Rel_Gamma_mean'] > g):.1f}% | "
              f"sujeitos sem nenhuma época restante: {d.subject_id.nunique() - len(rest)}")
for m in SENS["classificadores"]:
    if m in FABRICAS:
        VARIANTES.append((f"{m}, 11 features", m, FEATURES, None))
    else:
        alerta(f"Sensibilidade com {m} pulada (biblioteca ausente).")

SENS_ROWS = []
for nome, modelo, feats, gama in VARIANTES:
    print(f"\n— {nome}")
    SENS_ROWS.append(resumo_variante(nome, modelo, feats, gama))
TAB_SENS = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in SENS_ROWS])
display(TAB_SENS)
TAB_SENS.to_csv(OUT / "tabelas" / "S5_sensibilidade.csv", index=False, encoding="utf-8-sig")
for r in SENS_ROWS[1:]:
    registrar("Sensibilidade", r["variante"],
              f"RQ1a OR {r['RQ1a OR/DP']} (p {r['RQ1a p']}); RQ1b OR {r['RQ1b OR/DP']} (p unil. {r['RQ1b p unilateral']}); "
              f"BL→GR {r['RQ3a BL→GR']}; GR→BL {r['RQ3b GR→BL']}", "Results; Suppl. Table S5",
              f"muda a leitura de: {r['muda a leitura de']}")

---
## 10. Figuras

> **O que faz:** Fig. 1 — curvas ROC nas duas coortes (EEG interno, modelo da outra coorte, demografia e demografia + EEG);
> Fig. 2 — OR por 1 DP do escore de EEG além da demografia (BrainLat, grego, conjunto e sensibilidades); Fig. 3 — AUC na
> coorte completa e nos desenhos sem confusão. PDFs com fontes TrueType em `outputs_journal/figuras/`.

In [ ]:
# 10 — Figuras
from matplotlib.ticker import NullFormatter
from matplotlib.lines import Line2D


def limpar_eixos(ax):
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)


def curva(ax, y, p, cor, rot, estilo="-"):
    fpr, tpr, _ = roc_curve(y, p)
    ax.plot(fpr, tpr, estilo, color=cor, label=f"{rot}: {roc_auc_score(y, p):.2f}")


# Fig. 1 — ROC nas duas coortes
fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.0))
for ax, nome_c, demo_rot, transp in ((axes[0], "BrainLat", "Age + sex + education", "RQ3b"),
                                     (axes[1], "Grego", "Age + sex", "RQ3a")):
    b = BASE[nome_c]
    curva(ax, b[EEG_ROT]["y"], b[EEG_ROT]["p"], CORES["EEG"], "EEG, within-cohort model")
    tp = TRANSP[transp]["pred"]
    curva(ax, tp["y"], tp["p"], CORES["EEG_outra"], "EEG, other cohort's model", "--")
    curva(ax, b[demo_rot]["y"], b[demo_rot]["p"], CORES["Demo"], demo_rot)
    curva(ax, b[COMB_ROT]["y"], b[COMB_ROT]["p"], CORES["Demo_EEG"], f"{demo_rot} + EEG", ":")
    ax.plot([0, 1], [0, 1], color="0.75", lw=0.6)
    ax.set_title("BrainLat" if nome_c == "BrainLat" else "Greek cohort (ds004504)")
    ax.set_xlabel("1 − specificity")
    ax.set_ylabel("Sensitivity")
    ax.legend(frameon=False, loc="lower right", title="AUC", title_fontsize=6.5, fontsize=6.5, handlelength=1.8)
    ax.set_aspect("equal")
    limpar_eixos(ax)
plt.tight_layout()
salvar_fig(fig, "fig1_roc_duas_coortes")
plt.show()

# Fig. 2 — OR por 1 DP do escore de EEG além da demografia
linhas_f = [("BrainLat (age, sex, education)", *(INC["BrainLat"][k] for k in ("OR", "OR_lo", "OR_hi"))),
            ("Greek (age, sex)", *(INC["Grego"][k] for k in ("OR", "OR_lo", "OR_hi"))),
            ("Pooled (cohort-specific demographics)", R_POOL["OR"], R_POOL["OR_lo"], R_POOL["OR_hi"]),
            ("BrainLat: score from the Greek model", *(INC["BrainLat (modelo do grego)"][k] for k in ("OR", "OR_lo", "OR_hi"))),
            ("Greek: score from the BrainLat model", *(INC["Grego (modelo do BrainLat)"][k] for k in ("OR", "OR_lo", "OR_hi"))),
            ("BrainLat: + site", *(INC["BrainLat + sítio"][k] for k in ("OR", "OR_lo", "OR_hi")))]
for r in SENS_ROWS[1:]:
    for nome_c, rot in (("BrainLat", "BrainLat"), ("Grego", "Greek")):
        linhas_f.append((f"{rot}: {r['variante']}", *r["_or"][nome_c]))
fig, ax = plt.subplots(figsize=(COL2, 0.22 * len(linhas_f) + 0.7))
for i, (rot, o, lo, hi) in enumerate(linhas_f):
    yy = len(linhas_f) - 1 - i
    principal = i < 5
    ax.errorbar(o, yy, xerr=[[o - lo if np.isfinite(lo) else 0], [hi - o if np.isfinite(hi) else 0]],
                fmt="D" if i == 2 else "s" if i in (3, 4) else "o", color="black" if principal else "0.55", ms=4 if principal else 3,
                capsize=2, lw=0.8)
ax.axhline(len(linhas_f) - 3.5, color="0.85", lw=0.6)
ax.axhline(len(linhas_f) - 5.5, color="0.85", lw=0.6)
ax.axvline(1, color="0.6", lw=0.6, ls="--")
ax.set_yticks(range(len(linhas_f)))
ax.set_yticklabels([r[0] for r in linhas_f][::-1])
ax.set_xscale("log")
vals = np.array([v for r in linhas_f for v in r[1:] if np.isfinite(v) and v > 0])
marcas = [t for t in (0.125, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256) if vals.min() / 1.5 <= t <= vals.max() * 1.5]
ax.set_xticks(marcas)
ax.set_xticklabels([f"{t:g}" for t in marcas])
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("Odds ratio per 1 SD of the EEG score, beyond demographics (95% CI)")
limpar_eixos(ax)
plt.tight_layout()
salvar_fig(fig, "fig2_or_incremental")
plt.show()

# Fig. 3 — AUC/concordância na coorte completa e nos desenhos sem confusão
CURTO = {"Full cohort": "Full cohort",
         f"Education-matched 1:1 (±{CAL} y)": "Matched pairs (education)",
         "Comparable-pair concordance (education ±2 y)": "Comparable pairs (education)",
         "Overlap-weighted (age, sex, education)": "Overlap weights (age, sex, educ.)",
         "Overlap-weighted (age, sex)": "Overlap weights (age, sex)",
         "Covariate-adjusted AUC (age, sex, education)": "Adjusted AUC (age, sex, educ.)",
         "Covariate-adjusted AUC (age, sex)": "Adjusted AUC (age, sex)",
         f"Comparable-pair concordance (same sex, age ±{CAL_IDADE_GR} y)": "Comparable pairs (sex, age)"}
tc = TAB_CONF.reset_index(drop=True)
fig, ax = plt.subplots(figsize=(COL1, 0.27 * len(tc) + 0.75))
for i, l in tc.iterrows():
    yy = len(tc) - 1 - i
    cor = CORES["EEG"] if l["coorte"] == "BrainLat" else CORES["Demo"]
    ax.errorbar(l["AUC"], yy, xerr=[[l["AUC"] - l["lo"]], [l["hi"] - l["AUC"]]], fmt="o", color=cor, ms=3.5,
                capsize=2, lw=0.8)
ax.axvline(0.5, color="0.6", lw=0.6, ls="--")
ax.set_yticks(range(len(tc)))
ax.set_yticklabels([CURTO.get(a, a) for a in tc["análise"]][::-1])
ax.set_xlabel("AUC or concordance (95% CI)")
ax.set_xlim(max(0.0, min(0.45, np.nanmin(tc["lo"]) - 0.05)), 1.0)
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=CORES["EEG"], ms=3.5, label="BrainLat"),
                   Line2D([], [], marker="o", ls="", color=CORES["Demo"], ms=3.5, label="Greek")],
          frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=6.5)
limpar_eixos(ax)
plt.tight_layout()
salvar_fig(fig, "fig3_confundimento")
plt.show()

---
## 11. Resumo e pacote para enviar

Grava `outputs_journal/RESUMO_JOURNAL.md` (todos os números, as leituras pela regra da SPEC e os alertas), a Tabela 2
(`T2_resultados_principais.csv`) e `para_enviar.zip` com resumo, tabelas, figuras, SPEC e log.

In [ ]:
# 11 — Resumo, Tabela 2 e zip
T2 = []
for nome_c in ("BrainLat", "Grego"):
    r, bn = INC[nome_c], BOOT[nome_c]
    T2.append({"bloco": "RQ1 — incremental value", "coorte": nome_c,
               "análise": f"Within-cohort EEG score beyond {r['covariaveis']}",
               "estimativa": f"OR/SD {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)}",
               "p": (f"null bootstrap {'one-sided ' if bn['unilateral'] else ''}{fmt_p(bn['p'])} "
                     f"({bn['replicas']} replicates); analytic {fmt_p(r['p_unilateral'] if bn['unilateral'] else r['p'])}"),
               "leitura": VEREDITO["RQ1a" if nome_c == "BrainLat" else "RQ1b"]})
for chave, rot in (("BrainLat (modelo do grego)", "BrainLat"), ("Grego (modelo do BrainLat)", "Grego")):
    r = INC[chave]
    T2.append({"bloco": "RQ1 — incremental value", "coorte": rot,
               "análise": f"Score from the other cohort's model beyond {r['covariaveis']} (independent)",
               "estimativa": f"OR/SD {fmt_ci(r['OR'], r['OR_lo'], r['OR_hi'], 2)}",
               "p": fmt_p(r["p_unilateral"]) + " one-sided" if rot == "Grego" else fmt_p(r["p"]),
               "leitura": "confirmação independente"})
rs = INC["BrainLat + sítio"]
T2.append({"bloco": "RQ1 — incremental value", "coorte": "BrainLat", "análise": "Check: + site",
           "estimativa": f"OR/SD {fmt_ci(rs['OR'], rs['OR_lo'], rs['OR_hi'], 2)}", "p": fmt_p(rs["p"]),
           "leitura": "verificação"})
T2.append({"bloco": "RQ1 — incremental value", "coorte": "Pooled",
           "análise": "EEG score beyond cohort-specific demographics (common EEG effect)",
           "estimativa": f"OR/SD {fmt_ci(R_POOL['OR'], R_POOL['OR_lo'], R_POOL['OR_hi'], 2)}",
           "p": f"{fmt_p(R_POOL['p'])}; cohort × EEG {fmt_p(R_HET['p'])}", "leitura": VEREDITO["RQ1c"]})
for _, l in pd.concat(TAB_AUC).iterrows():
    T2.append({"bloco": "Discrimination (logistic: leave-one-out; EEG: LOSO)", "coorte": l["coorte"],
               "análise": l["modelo"], "estimativa": f"AUC {fmt_ci(l['AUC'], l['IC_inf'], l['IC_sup'])}",
               "p": "", "leitura": ""})
for nome_c, m in MET_EEG.items():
    T2.append({"bloco": "Discrimination (logistic: leave-one-out; EEG: LOSO)", "coorte": nome_c,
               "análise": f"EEG score, threshold {LIMIAR}", "estimativa": fmt_metricas(m), "p": "", "leitura": ""})
for _, l in TAB_CONF.iterrows():
    T2.append({"bloco": "RQ2 — confounding removed by design", "coorte": l["coorte"], "análise": l["análise"],
               "estimativa": f"{fmt_ci(l['AUC'], l['lo'], l['hi'])} ({l['n']})", "p": "",
               "leitura": "" if l["análise"] == "Full cohort" else VEREDITO["RQ2a" if l["coorte"] == "BrainLat" else "RQ2b"]})
for k, t in TRANSP.items():
    T2.append({"bloco": "RQ3 — transportability", "coorte": "Grego" if k == "RQ3a" else "BrainLat",
               "análise": t["rotulo"], "estimativa": f"AUC {fmt_ci(*t['AUC'])}; {fmt_metricas(t['metricas'])}",
               "p": f"calibration slope {t['inclinacao']:.2f}; calibration-in-the-large {t['citl']:.2f}",
               "leitura": VEREDITO[k]})
for k, t in TRANSP_Z.items():
    T2.append({"bloco": "RQ3 — transportability", "coorte": "Grego" if k == "RQ3a" else "BrainLat",
               "análise": t["rotulo"], "estimativa": f"AUC {fmt_ci(*t['AUC'])}",
               "p": f"calibration slope {t['inclinacao']:.2f}; calibration-in-the-large {t['citl']:.2f}",
               "leitura": "secundária"})
TAB2 = pd.DataFrame(T2)
TAB2.to_csv(OUT / "tabelas" / "T2_resultados_principais.csv", index=False, encoding="utf-8-sig")
display(TAB2)

dur = (time.time() - T0_GLOBAL) / 60
cab = [f"# Journal — EEG × demografia em duas coortes — {datetime.datetime.now().isoformat(timespec='seconds')}", "",
       f"**{SPEC['titulo_de_trabalho']}**", "",
       f"Modo: {'RÁPIDO (NÃO USAR)' if RAPIDO else 'completo'} | SPEC {SPEC_HASH}, congelada em {SPEC_CONGELADA_EM} | "
       f"rótulos gregos lidos pela primeira vez em {LOG.get('rotulos_gregos_lidos_em', 'n/d')} | duração {dur:.0f} min", "",
       f"ds004504 v{DS_VERSAO} (doi:{DS_DOI}); DOI na pasta: {CONF_DL.get('doi_na_pasta') or 'n/d'}; "
       f"{CONF_DL.get('arquivos_ok')} de {CONF_DL.get('arquivos_manifesto')} arquivos conferidos por SHA-256", "",
       " | ".join(f"{k} {v}" for k, v in VERSOES.items() if k != "sistema"), "",
       "## Leituras pela regra da SPEC", "", "| Pergunta | Leitura |", "|---|---|"]
def esc(s):
    return str(s).replace("|", "\\|")


cab += [f"| {k} | {v} |" for k, v in VEREDITO.items()]
cab += ["", "## Resultados", "", "| RQ | Item | Valor | Onde entra | Nota |", "|---|---|---|---|---|"]
cab += ["| " + " | ".join(esc(r[c]) for c in ("rq", "item", "valor", "onde", "nota")) + " |" for r in REG]
cab += ["", "## Alertas", ""] + ([f"- {a}" for a in ALERTAS] or ["- nenhum"])
if SPEC["historico_alteracoes"] or len(LOG.get("congelamentos", [])) > 1:
    cab += ["", "## Alterações da SPEC depois do primeiro congelamento", ""]
    cab += [f"- congelamento: {c['congelada_em']} (hash {c['hash']})" for c in LOG.get("congelamentos", [])]
    cab += [f"- {h}" for h in SPEC["historico_alteracoes"]]
f_res = OUT / "RESUMO_JOURNAL.md"
f_res.write_text("\n".join(cab) + "\n", encoding="utf-8")

f_zip = OUT / "para_enviar.zip"
with zipfile.ZipFile(f_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [f_res, f_spec, f_log, OUT / "ambiente.txt", OUT / "tabelas" / "desenho_confundimento.json",
              OUT / "download_ds004504.json"]:
        if f.exists():
            z.write(f, f.name)
    for pasta in ("tabelas", "figuras"):
        for f in sorted((OUT / pasta).glob("*")):
            if f.suffix in (".csv", ".pdf", ".png", ".json"):
                z.write(f, f"{pasta}/{f.name}")
print(f"\nLeituras: {VEREDITO}")
print(f"Resumo: {f_res}")
print(f"Pacote para enviar: {f_zip}")
print(f"Duração total: {dur:.1f} min")
if ALERTAS:
    print(f"\n{len(ALERTAS)} alerta(s): veja o fim do resumo.")

---
## 12. Base na literatura

| Escolha no notebook | Por quê | Referência |
|---|---|---|
| ds004504: `derivatives` (0,5–45 Hz, ASR, ICA de olho e mandíbula), sem escolaridade, AD e controles | Descrição oficial do banco e dos metadados | [Miltiadous et al., 2023](https://doi.org/10.3390/data8060095); [ds004504 v1.0.9](https://doi.org/10.18112/openneuro.ds004504.v1.0.9) |
| BrainLat: BioSemi 128, referência média, Butterworth 0,5–40 Hz (ordem 8), ICA ocular, 512 Hz | Descrição oficial do banco; base para igualar a banda do ds004504 em 40 Hz | [Prado et al., 2023](https://doi.org/10.1038/s41597-023-02806-8) |
| LOSO, pipeline aninhado, épocas como unidade, várias métricas | Critérios C1–C7 (Validity 1) do benchmark do ds004504; estudos com vazamento relatam ~97% de acurácia, com validação por sujeito ~82% | [Miltiadous et al., 2026](https://doi.org/10.1007/s11571-026-10464-w) |
| Valor incremental: logística demografia vs demografia + escore | Testar o coeficiente do novo marcador basta; testar ΔAUC é redundante e tem pouco poder | [Vickers et al., 2011](https://doi.org/10.1186/1471-2288-11-13); [Pepe et al., 2013](https://doi.org/10.1002/sim.5727) |
| Predições de ML ajustadas por confundidores no nível da predição | Proposta original do teste de predição além dos confundidores | [Dinga et al., 2020](https://doi.org/10.1101/2020.08.17.255034) |
| p-valor por bootstrap nulo de todo o LOSO | O teste analítico com escores de validação cruzada (*pre-validation*) pode ser enviesado; correção por reamostragem que preserva a correlação entre os blocos de variáveis | [Höfling & Tibshirani, 2008](https://doi.org/10.1214/07-AOAS152); [Shang et al., 2025](https://arxiv.org/abs/2505.14985) |
| Parada sequencial do bootstrap | Para cedo quando o p já é claramente grande, sem perder validade | [Besag & Clifford, 1991](https://doi.org/10.1093/biomet/78.2.301) |
| Firth quando há separação; IC por verossimilhança perfilada | Solução padrão para separação em logística | [Heinze & Schemper, 2002](https://doi.org/10.1002/sim.1047) |
| E-value para a escolaridade não medida; comparação com a força da escolaridade no BrainLat | Sensibilidade a confundimento não medido | [VanderWeele & Ding, 2017](https://doi.org/10.7326/M16-2607) |
| Modelo conjunto com intercepto e covariáveis por coorte; interação coorte × EEG | Recomendação para modelos de um estágio com dados individuais de vários estudos | [Burke et al., 2017](https://doi.org/10.1002/sim.7141) |
| Avaliar o escore em subconjuntos sem associação entre confundidor e diagnóstico, sem retreinar | Acurácia livre de confusão | [Chyzhyk et al., 2022](https://doi.org/10.1093/gigascience/giac014) |
| Pareamento 1:1 com calibre; IC por bootstrap de pares | Pareamento e variância em amostras pareadas | [Stuart, 2010](https://doi.org/10.1214/09-STS313); [Austin & Small, 2014](https://doi.org/10.1002/sim.6276) |
| Overlap weights | Equilíbrio exato das médias e foco em quem é compatível com os dois grupos | [Li et al., 2018](https://doi.org/10.1080/01621459.2016.1260466) |
| AUC ajustada por covariáveis e concordância em pares comparáveis | Medida padrão de acurácia ajustada por covariáveis | [Janes & Pepe, 2008](https://doi.org/10.1093/aje/kwn099); [Janes & Pepe, 2009](https://doi.org/10.1093/biomet/asp002) |
| Transporte: AUC, inclinação, *calibration-in-the-large*, Brier; calibração só descritiva | Hierarquia de calibração; validação externa precisa de ≥ 100 eventos para calibração precisa | [Van Calster et al., 2016](https://doi.org/10.1016/j.jclinepi.2015.12.005); [Van Calster et al., 2019](https://doi.org/10.1186/s12916-019-1466-7); [Collins et al., 2016](https://doi.org/10.1002/sim.6787) |
| Ler o transporte à luz das diferenças de *case-mix* | Reprodutibilidade vs transportabilidade | [Debray et al., 2015](https://doi.org/10.1016/j.jclinepi.2014.06.018) |
| Padronização das features por coorte (secundária) | Efeito de sítio em features de EEG de repouso; versão sem rótulos do ajuste de posição e escala do ComBat | [Jaramillo-Jimenez et al., 2024](https://doi.org/10.1016/j.clinph.2024.09.019) |
| Sensibilidades sem gama | EEG de escalpo acima de 20 Hz é dominado por EMG | [Whitham et al., 2007](https://doi.org/10.1016/j.clinph.2007.04.027) |
| Checklist de relato e fluxograma (seção 13) | Guia de relato para modelos de predição com regressão ou ML; substitui o TRIPOD 2015 | [Collins et al., 2024](https://doi.org/10.1136/bmj-2023-078378) |
| Estágios de gravidade pelo MMSE (seção 13.2) | Faixas do MMSE mapeadas nos estágios do CDR: 21–25 leve, 11–20 moderada, 0–10 grave | [Perneczky et al., 2006](https://doi.org/10.1097/01.JGP.0000192478.82189.a8) |
| As 11 features (potência relativa, razões, entropia, sincronização) | Alterações de delta, teta e alfa e de "interrelatedness" são os achados mais consistentes no EEG de repouso da DA | [Babiloni et al., 2021](https://doi.org/10.1002/alz.12311) |

**Limitações que a literatura aponta e o notebook não resolve**
- O PLV é sensível à condução de volume e à referência; medidas como o PLI sofrem menos com fontes comuns
  ([Stam et al., 2007](https://doi.org/10.1002/hbm.20346)). Ficou o PLV para manter as 11 features do artigo; vale dizer
  isso na Discussão.
- AUC de LOSO calculada juntando as predições de todos os folds pode ser pessimista em amostras pequenas
  ([Parker et al., 2007](https://doi.org/10.1186/1471-2105-8-326)). O peso balanceado das classes reduz parte disso; a
  inferência da RQ1 não depende dessa AUC.
- Com 35–36 AD por coorte, a calibração no transporte é imprecisa ([Collins et al., 2016](https://doi.org/10.1002/sim.6787)).
- Diferenças de aparelho, referência original e limpeza de artefatos (ICA ocular no BrainLat; ASR e ICA ocular e de
  mandíbula no ds004504) não podem ser igualadas a partir dos arquivos distribuídos; a tabela S2 mostra o tamanho delas.

---
## 13. Material para a submissão: fluxograma, gravidade, TRIPOD+AI e pacote do preprint

> **O que faz:** monta quatro coisas que revisores e o medRxiv pedem. Nenhuma delas muda os resultados.
> - 13.1: fluxograma dos participantes (TRIPOD+AI, item 20a).
> - 13.2: tabela de gravidade cognitiva por coorte e comparação entre incluídos e excluídos do BrainLat (itens 20b e 26).
> - 13.3: checklist TRIPOD+AI pré-preenchido, dizendo onde cada item já está no notebook e o que falta escrever.
> - 13.4: a pasta `pacote_preprint/`, com figuras, tabelas, checklist e rascunhos das declarações, pronta para virar o PDF
>   do medRxiv. Os arquivos novos também entram em `para_enviar.zip`.
>
> **O que não faz:** postar o preprint. O medRxiv exige o manuscrito em PDF, o consentimento de todos os autores e as
> declarações assinadas por vocês. O passo a passo fica em `pacote_preprint/LEIA_ME_PREPRINT.md`.
>
> **Precisa de:** seções 1 a 3 rodadas (são rápidas, as features vêm do cache). O normal é rodar depois da seção 11.

In [ ]:
# 13.1 — Fluxograma dos participantes (TRIPOD+AI 20a e 21)
import shutil
import textwrap

faltam_v = [v for v in ("subj_bl", "subj_gr", "arq_bl", "demo_all", "part", "fal_bl", "fal_gr", "df_cog")
            if v not in globals()]
if faltam_v:
    raise RuntimeError(f"Rode as seções 1 a 3 antes desta (faltam: {faltam_v}).")
COV_BL_ = list(SPEC["coortes"]["brainlat"]["covariaveis"])
COV_GR_ = list(SPEC["coortes"]["grego"]["covariaveis"])
NOME_COV = {"Age": "age", "sex_fem": "sex", "years_education": "education"}


def nomes_cov(covs):
    return ", ".join(NOME_COV.get(c, c) for c in covs)


def contagem(d):
    return {"n": int(len(d)), "AD": int((d["y"] == 1).sum()), "HC": int((d["y"] == 0).sum())}


def motivo_falha(erro):
    e = str(erro).lower()
    if ".fdt" in e or "no such file" in e or "not found" in e:
        return "EEG data file (.fdt) missing"
    if "época" in e or "epoch" in e:
        return "no valid 4-s epoch"
    if "canal" in e or "channel" in e:
        return "required channels missing"
    return "file could not be read"


def motivos(ids, fal):
    if not len(ids):
        return {}
    m = dict(zip(fal["subject_id"].astype(str), fal["erro"])) if len(fal) else {}
    cont = pd.Series([motivo_falha(m[i]) if i in m else "no valid 4-s epoch" for i in ids]).value_counts()
    return cont.to_dict()


FLUXO = []


def etapa(coorte, nome, d, excluidos=None, motivo=None):
    c = contagem(d)
    FLUXO.append({"coorte": coorte, "etapa": nome, **c,
                  "excluidos_na_etapa": "" if excluidos is None else excluidos,
                  "motivo": "" if not motivo else "; ".join(f"{k} (n = {v})" for k, v in motivo.items())})
    return c


# BrainLat
demo_ids = (demo_all.assign(y=demo_all["grupo_pasta"].map(SPEC["rotulos"]))
            .drop_duplicates("subject_id")[["subject_id", "y"]])
sem_set = demo_ids[~demo_ids.subject_id.isin(arq_bl.subject_id)]
excl_bl_ids = sorted(set(arq_bl.subject_id) - set(subj_bl.subject_id))
mot_bl = motivos(excl_bl_ids, fal_bl)
etapa("BrainLat", "Listed in the distributed demographic files (AD and HC folders)", demo_ids,
      len(sem_set), {"no resting-state EEG file (.set)": len(sem_set)} if len(sem_set) else None)
etapa("BrainLat", "Resting-state EEG file (.set) available", arq_bl, len(excl_bl_ids), mot_bl)
cc_bl = subj_bl.dropna(subset=COV_BL_)
etapa("BrainLat", "Analysable EEG: within-cohort LOSO (RQ1a, RQ2a) and transport target (RQ3b)", subj_bl,
      len(subj_bl) - len(cc_bl), {"missing covariate": len(subj_bl) - len(cc_bl)} if len(subj_bl) > len(cc_bl) else None)
etapa("BrainLat", f"Complete covariates ({nomes_cov(COV_BL_)}): incremental-value model", cc_bl)
etapa("BrainLat", "AD with MoCA (exploratory)",
      subj_bl[(subj_bl.y == 1) & subj_bl.moca_total.notna()])

# ds004504
part_bruto = pd.read_csv(DS_DIR / "participants.tsv", sep="\t")
part_bruto.columns = [c.strip().lower() for c in part_bruto.columns]
grp = part_bruto["group"].astype(str).str.strip().str.upper()
n_bruto = {"n": int(len(part_bruto)), "AD": int((grp == "A").sum()), "HC": int((grp == "C").sum())}
n_ftd = int((~grp.isin(list(SPEC["coortes"]["grego"]["grupos"]))).sum())
FLUXO.append({"coorte": "Grego", "etapa": "Listed in participants.tsv (ds004504 v1.0.9)", **n_bruto, "FTD": n_ftd,
              "excluidos_na_etapa": n_ftd, "motivo": f"frontotemporal dementia, excluded by design (n = {n_ftd})"})
com_set_gr = part[part.arquivo.notna()]
etapa("Grego", "AD and cognitively normal controls", part, len(part) - len(com_set_gr),
      {"no .set file": len(part) - len(com_set_gr)} if len(part) > len(com_set_gr) else None)
excl_gr_ids = sorted(set(com_set_gr.rename(columns={"participant_id": "subject_id"}).subject_id.astype(str))
                     - set(subj_gr.subject_id))
etapa("Grego", "Resting-state EEG file (.set) available", com_set_gr, len(excl_gr_ids), motivos(excl_gr_ids, fal_gr))
cc_gr = subj_gr.dropna(subset=COV_GR_)
etapa("Grego", "Analysable EEG: within-cohort LOSO (RQ1b, RQ2b) and transport target (RQ3a)", subj_gr,
      len(subj_gr) - len(cc_gr), {"missing covariate": len(subj_gr) - len(cc_gr)} if len(subj_gr) > len(cc_gr) else None)
etapa("Grego", f"Complete covariates ({nomes_cov(COV_GR_)}): incremental-value model", cc_gr)
etapa("Grego", "AD with MMSE (exploratory)", subj_gr[(subj_gr.y == 1) & subj_gr.mmse.notna()])

TAB_FLUXO = pd.DataFrame(FLUXO)
TAB_FLUXO.to_csv(OUT / "tabelas" / "F0_fluxo_participantes.csv", index=False, encoding="utf-8-sig")
display(TAB_FLUXO)


def caixa(ax, x, y, txt, cinza=False, largura=38):
    ax.text(x, y, "\n".join(textwrap.fill(t, largura) for t in txt.split("\n")), ha="center", va="center",
            fontsize=7.2, bbox=dict(boxstyle="round,pad=0.45", fc="#f2f2f2" if cinza else "white",
                                    ec="0.45" if cinza else "black", lw=0.8))


def desenhar_fluxo(ax, titulo, linhas):
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(titulo, fontsize=9.5, fontweight="bold")
    ys = np.linspace(0.90, 0.07, len(linhas))
    for i, (y, l) in enumerate(zip(ys, linhas.itertuples())):
        ftd = getattr(l, "FTD", np.nan)
        extra = f", FTD {int(ftd)}" if pd.notna(ftd) else ""
        caixa(ax, 0.33, y, f"{l.etapa}\nn = {l.n} (AD {l.AD}{extra}, HC {l.HC})")
        if i < len(linhas) - 1:
            ax.annotate("", xy=(0.33, ys[i + 1] + 0.055), xytext=(0.33, y - 0.055),
                        arrowprops=dict(arrowstyle="-|>", lw=0.8, color="black"))
            if str(l.excluidos_na_etapa) not in ("", "0"):
                ym = (y + ys[i + 1]) / 2
                ax.annotate("", xy=(0.66, ym), xytext=(0.33, ym), arrowprops=dict(arrowstyle="-|>", lw=0.8, color="0.3"))
                caixa(ax, 0.83, ym, f"Excluded: n = {l.excluidos_na_etapa}\n{l.motivo}", cinza=True, largura=30)


fig, axs = plt.subplots(1, 2, figsize=(11, 7.5))
desenhar_fluxo(axs[0], "BrainLat (Argentina and Chile)", TAB_FLUXO[TAB_FLUXO.coorte == "BrainLat"])
desenhar_fluxo(axs[1], "Greek cohort (ds004504)", TAB_FLUXO[TAB_FLUXO.coorte == "Grego"])
fig.tight_layout()
salvar_fig(fig, "fig0_fluxograma_participantes")
plt.show()
registrar("Submissão", "Fluxo de participantes (fig0; F0)",
          f"BrainLat {len(arq_bl)} com .set → {len(subj_bl)} analisáveis ({len(excl_bl_ids)} excluídos: "
          + "; ".join(f"{k} {v}" for k, v in mot_bl.items()) + f"); grego {n_bruto['n']} → {len(part)} AD+HC "
          f"(DFT {n_ftd} excluídos por desenho) → {len(subj_gr)} analisáveis", "Results (Fig. 1)")

### 13.2 Gravidade cognitiva por coorte e quem ficou de fora no BrainLat

> **O que faz:**
> - Para cada escala (MoCA e IFS no BrainLat; MMSE no grego), mostra n, média ± DP, mediana [IIQ] e faixa em AD e controles.
>   Também calcula o g de Hedges AD − HC com IC 95%: quanto os AD estão abaixo dos próprios controles, em desvios-padrão.
>   Essa é a comparação de gravidade que dá para fazer entre coortes que usaram escalas diferentes.
> - Nos AD do grego, distribui o MMSE nos estágios de Perneczky et al. (2006): 21–25 leve, 11–20 moderada, 0–10 grave.
>   O MoCA não tem estágios de gravidade validados, e MoCA e MMSE não são intercambiáveis. Por isso as médias brutas
>   não devem ser comparadas entre as coortes.
> - Compara incluídos e excluídos do BrainLat (diagnóstico, sítio, idade, sexo, escolaridade, MoCA). Com poucos
>   excluídos, a comparação é só descritiva.
>
> **Por que importa:** a diferença de AUC entre as coortes (0,87 × 0,68) pode vir em parte de gravidade diferente
> (efeito de espectro). Os revisores vão perguntar.
>
> **Onde entra:** Table 1 (ou 1b), Results (participants) e Limitations.

In [ ]:
# 13.2 — Gravidade por coorte (TRIPOD+AI 20b) e incluídos × excluídos no BrainLat
def hedges_g(a, b):
    a, b = pd.Series(a).dropna().astype(float), pd.Series(b).dropna().astype(float)
    n1, n2 = len(a), len(b)
    if n1 < 2 or n2 < 2:
        return np.nan, np.nan, np.nan
    sp = np.sqrt(((n1 - 1) * a.var() + (n2 - 1) * b.var()) / (n1 + n2 - 2))
    if not np.isfinite(sp) or sp == 0:
        return np.nan, np.nan, np.nan
    g = (1 - 3 / (4 * (n1 + n2) - 9)) * (a.mean() - b.mean()) / sp
    se = np.sqrt((n1 + n2) / (n1 * n2) + g ** 2 / (2 * (n1 + n2)))
    return g, g - 1.96 * se, g + 1.96 * se


def mediana_iiq(s):
    s = pd.Series(s).dropna()
    if len(s) == 0:
        return "—"
    return f"{s.median():.1f} [{s.quantile(.25):.1f}, {s.quantile(.75):.1f}]"


def faixa(s):
    s = pd.Series(s).dropna()
    return "—" if len(s) == 0 else f"{s.min():.0f}–{s.max():.0f}"


GRAV = []
for coorte, s, col, rot in (("BrainLat", subj_bl, "moca_total", "MoCA (0–30)"),
                            ("BrainLat", subj_bl, "ifs_total_score", "IFS (0–30)"),
                            ("Grego (ds004504)", subj_gr, "mmse", "MMSE (0–30)")):
    if col not in s or s[col].notna().sum() == 0:
        continue
    AD, HC = s.loc[s.y == 1, col], s.loc[s.y == 0, col]
    g, glo, ghi = hedges_g(AD, HC)
    for grupo, v in (("AD", AD), ("HC", HC)):
        GRAV.append({"coorte": coorte, "escala": rot, "grupo": grupo,
                     "n com escore / n": f"{v.notna().sum()} / {len(v)}", "média ± DP": media_dp(v),
                     "mediana [IIQ]": mediana_iiq(v), "faixa": faixa(v),
                     "g de Hedges AD − HC [IC 95%]": fmt_ci(g, glo, ghi, 2) if grupo == "AD" else ""})
    registrar("Submissão", f"{coorte}: gravidade ({rot.split(' ')[0]}), g de Hedges AD − HC",
              f"{fmt_ci(g, glo, ghi, 2)}; AD {media_dp(AD)} (n = {AD.notna().sum()}), HC {media_dp(HC)} "
              f"(n = {HC.notna().sum()})", "Table 1b; Limitations")
TAB_GRAV = pd.DataFrame(GRAV)
TAB_GRAV.to_csv(OUT / "tabelas" / "T1b_gravidade_por_coorte.csv", index=False, encoding="utf-8-sig")
display(TAB_GRAV)

# Estágios do MMSE nos AD do grego (Perneczky et al., 2006)
mm = subj_gr.loc[subj_gr.y == 1, "mmse"].dropna()
estagios = pd.cut(mm, bins=[-0.1, 10, 20, 25, 29, 30],
                  labels=["severe (0–10)", "moderate (11–20)", "mild (21–25)", "questionable (26–29)", "none (30)"])
TAB_EST = (estagios.value_counts().reindex(estagios.cat.categories).rename_axis("estágio (MMSE)")
           .reset_index(name="n AD"))
TAB_EST["%"] = (100 * TAB_EST["n AD"] / max(len(mm), 1)).round(1)
TAB_EST.to_csv(OUT / "tabelas" / "T1c_estagios_MMSE_grego.csv", index=False, encoding="utf-8-sig")
print(f"Grego, AD com MMSE (n = {len(mm)}), estágios de Perneczky et al. (2006):")
display(TAB_EST)
if len(mm):
    registrar("Submissão", "Grego: AD por estágio do MMSE (Perneczky 2006)",
              "; ".join(f"{r['estágio (MMSE)']}: {r['n AD']} ({r['%']}%)" for _, r in TAB_EST.iterrows()),
              "Table 1b; Discussion (spectrum)")

# Incluídos × excluídos no BrainLat
bl_all = (arq_bl.merge(demo_all[["subject_id", "sex", "Age", "years_education"]].drop_duplicates("subject_id"),
                       on="subject_id", how="left")
                .merge(df_cog[["subject_id", "moca_total"]], on="subject_id", how="left"))
for c in ("sex", "Age", "years_education", "moca_total"):
    bl_all[c] = pd.to_numeric(bl_all[c], errors="coerce")
bl_all["sex_fem"] = np.where(bl_all["sex"].isna(), np.nan, (bl_all["sex"] == FEM_BL).astype(float))
bl_all["incluido"] = bl_all.subject_id.isin(subj_bl.subject_id)
I, E = bl_all[bl_all.incluido], bl_all[~bl_all.incluido]
COMP = [{"variável": "n", "incluídos": len(I), "excluídos": len(E), "p": "", "teste": ""}]
if len(E):
    def tab2(cond):
        return [[int(cond(I).sum()), int((~cond(I)).sum())], [int(cond(E).sum()), int((~cond(E)).sum())]]
    for rot, cond in (("AD, n (%)", lambda d: d.y == 1), ("Chile, n (%)", lambda d: d.site == "Chile")):
        t = tab2(cond)
        COMP.append({"variável": rot, "incluídos": f"{t[0][0]} ({100 * t[0][0] / max(len(I), 1):.0f})",
                     "excluídos": f"{t[1][0]} ({100 * t[1][0] / max(len(E), 1):.0f})", "p": fmt_p(p_fisher(t)),
                     "teste": "Fisher"})
    fi, fe = I.sex_fem.dropna(), E.sex_fem.dropna()
    t = [[int(fi.sum()), int((1 - fi).sum())], [int(fe.sum()), int((1 - fe).sum())]]
    COMP.append({"variável": "Female, n (%)", "incluídos": f"{int(fi.sum())} ({100 * fi.mean():.0f})" if len(fi) else "—",
                 "excluídos": f"{int(fe.sum())} ({100 * fe.mean():.0f})" if len(fe) else "—",
                 "p": fmt_p(p_fisher(t)) if len(fi) and len(fe) else "n/d", "teste": "Fisher"})
    for rot, col in (("Age (years)", "Age"), ("Education (years)", "years_education"), ("MoCA", "moca_total")):
        COMP.append({"variável": rot, "incluídos": media_dp(I[col], True), "excluídos": media_dp(E[col], True),
                     "p": fmt_p(p_mwu(I[col], E[col])), "teste": "Mann–Whitney"})
    COMP.append({"variável": "Diagnosis × site of the excluded", "incluídos": "",
                 "excluídos": "; ".join(f"{('AD' if y == 1 else 'HC')} {s}: {n}" for (y, s), n in
                                        E.groupby(["y", "site"]).size().items()), "p": "", "teste": ""})
TAB_EXCL = pd.DataFrame(COMP)
TAB_EXCL.to_csv(OUT / "tabelas" / "S7_brainlat_incluidos_vs_excluidos.csv", index=False, encoding="utf-8-sig")
print(f"BrainLat: incluídos ({len(I)}) × excluídos ({len(E)}):")
display(TAB_EXCL)
if len(E):
    registrar("Submissão", "BrainLat: quem foi excluído (sem EEG analisável)",
              TAB_EXCL.iloc[-1]["excluídos"], "Methods; Limitations")

### 13.3 Checklist TRIPOD+AI pré-preenchido

> **O que faz:** lista os 27 itens (52 subitens) do TRIPOD+AI ([Collins et al., 2024](https://doi.org/10.1136/bmj-2023-078378)).
> Para cada um, diz se o notebook já tem o material (e onde) ou se ele precisa ser escrito no manuscrito. O texto dos itens
> é um resumo. Na submissão, usem o formulário oficial (anexo do artigo do BMJ), com os números de página do manuscrito.
> Este estudo **desenvolve** (LOSO dentro de cada coorte) e **avalia** (transporte entre coortes), então valem os itens
> D e E.
>
> **Onde entra:** material suplementar (o medRxiv e as revistas da Springer aceitam o checklist como arquivo à parte).

In [ ]:
# 13.3 — TRIPOD+AI: onde cada item já está e o que falta
P, PA, ESC, NA = "pronto no notebook", "parcial", "escrever no manuscrito", "não se aplica (dizer por quê)"
TRIPOD = [
    ("Title", "1", "D;E", "Identify development/evaluation of a multivariable prediction model, target population and outcome",
     "Title", "título de trabalho na SPEC", PA),
    ("Abstract", "2", "D;E", "See TRIPOD+AI for Abstracts", "Abstract", "—", ESC),
    ("Introduction", "3a", "D;E", "Healthcare context and rationale, including existing models",
     "Introduction", "seção 12 (literatura); benchmark AHEPA", PA),
    ("Introduction", "3b", "D;E", "Target population, intended purpose in the care pathway, intended users",
     "Introduction", "—", ESC),
    ("Introduction", "3c", "D;E", "Known health inequalities between sociodemographic groups",
     "Introduction", "escolaridade (RQ1, RQ2); coorte latino-americana", PA),
    ("Introduction", "4", "D;E", "Objectives, including whether the study develops, evaluates or both",
     "Introduction", "cabeçalho do notebook (Objective, RQ1–RQ3)", P),
    ("Methods — data", "5a", "D;E", "Data sources for development and evaluation, rationale and representativeness",
     "Methods", "seções 1.2b, 2.1, 2.2; DOIs dos bancos", P),
    ("Methods — data", "5b", "D;E", "Dates of data collection, start and end of accrual",
     "Methods", "descritores dos bancos (Prado 2023; Miltiadous 2023)", ESC),
    ("Methods — participants", "6a", "D;E", "Study setting, number and location of centres",
     "Methods", "T1 (sítios do BrainLat); ds004504: um centro", P),
    ("Methods — participants", "6b", "D;E", "Eligibility criteria", "Methods", "2.1, 2.2, 3.3; F0 e fig0", P),
    ("Methods — participants", "6c", "D;E", "Treatments received and how they were handled",
     "Methods", "não disponível nos bancos: declarar", ESC),
    ("Methods — data preparation", "7", "D;E", "Pre-processing and quality checks, and whether similar across groups",
     "Methods", "3.1–3.3 (19 canais, banda 40 Hz, épocas); S1; S2", P),
    ("Methods — outcome", "8a", "D;E", "Outcome definition, how and when assessed, consistency across groups",
     "Methods", "diagnóstico clínico de cada banco (descritores)", PA),
    ("Methods — outcome", "8b", "D;E", "Qualifications of outcome assessors if subjective",
     "Methods", "descritores dos bancos", ESC),
    ("Methods — outcome", "8c", "D;E", "Blinding of outcome assessment",
     "Methods", "diagnóstico anterior e independente das features; declarar", ESC),
    ("Methods — predictors", "9a", "D", "How initial predictors were chosen; any pre-selection",
     "Methods", "SPEC: as 11 features do artigo da ACDSA, fixadas antes", P),
    ("Methods — predictors", "9b", "D;E", "Definition of all predictors, how and when measured, blinding",
     "Methods", "3.1 (definições); demografia em 2.1, 2.2", P),
    ("Methods — predictors", "9c", "D;E", "Qualifications of predictor assessors if subjective",
     "Methods", "features automáticas", NA),
    ("Methods — sample size", "10", "D;E", "How study size was arrived at and why it is sufficient",
     "Methods", "amostra = bancos disponíveis; largura dos IC no RESUMO", ESC),
    ("Methods — missing data", "11", "D;E", "Handling of missing data and reasons for omitted data",
     "Methods", "casos completos; F0; S7", P),
    ("Methods — analysis", "12a", "D", "How data were used and partitioned, given sample size",
     "Methods", "LOSO (1.7, 4); transporte (7)", P),
    ("Methods — analysis", "12b", "D", "Handling of predictors (functional form, rescaling, standardisation)",
     "Methods", "RF sem padronização; escore em logit por DP (1.5)", P),
    ("Methods — analysis", "12c", "D", "Model type, rationale, building steps, hyperparameters, internal validation",
     "Methods", "SPEC (hiperparâmetros fixos); 1.7; sensibilidade SVM/XGBoost (9)", P),
    ("Methods — analysis", "12d", "D;E", "Heterogeneity across clusters: handling and quantification",
     "Methods", "RQ1c (coorte × EEG, 5.3); verificação com sítio (5.1)", P),
    ("Methods — analysis", "12e", "D;E", "All performance measures and plots, with rationale; model comparisons",
     "Methods", "AUC, ΔAUC, OR/DP, inclinação, CITL, Brier, métricas no limiar", P),
    ("Methods — analysis", "12f", "E", "Model updating (e.g., recalibration) after evaluation",
     "Methods", "sem atualização; padronização por coorte é secundária (7.3)", P),
    ("Methods — analysis", "12g", "E", "How predictions were calculated for evaluation",
     "Methods", "modelos congelados em modelos/*.pkl; predições em tabelas/", P),
    ("Methods — class imbalance", "13", "D;E", "Class-imbalance methods, why and how; recalibration",
     "Methods", "peso balanceado de classes no RF (SPEC); sem reamostragem", P),
    ("Methods — fairness", "14", "D;E", "Approaches to address model fairness",
     "Methods", "confundimento por escolaridade (RQ2); equidade não avaliada formalmente: declarar", PA),
    ("Methods — model output", "15", "D", "Model output and how classification thresholds were chosen",
     "Methods", "probabilidade; limiar 0,5 só descritivo", P),
    ("Methods — training vs evaluation", "16", "D;E", "Differences between development and evaluation data",
     "Methods", "T1; S2 (mudança de domínio); 2.1 vs 2.2", P),
    ("Methods — ethics", "17", "D;E", "Ethics committee and informed consent or waiver",
     "Methods", "pacote_preprint/DECLARACOES_rascunho.md", ESC),
    ("Open science", "18a", "D;E", "Funding source and role of funders", "Declarations", "—", ESC),
    ("Open science", "18b", "D;E", "Conflicts of interest and financial disclosures", "Declarations", "—", ESC),
    ("Open science", "18c", "D;E", "Where the protocol can be accessed, or that none was prepared",
     "Methods", "SPEC_journal.json + hash + data de congelamento (log_execucao.json)", P),
    ("Open science", "18d", "D;E", "Registration details, or that the study was not registered",
     "Methods", "não registrado externamente: declarar", ESC),
    ("Open science", "18e", "D;E", "Availability of study data", "Declarations", "DOIs dos bancos; termos do BrainLat", PA),
    ("Open science", "18f", "D;E", "Availability of analytical code", "Declarations",
     "este notebook; depositar no Zenodo com DOI", PA),
    ("Patient and public involvement", "19", "D;E", "Patient and public involvement, or state none",
     "Methods", "nenhum (dados secundários): declarar", ESC),
    ("Results — participants", "20a", "D;E", "Participant flow, numbers with and without the outcome (diagram)",
     "Results; Fig. 1", "13.1: fig0_fluxograma_participantes; F0", P),
    ("Results — participants", "20b", "D;E", "Participant characteristics by data source, predictors, missing data",
     "Results; Table 1", "T1; T1b; T1c; S7", P),
    ("Results — participants", "20c", "E", "Compare predictors and outcome between evaluation and development data",
     "Results", "T1; S2 (SMD entre coortes)", P),
    ("Results — model development", "21", "D;E", "Number of participants and outcome events in each analysis",
     "Results", "F0 (n por análise); T2", P),
    ("Results — model specification", "22", "D", "Full model (code, object or API) for predictions and evaluation",
     "Declarations", "modelos/*.pkl e *.json; notebook", P),
    ("Results — performance", "23a", "D;E", "Performance estimates with confidence intervals, incl. key subgroups",
     "Results; Table 2", "T2; fig1–fig3", P),
    ("Results — performance", "23b", "D;E", "Heterogeneity in performance across clusters, if examined",
     "Results", "RQ1c; desempenho por coorte", P),
    ("Results — model updating", "24", "E", "Results of any model updating", "Results",
     "sem atualização; 7.3 secundária", NA),
    ("Discussion", "25", "D;E", "Overall interpretation, including fairness, versus objectives and previous studies",
     "Discussion", "leituras pela regra da SPEC (RESUMO)", ESC),
    ("Discussion", "26", "D;E", "Limitations: representativeness, size, overfitting, missing data; effects on bias",
     "Discussion", "fim da seção 12 (limitações); S7; T1b", PA),
    ("Discussion", "27a", "D", "How poor-quality or unavailable input data should be handled in practice",
     "Discussion", "—", ESC),
    ("Discussion", "27b", "D", "Whether users interact with inputs/model and the expertise required",
     "Discussion", "—", ESC),
    ("Discussion", "27c", "D;E", "Next steps for research toward applicability and generalisability",
     "Discussion", "—", ESC),
]
TAB_TRIPOD = pd.DataFrame(TRIPOD, columns=["seção", "item", "aplica", "resumo do item (TRIPOD+AI)",
                                           "onde no manuscrito", "onde está no notebook / fonte", "status"])
TAB_TRIPOD.to_csv(OUT / "tabelas" / "TRIPOD_AI_checklist.csv", index=False, encoding="utf-8-sig")
md_t = ["# TRIPOD+AI — checklist pré-preenchido", "",
        "Referência: Collins GS et al. BMJ 2024;385:e078378 — https://doi.org/10.1136/bmj-2023-078378", "",
        "Resumo dos itens; na submissão, usem o formulário oficial com os números de página.", "",
        "| " + " | ".join(TAB_TRIPOD.columns) + " |", "|" + "---|" * len(TAB_TRIPOD.columns)]
md_t += ["| " + " | ".join(str(v).replace("|", "/") for v in r) + " |" for r in TAB_TRIPOD.itertuples(index=False)]
(OUT / "tabelas" / "TRIPOD_AI_checklist.md").write_text("\n".join(md_t) + "\n", encoding="utf-8")
print(TAB_TRIPOD["status"].value_counts().to_string())
display(TAB_TRIPOD)

### 13.4 Pacote do preprint (medRxiv)

> **O que faz:** cria `pacote_preprint/` (e `pacote_preprint.zip`) com as figuras, as tabelas, o checklist TRIPOD+AI,
> a SPEC, o log, o resumo, um rascunho das declarações que o medRxiv pede e um passo a passo. Depois refaz
> `para_enviar.zip` com os arquivos novos e acrescenta ao `RESUMO_JOURNAL.md` os números desta seção.
>
> **O que o medRxiv pede** (página "Submit a manuscript"): PDF com título, autores, afiliações e resumo; declarações de
> ética, conflitos de interesse, financiamento e disponibilidade de dados; consentimento de todos os autores. Ele **não
> posta** manuscrito já aceito ou publicado, então o preprint tem que sair antes do aceite na revista. Submeter à revista
> antes ou ao mesmo tempo é permitido.

In [ ]:
# 13.4 — Pacote do preprint + atualização do zip e do resumo
PKG = OUT / "pacote_preprint"
if PKG.exists():
    shutil.rmtree(PKG)
for sub in ("figuras", "tabelas", "suplementar"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
for f in sorted((OUT / "figuras").glob("*")):
    if f.suffix in (".pdf", ".png"):
        shutil.copy2(f, PKG / "figuras" / f.name)
for f in sorted((OUT / "tabelas").glob("*")):
    if f.suffix in (".csv", ".md", ".json"):
        shutil.copy2(f, PKG / "tabelas" / f.name)
for f in (OUT / "RESUMO_JOURNAL.md", OUT / "SPEC_journal.json", OUT / "log_execucao.json", OUT / "ambiente.txt",
          OUT / "download_ds004504.json"):
    if f.exists():
        shutil.copy2(f, PKG / "suplementar" / f.name)

congelada = LOG.get("spec_congelada_em", "n/d")
lidos = LOG.get("rotulos_gregos_lidos_em", "n/d")
DECL = f'''# Rascunho das declarações (inglês) — revisar e completar os campos [PREENCHER]

## Ethics statement
This study is a secondary analysis of de-identified data from two datasets: BrainLat (Prado et al., 2023,
https://doi.org/10.1038/s41597-023-02806-8) and the OpenNeuro dataset ds004504 v1.0.9
(https://doi.org/10.18112/openneuro.ds004504.v1.0.9; Miltiadous et al., 2023, https://doi.org/10.3390/data8060095).
Ethics approval and informed consent for the original data collection were obtained by the original investigators, as
described in the data descriptors. [PREENCHER: posição do CEP da UEPB sobre análise secundária de dados públicos
anonimizados — confirmar com o CEP antes de escrever qualquer dispensa.]

## Data availability
ds004504 is openly available on OpenNeuro (https://doi.org/10.18112/openneuro.ds004504.v1.0.9). BrainLat data are
available from the BrainLat consortium under its data access terms (https://doi.org/10.1038/s41597-023-02806-8)
[PREENCHER: link exato e condições de acesso do BrainLat].

## Code availability
All analyses were run from a single notebook. The analysis specification (SPEC hash {SPEC_HASH}) was frozen on
{congelada}. The Greek diagnostic labels were first read only after the freeze, in the same execution (logged at
{lidos}; the code freezes the specification before reading them). This is an internal, time-stamped analysis plan, not an
external preregistration. Code, specification and execution log: [PREENCHER: DOI do Zenodo].

## Reporting guideline
Reported following TRIPOD+AI (Collins et al., 2024, https://doi.org/10.1136/bmj-2023-078378); checklist in the
Supplementary Material.

## Funding
[PREENCHER: fontes de financiamento e papel dos financiadores, ou "This research received no specific funding."]

## Competing interests
[PREENCHER: pagamentos ou serviços de terceiros nos últimos 36 meses, ou "The authors declare no competing interests."]

## Author contributions
[PREENCHER: CRediT]

## Prior presentation
[PREENCHER se o artigo da ACDSA 2027 for mantido: citar o artigo da conferência e dizer o que é novo aqui — segunda
coorte, testes cruzados independentes, confundimento por desenho e calibração no transporte.]
'''
(PKG / "DECLARACOES_rascunho.md").write_text(DECL, encoding="utf-8")

LEIA = '''# Como postar o preprint no medRxiv

1. Escrevam o manuscrito e gerem um PDF único com página de rosto (título, autores, afiliações e resumo), texto,
   figuras e tabelas. O material suplementar vai em arquivos separados: as tabelas S*, os arquivos T1b, T1c e S7 e o
   TRIPOD_AI_checklist (CSV ou MD).
2. Todos os autores precisam concordar com o depósito.
3. Completem `DECLARACOES_rascunho.md`: ética, conflitos de interesse, financiamento e disponibilidade de dados são
   obrigatórios no medRxiv.
4. Depositem o código (notebook + SPEC + log) no Zenodo para ter um DOI e coloquem esse DOI na declaração de código.
5. Licença: CC BY é a mais simples e combina com o acesso aberto pelo acordo CAPES–Springer Nature, que exige CC BY.
6. Submetam em https://www.medrxiv.org/submit-a-manuscript, na área Neurology. Cada autor que deposita precisa de conta.
7. O medRxiv não posta manuscrito já aceito ou publicado. Postem antes do aceite; submeter à revista ao mesmo tempo é
   permitido.
8. Depois de postado, citem o DOI do preprint na carta ao editor e na seção de declarações.
9. Se o artigo da ACDSA 2027 for mantido, citem-no no preprint e no manuscrito e expliquem o que é novo.

Conteúdo desta pasta:
- figuras/: fig0 (fluxograma), fig1 (ROC), fig2 (OR incremental), fig3 (confundimento), em PDF e PNG
- tabelas/: T1, T1b (gravidade), T1c (estágios do MMSE), T2, F0 (fluxo), S1–S7, TRIPOD_AI_checklist
- suplementar/: RESUMO_JOURNAL.md, SPEC_journal.json, log_execucao.json, ambiente.txt, download_ds004504.json
'''
(PKG / "LEIA_ME_PREPRINT.md").write_text(LEIA, encoding="utf-8")

f_pkg = OUT / "pacote_preprint.zip"
with zipfile.ZipFile(f_pkg, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(PKG.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(PKG.parent))

# Números desta seção no fim do RESUMO (substitui o bloco de uma execução anterior)
f_res13 = OUT / "RESUMO_JOURNAL.md"
if f_res13.exists():
    txt = f_res13.read_text(encoding="utf-8").split("\n## Material para submissão (seção 13)")[0].rstrip("\n")
    bloco = ["", "## Material para submissão (seção 13)", "", "| Item | Valor | Onde entra |", "|---|---|---|"]
    reg13 = {r["item"]: r for r in REG if r["rq"] == "Submissão"}   # sem repetir se a célula rodar de novo
    bloco += [f"| {str(r['item']).replace('|', '/')} | {str(r['valor']).replace('|', '/')} | {r['onde']} |"
              for r in reg13.values()]
    bloco += ["", f"TRIPOD+AI: {', '.join(f'{k}: {v}' for k, v in TAB_TRIPOD['status'].value_counts().items())}"]
    f_res13.write_text(txt + "\n" + "\n".join(bloco) + "\n", encoding="utf-8")
    shutil.copy2(f_res13, PKG / "suplementar" / f_res13.name)

f_zip13 = OUT / "para_enviar.zip"
with zipfile.ZipFile(f_zip13, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [f_res13, OUT / "SPEC_journal.json", OUT / "log_execucao.json", OUT / "ambiente.txt",
              OUT / "tabelas" / "desenho_confundimento.json", OUT / "download_ds004504.json"]:
        if f.exists():
            z.write(f, f.name)
    for pasta in ("tabelas", "figuras"):
        for f in sorted((OUT / pasta).glob("*")):
            if f.suffix in (".csv", ".pdf", ".png", ".json", ".md"):
                z.write(f, f"{pasta}/{f.name}")
    for f in sorted(PKG.rglob("*")):
        if f.is_file():
            z.write(f, str(f.relative_to(OUT)))
print(f"Pacote do preprint: {PKG}  ({f_pkg.name})")
print(f"para_enviar.zip refeito com os arquivos novos: {f_zip13}")
print("Falta fazer à mão: manuscrito em PDF, declarações [PREENCHER], DOI do Zenodo e o depósito no medRxiv.")